In [1]:

from pathlib import Path
import polars as pl

# Original dataset — read only
DATA_PATH = Path.cwd() / "2019-Oct.csv"

# One cumulative output file for all four weeks
OUTPUT_PATH = Path.cwd() / "2019-Oct_augmented.csv"

# Ground-truth record of the injected anomalies
MANIFEST_PATH = Path.cwd() / "anomaly_manifest.csv"

# Week 1 date range
WEEK_NAME = "Week 1"
START_DATE = "2019-10-01"
END_DATE = "2019-10-07"

# Your approved Week 1 injection targets
WEEK1_TARGETS = {
    "Bot-like Crawling Loops": 53_061,
    "Seasonal Promotional Purchase Surges": 42_449,
    "Duplicate Events": 42_449,
    "Impossible Sequences": 42_449,
    "Cart Abandonment Scenarios": 31_837,
    "Missing Data": 212_244,
}

ORIGINAL_COLUMNS = [
    "event_time",
    "event_type",
    "product_id",
    "category_id",
    "category_code",
    "brand",
    "price",
    "user_id",
    "user_session",
]

NEW_COLUMNS = [
    "marketing_channel",
    "platform",
]

ALL_COLUMNS = ORIGINAL_COLUMNS + NEW_COLUMNS

EXPECTED_INJECTED_ROWS = sum(WEEK1_TARGETS.values())

assert DATA_PATH.exists(), f"Original dataset not found: {DATA_PATH}"
assert EXPECTED_INJECTED_ROWS == 424_489

# Never overwrite an existing output from an earlier run.
if OUTPUT_PATH.exists() or MANIFEST_PATH.exists():
    raise FileExistsError(
        "Output or manifest already exists. "
        "Choose a new output filename or verify the previous run first."
    )

print("Setup completed.")
print(f"Original dataset: {DATA_PATH}")
print(f"Week: {WEEK_NAME} ({START_DATE} to {END_DATE})")
print(f"Injected rows planned: {EXPECTED_INJECTED_ROWS:,}")
print(f"Expected Week 1 output: 8,829,315 + 424,489 = 9,253,804 rows")


Setup completed.
Original dataset: d:\Samsung\Graduation Project\Data\Old Data\2019-Oct.csv
Week: Week 1 (2019-10-01 to 2019-10-07)
Injected rows planned: 424,489
Expected Week 1 output: 8,829,315 + 424,489 = 9,253,804 rows


In [3]:

# Read the source lazily
source = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
)

# Validate the original schema
schema = source.collect_schema()
assert schema.names() == ORIGINAL_COLUMNS, (
    f"Unexpected source schema: {schema.names()}"
)

# Filter by the first 10 characters of event_time (YYYY-MM-DD)
week1 = source.filter(
    (pl.col("event_time") >= f"{START_DATE} 00:00:00 UTC")
    & (pl.col("event_time") < f"2019-10-08 00:00:00 UTC")
)

print("Original schema:")
print(schema)

print("\nWeek 1 preview:")
print(week1.head(5).collect())

print("\nWeek 1 event counts:")
print(
    week1.group_by("event_type")
    .agg(pl.len().alias("row_count"))
    .sort("event_type")
    .collect()
)


Original schema:
Schema({'event_time': String, 'event_type': String, 'product_id': Int64, 'category_id': Int64, 'category_code': String, 'brand': String, 'price': Float64, 'user_id': Int64, 'user_session': String})

Week 1 preview:
shape: (5, 9)
┌────────────┬────────────┬───────────┬───────────┬───┬──────────┬─────────┬───────────┬───────────┐
│ event_time ┆ event_type ┆ product_i ┆ category_ ┆ … ┆ brand    ┆ price   ┆ user_id   ┆ user_sess │
│ ---        ┆ ---        ┆ d         ┆ id        ┆   ┆ ---      ┆ ---     ┆ ---       ┆ ion       │
│ str        ┆ str        ┆ ---       ┆ ---       ┆   ┆ str      ┆ f64     ┆ i64       ┆ ---       │
│            ┆            ┆ i64       ┆ i64       ┆   ┆          ┆         ┆           ┆ str       │
╞════════════╪════════════╪═══════════╪═══════════╪═══╪══════════╪═════════╪═══════════╪═══════════╡
│ 2019-10-01 ┆ view       ┆ 44600062  ┆ 210380745 ┆ … ┆ shiseido ┆ 35.79   ┆ 541312140 ┆ 72d76fde- │
│ 00:00:00   ┆            ┆           ┆ 9595387

In [4]:

# Add synthetic marketing attributes to Week 1.
week1_enriched = week1.with_columns(
    pl.when(pl.col("user_id") % 6 == 0)
      .then(pl.lit("Organic Search"))
    .when(pl.col("user_id") % 6 == 1)
      .then(pl.lit("Paid Search"))
    .when(pl.col("user_id") % 6 == 2)
      .then(pl.lit("Social Media"))
    .when(pl.col("user_id") % 6 == 3)
      .then(pl.lit("Email"))
    .when(pl.col("user_id") % 6 == 4)
      .then(pl.lit("Direct"))
    .otherwise(pl.lit("Referral"))
      .alias("marketing_channel"),

    pl.when(pl.col("user_id") % 2 == 0)
      .then(pl.lit("Web"))
      .otherwise(pl.lit("Mobile App"))
      .alias("platform"),
)

# Save Week 1 original records plus the two new columns.
# Do not overwrite an existing file.
if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"Output already exists: {OUTPUT_PATH}"
    )

week1_enriched.sink_csv(
    OUTPUT_PATH,
    include_header=True,
    maintain_order=True,
    engine="streaming",
)

print("Week 1 original data exported successfully.")
print(f"Output: {OUTPUT_PATH}")
print("Expected rows: 8,829,315")
print("Expected columns: 11")


Week 1 original data exported successfully.
Output: d:\Samsung\Graduation Project\Data\Old Data\2019-Oct_augmented.csv
Expected rows: 8,829,315
Expected columns: 11


In [5]:

from pathlib import Path
import polars as pl

OUTPUT_PATH = Path.cwd() / "2019-Oct_augmented.csv"

# Read the exported Week 1 file lazily
week1_output = pl.scan_csv(
    OUTPUT_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
)

# 1. Check schema
schema = week1_output.collect_schema()
print("Columns:", schema.names())
print("Number of columns:", len(schema))

# 2. Check row count
row_count = week1_output.select(
    pl.len().alias("row_count")
).collect().item()

print(f"\nCurrent rows: {row_count:,}")
print("Expected rows before injection: 8,829,315")

# 3. Check event distribution
print("\nEvent distribution:")
print(
    week1_output.group_by("event_type")
    .agg(pl.len().alias("rows"))
    .sort("event_type")
    .collect()
)

# 4. Check the two new columns
print("\nMarketing channel distribution:")
print(
    week1_output.group_by("marketing_channel")
    .agg(pl.len().alias("rows"))
    .sort("marketing_channel")
    .collect()
)

print("\nPlatform distribution:")
print(
    week1_output.group_by("platform")
    .agg(pl.len().alias("rows"))
    .sort("platform")
    .collect()
)

# 5. Check null counts
print("\nNull counts:")
print(
    week1_output.select([
        pl.col(column).null_count().alias(column)
        for column in schema.names()
    ]).collect()
)

assert len(schema.names()) == 11
assert row_count == 8_829_315

print("\nWeek 1 validation passed.")


Columns: ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session', 'marketing_channel', 'platform']
Number of columns: 11

Current rows: 8,829,315
Expected rows before injection: 8,829,315

Event distribution:
shape: (3, 2)
┌────────────┬─────────┐
│ event_type ┆ rows    │
│ ---        ┆ ---     │
│ str        ┆ u32     │
╞════════════╪═════════╡
│ cart       ┆ 182773  │
│ purchase   ┆ 152115  │
│ view       ┆ 8494427 │
└────────────┴─────────┘

Marketing channel distribution:
shape: (6, 2)
┌───────────────────┬─────────┐
│ marketing_channel ┆ rows    │
│ ---               ┆ ---     │
│ str               ┆ u32     │
╞═══════════════════╪═════════╡
│ Direct            ┆ 1465354 │
│ Email             ┆ 1471870 │
│ Organic Search    ┆ 1462165 │
│ Paid Search       ┆ 1471670 │
│ Referral          ┆ 1483762 │
│ Social Media      ┆ 1474494 │
└───────────────────┴─────────┘

Platform distribution:
shape: (2, 2)
┌────────────┬──────

## Missing Data Injection

### Objective
Simulate incomplete e-commerce event records by introducing missing values into selected fields. This helps test data quality checks, missing-value detection, and data cleaning processes.

### Methodology
1. Load the Week 1 dataset from the augmented output file before anomaly injection.
2. Validate the source schema and confirm the expected baseline of **8,829,315 rows**.
3. Select **212,244 source records** distributed across the entire week.
4. Copy each selected record and set one non-null field to `null`.
5. Distribute missing values across the 11 columns, including the original dataset fields and the synthetic `marketing_channel` and `platform` fields.
6. Record each injected anomaly in `anomaly_manifest.csv`, including its synthetic ID, source row, affected column, and original event information.
7. Append the generated records to the output CSV without modifying the original source dataset.

### Validation
- Confirm the baseline row count and column names.
- Verify that exactly **212,244 synthetic records** are generated.
- Ensure each generated record contains at least one missing value.
- Confirm that all synthetic row IDs are unique.
- Validate the final output and manifest counts.

### Expected Results

| Metric | Expected Value |
|---|---:|
| Original Week 1 rows | 8,829,315 |
| Synthetic rows added | 212,244 |
| Total rows after injection | 9,041,559 |
| Manifest records | 212,244 |

### Important Note
The injected records are copies of existing source records with one field intentionally set to `null`. The original dataset remains unchanged, while the manifest provides traceability for every synthetic anomaly.


In [6]:

import polars as pl
from pathlib import Path

# Final output file containing Week 1 before anomaly injection
FINAL_PATH = OUTPUT_PATH
MANIFEST_PATH = Path.cwd() / "anomaly_manifest.csv"

TARGET_ROWS = WEEK1_TARGETS["Missing Data"]
EXPECTED_BASE_ROWS = 8_829_315

# 1. Validate the current output file
if not FINAL_PATH.exists():
    raise FileNotFoundError(f"Output file not found: {FINAL_PATH}")

if MANIFEST_PATH.exists():
    raise FileExistsError(
        "The manifest already exists. Inspect the output file and manifest "
        "before proceeding to avoid duplicate injection."
    )

source = pl.scan_csv(
    FINAL_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
)

schema = source.collect_schema()
base_count = source.select(pl.len()).collect().item()

assert schema.names() == ALL_COLUMNS
assert base_count == EXPECTED_BASE_ROWS, (
    f"Expected {EXPECTED_BASE_ROWS:,} rows, found {base_count:,}"
)

# 2. Select source rows distributed across the entire week
indexed = source.with_row_index("_source_row")
idx = pl.col("_source_row").cast(pl.Int64)

candidates = (
    indexed
    .filter(
        (((idx + 1) * TARGET_ROWS) // base_count)
        > ((idx * TARGET_ROWS) // base_count)
    )
    .collect(engine="streaming")
)

assert candidates.height == TARGET_ROWS, (
    f"Expected {TARGET_ROWS:,} candidate rows, "
    f"but found {candidates.height:,}"
)

# 3. Generate Missing Data records
generated_rows = []
manifest_rows = []

for number, record in enumerate(
    candidates.iter_rows(named=True), start=1
):
    source_row_number = int(record["_source_row"]) + 1

    # Distribute missing values across the columns.
    # If the selected value is already null, choose the next non-null column.
    start = (number - 1) % len(ALL_COLUMNS)
    missing_column = None

    for offset in range(len(ALL_COLUMNS)):
        column = ALL_COLUMNS[(start + offset) % len(ALL_COLUMNS)]

        if record[column] is not None:
            missing_column = column
            break

    if missing_column is None:
        raise ValueError(
            f"Source row {source_row_number} has no non-null values."
        )

    # Copy the source record and set one value to null
    new_record = {col: record[col] for col in ALL_COLUMNS}
    new_record[missing_column] = None
    generated_rows.append(new_record)

    # Record the injected anomaly in the manifest
    manifest_rows.append({
        "week": WEEK_NAME,
        "anomaly_type": "Missing Data",
        "synthetic_row_id": f"W1-MD-{number:06d}",
        "source_row_number": source_row_number,
        "missing_column": missing_column,
        "source_event_time": record["event_time"],
        "source_event_type": record["event_type"],
        "source_user_id": record["user_id"],
        "source_product_id": record["product_id"],
    })

# 4. Build DataFrames and validate them before writing
missing_df = pl.DataFrame(
    generated_rows,
    schema=schema,
    orient="row",
    strict=False,
)

manifest_df = pl.DataFrame(manifest_rows)

assert missing_df.height == TARGET_ROWS
assert missing_df.columns == ALL_COLUMNS
assert manifest_df.height == TARGET_ROWS
assert manifest_df["synthetic_row_id"].n_unique() == TARGET_ROWS

# Ensure every generated record contains at least one null value
assert (
    missing_df.select(
        pl.sum_horizontal([
            pl.col(col).is_null().cast(pl.Int64)
            for col in ALL_COLUMNS
        ]).min()
    ).item() >= 1
)

# 5. Append the generated records directly to the final output file
# Do not write another CSV header
with open(FINAL_PATH, "ab") as output_file:
    missing_df.write_csv(output_file, include_header=False)

# Save the anomaly manifest
manifest_df.write_csv(MANIFEST_PATH)

print("Missing Data injection completed successfully.")
print(f"Rows before injection: {base_count:,}")
print(f"Rows added: {missing_df.height:,}")
print(f"Expected total rows: {base_count + TARGET_ROWS:,}")
print(f"Manifest records: {manifest_df.height:,}")
print(f"Output file: {FINAL_PATH}")
print(f"Manifest file: {MANIFEST_PATH}")

print("\nInjected missing values by column:")
print(
    manifest_df
    .group_by("missing_column")
    .agg(pl.len().alias("injected_nulls"))
    .sort("missing_column")
)


Missing Data injection completed successfully.
Rows before injection: 8,829,315
Rows added: 212,244
Expected total rows: 9,041,559
Manifest records: 212,244
Output file: d:\Samsung\Graduation Project\Data\Old Data\2019-Oct_augmented.csv
Manifest file: d:\Samsung\Graduation Project\Data\Old Data\anomaly_manifest.csv

Injected missing values by column:
shape: (11, 2)
┌────────────────┬────────────────┐
│ missing_column ┆ injected_nulls │
│ ---            ┆ ---            │
│ str            ┆ u32            │
╞════════════════╪════════════════╡
│ brand          ┆ 21042          │
│ category_code  ┆ 13174          │
│ category_id    ┆ 19295          │
│ event_time     ┆ 19295          │
│ event_type     ┆ 19295          │
│ …              ┆ …              │
│ platform       ┆ 19294          │
│ price          ┆ 23669          │
│ product_id     ┆ 19295          │
│ user_id        ┆ 19295          │
│ user_session   ┆ 19295          │
└────────────────┴────────────────┘


In [7]:

# Validate the output file after Missing Data injection

output_check = pl.scan_csv(
    OUTPUT_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
)

actual_rows = output_check.select(pl.len()).collect().item()

manifest_check = pl.read_csv(MANIFEST_PATH)

print(f"Actual output rows: {actual_rows:,}")
print(f"Expected output rows: {9_041_559:,}")
print(f"Manifest records: {manifest_check.height:,}")

assert actual_rows == 9_041_559
assert manifest_check.height == 212_244
assert manifest_check["synthetic_row_id"].n_unique() == 212_244

print("\nWeek 1 Missing Data validation passed.")


Actual output rows: 9,041,559
Expected output rows: 9,041,559
Manifest records: 212,244

Week 1 Missing Data validation passed.


## 2. Duplicate Events Injection

### Objective
Simulate duplicate product events within the same user session using a composite key consisting of `user_session` and `product_id`.

### Composite Key
```python
COMPOSITE_KEY = ["user_session", "product_id"]
```

### Methodology
1. Select source records from the original Week 1 dataset only.
2. Exclude records with null values in the composite key or `event_time`.
3. Generate **42,449 synthetic records** while preserving the composite key values.
4. Increase `event_time` by one second for each generated record so that the new record is not an exact copy of the source row.
5. Append the generated records to the augmented dataset.
6. Record the source row, composite key context, and anomaly type in the anomaly manifest.

### Validation
- Verify the number of generated records.
- Confirm that `user_session` and `product_id` remain unchanged for each generated record.
- Verify the total output row count and manifest record count.

### Expected Results
| Metric | Expected Value |
|---|---:|
| Records generated | 42,449 |
| Output rows after injection | 9,083,008 |
| Total manifest records | 254,693 |

### Important Note
The composite key identifies the same product within a user session, but it does not necessarily prove that two events are genuine duplicates. Repeated interactions with the same product may be legitimate. This scenario represents synthetic duplicate-key events for testing data quality and deduplication logic.

In [8]:

from datetime import datetime, timedelta
import polars as pl

# Configuration
FINAL_PATH = OUTPUT_PATH
MANIFEST_PATH = Path.cwd() / "anomaly_manifest.csv"

COMPOSITE_KEY = ["user_session", "product_id"]
TARGET_ROWS = WEEK1_TARGETS["Duplicate Events"]

ORIGINAL_WEEK1_ROWS = 8_829_315
EXPECTED_CURRENT_ROWS = 9_041_559
EXPECTED_MANIFEST_ROWS = 212_244

# 1. Validate the current files
if not FINAL_PATH.exists() or not MANIFEST_PATH.exists():
    raise FileNotFoundError("The output file or manifest file is missing.")

source = pl.scan_csv(
    FINAL_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
)

schema = source.collect_schema()
current_rows = source.select(pl.len()).collect().item()
manifest = pl.read_csv(MANIFEST_PATH)

assert schema.names() == ALL_COLUMNS
assert current_rows == EXPECTED_CURRENT_ROWS, (
    f"Expected {EXPECTED_CURRENT_ROWS:,} rows, found {current_rows:,}"
)
assert manifest.height == EXPECTED_MANIFEST_ROWS
assert not any(
    str(value).startswith("W1-DU-")
    for value in manifest["synthetic_row_id"].to_list()
), "Duplicate Events IDs already exist. Inspect the files before retrying."

# 2. Restrict source records to original Week 1 rows
# Exclude records with null values in the composite key or event_time.
original = (
    source
    .with_row_index("_source_row")
    .filter(pl.col("_source_row") < ORIGINAL_WEEK1_ROWS)
    .filter(
        pl.col("user_session").is_not_null()
        & pl.col("product_id").is_not_null()
        & pl.col("event_time").is_not_null()
    )
)

eligible_count = original.select(pl.len()).collect().item()

assert eligible_count >= TARGET_ROWS, (
    f"Not enough eligible source records: {eligible_count:,}"
)

# 3. Select exactly TARGET_ROWS, distributed across eligible records
eligible = original.with_row_index("_eligible_row")
eligible_idx = pl.col("_eligible_row").cast(pl.Int64)

candidates = (
    eligible
    .filter(
        (((eligible_idx + 1) * TARGET_ROWS) // eligible_count)
        > ((eligible_idx * TARGET_ROWS) // eligible_count)
    )
    .collect(engine="streaming")
)

assert candidates.height == TARGET_ROWS, (
    f"Expected {TARGET_ROWS:,} candidates, found {candidates.height:,}"
)

# 4. Generate records with the same composite key
duplicate_rows = []
duplicate_manifest_rows = []

for number, record in enumerate(
    candidates.iter_rows(named=True), start=1
):
    source_row_number = int(record["_source_row"]) + 1

    new_record = {column: record[column] for column in ALL_COLUMNS}

    # Change event_time so the new record is not a full-row copy.
    # The composite key remains unchanged.
    original_time = datetime.strptime(
        record["event_time"],
        "%Y-%m-%d %H:%M:%S UTC",
    )
    new_record["event_time"] = (
        (original_time + timedelta(seconds=1))
        .strftime("%Y-%m-%d %H:%M:%S UTC")
    )

    assert all(
        new_record[column] == record[column]
        for column in COMPOSITE_KEY
    )

    duplicate_rows.append(new_record)

    duplicate_manifest_rows.append({
        "week": WEEK_NAME,
        "anomaly_type": "Duplicate Events",
        "synthetic_row_id": f"W1-DU-{number:06d}",
        "source_row_number": source_row_number,
        "missing_column": None,
        "source_event_time": record["event_time"],
        "source_event_type": record["event_type"],
        "source_user_id": record["user_id"],
        "source_product_id": record["product_id"],
    })

duplicates_df = pl.DataFrame(
    duplicate_rows,
    schema=schema,
    orient="row",
    strict=False,
)

duplicates_manifest_df = pl.DataFrame(
    duplicate_manifest_rows,
    schema=manifest.schema,
    strict=False,
)

# 5. Validate generated records before writing
assert duplicates_df.height == TARGET_ROWS
assert duplicates_df.columns == ALL_COLUMNS
assert duplicates_manifest_df.height == TARGET_ROWS
assert (
    duplicates_manifest_df["synthetic_row_id"].n_unique()
    == TARGET_ROWS
)

# 6. Append records directly to the output file
with open(FINAL_PATH, "ab") as output_file:
    duplicates_df.write_csv(output_file, include_header=False)

# 7. Append manifest records without writing another header
with open(MANIFEST_PATH, "ab") as manifest_file:
    duplicates_manifest_df.write_csv(
        manifest_file,
        include_header=False,
    )

print("Duplicate Events injection completed successfully.")
print(f"Composite key: {COMPOSITE_KEY}")
print(f"Eligible source records: {eligible_count:,}")
print(f"Rows before injection: {current_rows:,}")
print(f"Duplicate-key records added: {duplicates_df.height:,}")
print(f"Expected total rows: {current_rows + TARGET_ROWS:,}")
print(f"Manifest records added: {duplicates_manifest_df.height:,}")
print(f"Expected manifest total: {manifest.height + TARGET_ROWS:,}")
print(f"Output file: {FINAL_PATH}")
print(f"Manifest file: {MANIFEST_PATH}")


Duplicate Events injection completed successfully.
Composite key: ['user_session', 'product_id']
Eligible source records: 8,829,314
Rows before injection: 9,041,559
Duplicate-key records added: 42,449
Expected total rows: 9,084,008
Manifest records added: 42,449
Expected manifest total: 254,693
Output file: d:\Samsung\Graduation Project\Data\Old Data\2019-Oct_augmented.csv
Manifest file: d:\Samsung\Graduation Project\Data\Old Data\anomaly_manifest.csv


In [9]:

# Validate the output after Duplicate Events injection

output_check = pl.scan_csv(
    OUTPUT_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
)

actual_rows = output_check.select(pl.len()).collect().item()
manifest_check = pl.read_csv(MANIFEST_PATH)

print(f"Actual output rows: {actual_rows:,}")
print(f"Expected output rows: {9_084_008:,}")
print(f"Actual manifest records: {manifest_check.height:,}")
print(f"Expected manifest records: {254_693:,}")

print("\nManifest records by anomaly type:")
print(
    manifest_check
    .group_by("anomaly_type")
    .agg(pl.len().alias("records"))
    .sort("anomaly_type")
)

assert actual_rows == 9_084_008
assert manifest_check.height == 254_693
assert (
    manifest_check["synthetic_row_id"].n_unique()
    == 254_693
)

print("\nWeek 1 Duplicate Events validation passed.")


Actual output rows: 9,084,008
Expected output rows: 9,084,008
Actual manifest records: 254,693
Expected manifest records: 254,693

Manifest records by anomaly type:
shape: (2, 2)
┌──────────────────┬─────────┐
│ anomaly_type     ┆ records │
│ ---              ┆ ---     │
│ str              ┆ u32     │
╞══════════════════╪═════════╡
│ Duplicate Events ┆ 42449   │
│ Missing Data     ┆ 212244  │
└──────────────────┴─────────┘

Week 1 Duplicate Events validation passed.


## 3. Impossible Sequences Injection

### Objective
Simulate logically inconsistent or unusual event sequences in e-commerce user sessions to test data quality validation and event-ordering rules.

### Scenario Definition
Generate synthetic records representing a `purchase → cart` sequence for the same product within the same user session, where the `cart` event occurs after the `purchase` event.

### Methodology
1. Identify suitable source records from the original Week 1 dataset.
2. Use `user_session` and `product_id` to associate events with the same session and product.
3. Generate **42,449 synthetic records** that satisfy the defined sequence rule.
4. Assign event timestamps to ensure that the `cart` event occurs after the `purchase` event.
5. Append the generated records to the augmented dataset.
6. Record the anomaly type, source information, and synthetic record identifiers in the anomaly manifest.

### Validation
- Verify that the expected number of records was generated.
- Confirm that each generated sequence uses the same `user_session` and `product_id`.
- Verify that the `cart` timestamp is later than the corresponding `purchase` timestamp.
- Confirm that the output row count and manifest record count match expectations.

### Expected Results
| Metric | Expected Value |
|---|---:|
| Records generated | 42,449 |
| Output rows after injection | 9,126,457 |
| Total manifest records | 297,142 |

### Important Note
The `purchase → cart` ordering is used as a defined synthetic anomaly scenario for this project. It is not universally impossible in every e-commerce system, since event tracking, delayed processing, or repeated interactions can produce unusual sequences. The scenario should therefore be documented as a rule violation under the project's chosen business logic.

In [11]:

import polars as pl
from pathlib import Path
from datetime import datetime, timedelta
import csv

# File paths
FINAL_PATH = OUTPUT_PATH
MANIFEST_PATH = Path.cwd() / "anomaly_manifest.csv"

# Expected counts for the current pipeline state
TARGET_ROWS = WEEK1_TARGETS["Impossible Sequences"]
ORIGINAL_WEEK1_ROWS = 8_829_315
EXPECTED_CURRENT_ROWS = 9_084_008
EXPECTED_CURRENT_MANIFEST = 254_693

EXPECTED_FINAL_ROWS = EXPECTED_CURRENT_ROWS + TARGET_ROWS
EXPECTED_FINAL_MANIFEST = EXPECTED_CURRENT_MANIFEST + TARGET_ROWS

COMPOSITE_KEY = ["user_session", "product_id"]

# 1. Validate files and current state
if not FINAL_PATH.exists():
    raise FileNotFoundError(f"Output file not found: {FINAL_PATH}")

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest file not found: {MANIFEST_PATH}")

source = pl.scan_csv(
    FINAL_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
)

schema = source.collect_schema()
current_rows = source.select(pl.len()).collect().item()

manifest = pl.read_csv(MANIFEST_PATH, infer_schema_length=10_000)
current_manifest_rows = manifest.height

if schema.names() != ALL_COLUMNS:
    raise ValueError(f"Unexpected output schema: {schema.names()}")

if current_rows != EXPECTED_CURRENT_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_CURRENT_ROWS:,} output rows, "
        f"found {current_rows:,}. Inspect the file before proceeding."
    )

if current_manifest_rows != EXPECTED_CURRENT_MANIFEST:
    raise ValueError(
        f"Expected {EXPECTED_CURRENT_MANIFEST:,} manifest records, "
        f"found {current_manifest_rows:,}. "
        "Inspect the manifest before proceeding."
    )

if (
    manifest.filter(
        pl.col("anomaly_type") == "Impossible Sequences"
    ).height > 0
):
    raise ValueError(
        "Impossible Sequences already exists in the manifest. "
        "Do not inject it again."
    )

print(f"Current output rows: {current_rows:,}")
print(f"Current manifest records: {current_manifest_rows:,}")

# 2. Read only original Week 1 purchase events
# The original records occupy the first ORIGINAL_WEEK1_ROWS data rows.
original = (
    source
    .head(ORIGINAL_WEEK1_ROWS)
    .with_row_index("_source_row")
)

purchase_candidates = (
    original
    .filter(
        (pl.col("event_type") == "purchase")
        & pl.col("event_time").is_not_null()
        & pl.col("user_session").is_not_null()
        & pl.col("product_id").is_not_null()
        # Avoid generating a timestamp outside Week 1.
        & (
            pl.col("event_time")
            < "2019-10-07 23:59:59 UTC"
        )
    )
    .collect(engine="streaming")
)

if purchase_candidates.height < TARGET_ROWS:
    raise ValueError(
        f"Not enough eligible purchase records: "
        f"{purchase_candidates.height:,}"
    )

print(
    f"Eligible original purchase records: "
    f"{purchase_candidates.height:,}"
)


# 3. Select exactly TARGET_ROWS, distributed across eligible purchases

candidate_count = purchase_candidates.height

if candidate_count < TARGET_ROWS:
    raise ValueError(
        f"Not enough eligible purchases: {candidate_count:,}"
    )

# Generate exactly TARGET_ROWS distinct candidate positions.
selected_indices = [
    (i * candidate_count) // TARGET_ROWS
    for i in range(TARGET_ROWS)
]

selected = (
    purchase_candidates
    .with_row_index("_candidate_index")
    .filter(
        pl.col("_candidate_index").is_in(selected_indices)
    )
)

if selected.height != TARGET_ROWS:
    raise ValueError(
        f"Expected {TARGET_ROWS:,} selected purchases, "
        f"found {selected.height:,}"
    )

print(f"Selected purchase records: {selected.height:,}")

# 4. Generate synthetic cart records after their source purchases
generated_rows = []
manifest_rows = []

for number, record in enumerate(
    selected.iter_rows(named=True),
    start=1,
):
    source_row_number = int(record["_source_row"]) + 1

    purchase_time_text = record["event_time"]
    purchase_time = datetime.strptime(
        purchase_time_text,
        "%Y-%m-%d %H:%M:%S UTC",
    )

    cart_time = purchase_time + timedelta(seconds=1)
    cart_time_text = cart_time.strftime(
        "%Y-%m-%d %H:%M:%S UTC"
    )

    # Copy the source record, then create the synthetic cart event.
    new_record = {
        column: record[column]
        for column in ALL_COLUMNS
    }
    new_record["event_time"] = cart_time_text
    new_record["event_type"] = "cart"

    # Keep the same user_session and product_id.
    assert new_record["user_session"] == record["user_session"]
    assert new_record["product_id"] == record["product_id"]
    assert cart_time > purchase_time

    generated_rows.append(new_record)

    manifest_rows.append({
        "week": WEEK_NAME,
        "anomaly_type": "Impossible Sequences",
        "synthetic_row_id": f"W1-IS-{number:06d}",
        "source_row_number": source_row_number,
        "missing_column": None,
        "source_event_time": purchase_time_text,
        "source_event_type": "purchase",
        "source_user_id": record["user_id"],
        "source_product_id": record["product_id"],
    })

# 5. Build and validate DataFrames before appending
impossible_df = pl.DataFrame(
    generated_rows,
    schema=schema,
    orient="row",
    strict=False,
)

new_manifest_df = pl.DataFrame(
    manifest_rows,
    schema=manifest.schema,
    strict=False,
)

assert impossible_df.height == TARGET_ROWS
assert impossible_df.columns == ALL_COLUMNS
assert new_manifest_df.height == TARGET_ROWS
assert (
    new_manifest_df["synthetic_row_id"].n_unique()
    == TARGET_ROWS
)
assert (impossible_df["event_type"] == "cart").all()

# 6. Append synthetic records without writing another CSV header
with open(FINAL_PATH, "ab") as output_file:
    impossible_df.write_csv(output_file, include_header=False)

# Append manifest rows without duplicating its header
with open(MANIFEST_PATH, "a", newline="", encoding="utf-8") as manifest_file:
    writer = csv.DictWriter(
        manifest_file,
        fieldnames=manifest.columns,
    )
    writer.writerows(manifest_rows)

# 7. Validate final counts
final_rows = (
    pl.scan_csv(
        FINAL_PATH,
        infer_schema_length=10_000,
        null_values=["", "null", "NULL"],
    )
    .select(pl.len())
    .collect()
    .item()
)

final_manifest = pl.read_csv(
    MANIFEST_PATH,
    infer_schema_length=10_000,
)

actual_injected = final_manifest.filter(
    pl.col("anomaly_type") == "Impossible Sequences"
).height

assert final_rows == EXPECTED_FINAL_ROWS, (
    f"Expected {EXPECTED_FINAL_ROWS:,} rows, found {final_rows:,}"
)
assert final_manifest.height == EXPECTED_FINAL_MANIFEST
assert actual_injected == TARGET_ROWS

print("\nImpossible Sequences injection completed successfully.")
print(f"Rows before injection: {current_rows:,}")
print(f"Rows added: {TARGET_ROWS:,}")
print(f"Rows after injection: {final_rows:,}")
print(f"Manifest records after injection: {final_manifest.height:,}")
print(f"Impossible Sequences records: {actual_injected:,}")
print(f"Output file: {FINAL_PATH}")
print(f"Manifest file: {MANIFEST_PATH}")


Current output rows: 9,084,008
Current manifest records: 254,693
Eligible original purchase records: 152,115
Selected purchase records: 42,449

Impossible Sequences injection completed successfully.
Rows before injection: 9,084,008
Rows added: 42,449
Rows after injection: 9,126,457
Manifest records after injection: 297,142
Impossible Sequences records: 42,449
Output file: d:\Samsung\Graduation Project\Data\Old Data\2019-Oct_augmented.csv
Manifest file: d:\Samsung\Graduation Project\Data\Old Data\anomaly_manifest.csv


In [12]:

import polars as pl
from datetime import datetime, timedelta
from pathlib import Path

FINAL_PATH = OUTPUT_PATH
MANIFEST_PATH = Path.cwd() / "anomaly_manifest.csv"

BASE_ROWS = 9_084_008
ORIGINAL_WEEK1_ROWS = 8_829_315
TARGET_ROWS = 42_449
EXPECTED_TOTAL_ROWS = 9_126_457
EXPECTED_MANIFEST_ROWS = 297_142

# 1. Check the manifest
manifest = pl.read_csv(
    MANIFEST_PATH,
    infer_schema_length=10_000,
)

scenario_manifest = (
    manifest
    .filter(pl.col("anomaly_type") == "Impossible Sequences")
    .sort("synthetic_row_id")
)

assert scenario_manifest.height == TARGET_ROWS, (
    f"Wrong scenario count: {scenario_manifest.height:,}"
)

assert scenario_manifest["synthetic_row_id"].n_unique() == TARGET_ROWS
assert scenario_manifest["source_event_type"].eq("purchase").all()

assert manifest.height == EXPECTED_MANIFEST_ROWS, (
    f"Unexpected manifest total: {manifest.height:,}"
)

# 2. Check output row count and retrieve the newly appended records
output = pl.scan_csv(
    FINAL_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
)

actual_total = output.select(pl.len()).collect().item()

assert actual_total == EXPECTED_TOTAL_ROWS, (
    f"Unexpected output total: {actual_total:,}"
)

# Row indices are zero-based: the appended section starts at BASE_ROWS.
new_records = (
    output
    .with_row_index("_row_index")
    .filter(pl.col("_row_index") >= BASE_ROWS)
    .collect(engine="streaming")
)

assert new_records.height == TARGET_ROWS
assert (new_records["event_type"] == "cart").all()

# 3. Match generated records to their manifest entries by append order
manifest_for_join = scenario_manifest.with_row_index("_check_index")
records_for_join = new_records.with_row_index("_check_index")

joined = records_for_join.join(
    manifest_for_join.select([
        "_check_index",
        "synthetic_row_id",
        "source_row_number",
        "source_event_time",
        "source_event_type",
        "source_product_id",
    ]),
    on="_check_index",
    how="inner",
)

assert joined.height == TARGET_ROWS

# Verify product IDs and source event types
assert (
    joined["product_id"].cast(pl.String)
    == joined["source_product_id"].cast(pl.String)
).all()

assert (joined["source_event_type"] == "purchase").all()

# 4. Verify each generated timestamp is exactly one second later
def parse_utc(value):
    return datetime.strptime(value, "%Y-%m-%d %H:%M:%S UTC")

time_checks = [
    parse_utc(cart_time) == parse_utc(purchase_time) + timedelta(seconds=1)
    for cart_time, purchase_time in zip(
        joined["event_time"].to_list(),
        joined["source_event_time"].to_list(),
    )
]

assert all(time_checks), "Some generated timestamps are incorrect."

# 5. Retrieve the original source rows to verify session and product keys
source_row_numbers = (
    scenario_manifest["source_row_number"]
    .cast(pl.UInt32)
    .to_list()
)

source_records = (
    output
    .head(ORIGINAL_WEEK1_ROWS)
    .with_row_index("_source_index")
    .filter(
        (pl.col("_source_index") + 1).is_in(source_row_numbers)
    )
    .select([
        (pl.col("_source_index") + 1).alias("source_row_number"),
        "event_type",
        "event_time",
        "user_session",
        "product_id",
    ])
    .collect(engine="streaming")
)

assert source_records.height == TARGET_ROWS

# Check source rows are the intended original purchases.
source_check = source_records.join(
    scenario_manifest.select([
        "source_row_number",
        "source_event_time",
        "source_event_type",
        "source_user_id",
        "source_product_id",
    ]),
    on="source_row_number",
    how="inner",
)

assert source_check.height == TARGET_ROWS
assert (source_check["event_type"] == "purchase").all()
assert (
    source_check["event_time"] == source_check["source_event_time"]
).all()
assert (
    source_check["product_id"].cast(pl.String)
    == source_check["source_product_id"].cast(pl.String)
).all()

# Match each generated record to its original source and verify the session key.
full_check = joined.join(
    source_records.select([
        "source_row_number",
        pl.col("user_session").alias("original_user_session"),
        pl.col("product_id").alias("original_product_id"),
    ]),
    on="source_row_number",
    how="inner",
)

assert full_check.height == TARGET_ROWS
assert (
    full_check["user_session"] == full_check["original_user_session"]
).all()
assert (
    full_check["product_id"].cast(pl.String)
    == full_check["original_product_id"].cast(pl.String)
).all()

print("All Impossible Sequences validation checks passed.")
print(f"Output rows: {actual_total:,}")
print(f"Manifest rows: {manifest.height:,}")
print(f"Impossible Sequences records: {scenario_manifest.height:,}")
print(f"Generated event type is cart: {(new_records['event_type'] == 'cart').all()}")
print(f"Product IDs match source: True")
print(f"User sessions match source: True")
print(f"Cart timestamps are exactly one second later: {all(time_checks)}")
print("No data was modified by this validation.")


All Impossible Sequences validation checks passed.
Output rows: 9,126,457
Manifest rows: 297,142
Impossible Sequences records: 42,449
Generated event type is cart: True
Product IDs match source: True
User sessions match source: True
Cart timestamps are exactly one second later: True
No data was modified by this validation.


## 4. Bot-like Crawling Loops Injection

### Objective
Simulate bot-like browsing behavior by generating repeated product-view events within the same user session over very short time intervals.

### Scenario Definition
Create synthetic `view` events that repeat the same `product_id` and `user_session` shortly after an existing view event. The generated event occurs 1–3 seconds after its source event.

### Methodology
1. Select eligible `view` events from the original Week 1 records.
2. Exclude records with missing timestamps, product IDs, or session IDs.
3. Select **53,061 source records**, distributed across eligible events.
4. Generate synthetic `view` events for the same product and session, using timestamps 1–3 seconds after the source event.
5. Append the generated records to the augmented dataset.
6. Record each synthetic event and its source information in `anomaly_manifest.csv`.

### Validation
- Confirm that exactly 53,061 records are generated.
- Verify that every generated record has `event_type = view`.
- Confirm that `user_session` and `product_id` match the source record.
- Verify that each generated timestamp is 1–3 seconds after its source event.
- Confirm the output and manifest counts match expectations.

### Expected Results

| Metric | Expected Value |
|---|---:|
| Synthetic rows generated | 53,061 |
| Output rows before injection | 9,126,457 |
| Output rows after injection | 9,179,518 |
| Manifest records before injection | 297,142 |
| Manifest records after injection | 350,203 |

### Important Note
This scenario simulates bot-like behavior for testing purposes. Repeated views alone do not prove that a user is a bot; production detection would normally consider additional signals such as request frequency, session patterns, and behavioral diversity. The original source dataset remains unchanged.


In [13]:

import polars as pl
from pathlib import Path
from datetime import datetime, timedelta
import csv

# File paths
FINAL_PATH = OUTPUT_PATH
MANIFEST_PATH = Path.cwd() / "anomaly_manifest.csv"

# Expected counts before and after injection
TARGET_ROWS = WEEK1_TARGETS["Bot-like Crawling Loops"]
ORIGINAL_WEEK1_ROWS = 8_829_315

EXPECTED_CURRENT_ROWS = 9_126_457
EXPECTED_CURRENT_MANIFEST = 297_142

EXPECTED_FINAL_ROWS = EXPECTED_CURRENT_ROWS + TARGET_ROWS
EXPECTED_FINAL_MANIFEST = EXPECTED_CURRENT_MANIFEST + TARGET_ROWS

ANOMALY_NAME = "Bot-like Crawling Loops"

# 1. Validate files and current pipeline state
if not FINAL_PATH.exists():
    raise FileNotFoundError(f"Output file not found: {FINAL_PATH}")

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest file not found: {MANIFEST_PATH}")

source = pl.scan_csv(
    FINAL_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
)

schema = source.collect_schema()
current_rows = source.select(pl.len()).collect().item()

manifest = pl.read_csv(
    MANIFEST_PATH,
    infer_schema_length=10_000,
)
current_manifest_rows = manifest.height

if schema.names() != ALL_COLUMNS:
    raise ValueError(f"Unexpected output schema: {schema.names()}")

if current_rows != EXPECTED_CURRENT_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_CURRENT_ROWS:,} output rows, "
        f"found {current_rows:,}. Inspect before proceeding."
    )

if current_manifest_rows != EXPECTED_CURRENT_MANIFEST:
    raise ValueError(
        f"Expected {EXPECTED_CURRENT_MANIFEST:,} manifest records, "
        f"found {current_manifest_rows:,}."
    )

if manifest.filter(
    pl.col("anomaly_type") == ANOMALY_NAME
).height > 0:
    raise ValueError(
        "Bot-like Crawling Loops already exists in the manifest. "
        "Do not inject it again."
    )

print(f"Current output rows: {current_rows:,}")
print(f"Current manifest records: {current_manifest_rows:,}")

# 2. Select eligible view events from original Week 1 records only
original = (
    source
    .head(ORIGINAL_WEEK1_ROWS)
    .with_row_index("_source_row")
)

candidates = (
    original
    .filter(
        (pl.col("event_type") == "view")
        & pl.col("event_time").is_not_null()
        & pl.col("user_session").is_not_null()
        & pl.col("product_id").is_not_null()
        # Leave room for adding up to three seconds.
        & (
            pl.col("event_time")
            < "2019-10-07 23:59:56 UTC"
        )
    )
    .collect(engine="streaming")
)

candidate_count = candidates.height

if candidate_count < TARGET_ROWS:
    raise ValueError(
        f"Not enough eligible view events: {candidate_count:,}"
    )

print(f"Eligible original view events: {candidate_count:,}")

# 3. Select exactly TARGET_ROWS, distributed across candidates
selected_indices = [
    (i * candidate_count) // TARGET_ROWS
    for i in range(TARGET_ROWS)
]

selected = (
    candidates
    .with_row_index("_candidate_index")
    .filter(
        pl.col("_candidate_index").is_in(selected_indices)
    )
)

if selected.height != TARGET_ROWS:
    raise ValueError(
        f"Expected {TARGET_ROWS:,} selected events, "
        f"found {selected.height:,}"
    )

print(f"Selected source events: {selected.height:,}")

# 4. Generate synthetic repeated view events
generated_rows = []
manifest_rows = []

for number, record in enumerate(
    selected.iter_rows(named=True),
    start=1,
):
    source_row_number = int(record["_source_row"]) + 1

    source_time_text = record["event_time"]
    source_time = datetime.strptime(
        source_time_text,
        "%Y-%m-%d %H:%M:%S UTC",
    )

    # Repeat views at short intervals of one to three seconds.
    delay_seconds = ((number - 1) % 3) + 1
    synthetic_time = source_time + timedelta(
        seconds=delay_seconds
    )

    new_record = {
        column: record[column]
        for column in ALL_COLUMNS
    }

    new_record["event_time"] = synthetic_time.strftime(
        "%Y-%m-%d %H:%M:%S UTC"
    )
    new_record["event_type"] = "view"

    assert new_record["user_session"] == record["user_session"]
    assert new_record["product_id"] == record["product_id"]
    assert 1 <= delay_seconds <= 3

    generated_rows.append(new_record)

    manifest_rows.append({
        "week": WEEK_NAME,
        "anomaly_type": ANOMALY_NAME,
        "synthetic_row_id": f"W1-BL-{number:06d}",
        "source_row_number": source_row_number,
        "missing_column": None,
        "source_event_time": source_time_text,
        "source_event_type": "view",
        "source_user_id": record["user_id"],
        "source_product_id": record["product_id"],
    })

# 5. Build and validate data before writing
bot_df = pl.DataFrame(
    generated_rows,
    schema=schema,
    orient="row",
    strict=False,
)

new_manifest_df = pl.DataFrame(
    manifest_rows,
    schema=manifest.schema,
    strict=False,
)

assert bot_df.height == TARGET_ROWS
assert bot_df.columns == ALL_COLUMNS
assert (bot_df["event_type"] == "view").all()

assert new_manifest_df.height == TARGET_ROWS
assert (
    new_manifest_df["synthetic_row_id"].n_unique()
    == TARGET_ROWS
)

# 6. Append synthetic events to the output file
with open(FINAL_PATH, "ab") as output_file:
    bot_df.write_csv(output_file, include_header=False)

# Append manifest entries without writing another header
with open(
    MANIFEST_PATH,
    "a",
    newline="",
    encoding="utf-8",
) as manifest_file:
    writer = csv.DictWriter(
        manifest_file,
        fieldnames=manifest.columns,
    )
    writer.writerows(manifest_rows)

# 7. Validate the files after writing
final_rows = (
    pl.scan_csv(
        FINAL_PATH,
        infer_schema_length=10_000,
        null_values=["", "null", "NULL"],
    )
    .select(pl.len())
    .collect()
    .item()
)

final_manifest = pl.read_csv(
    MANIFEST_PATH,
    infer_schema_length=10_000,
)

actual_injected = final_manifest.filter(
    pl.col("anomaly_type") == ANOMALY_NAME
).height

assert final_rows == EXPECTED_FINAL_ROWS, (
    f"Expected {EXPECTED_FINAL_ROWS:,} rows, found {final_rows:,}"
)

assert final_manifest.height == EXPECTED_FINAL_MANIFEST
assert actual_injected == TARGET_ROWS

print("\nBot-like Crawling Loops injection completed successfully.")
print(f"Rows before injection: {current_rows:,}")
print(f"Rows added: {TARGET_ROWS:,}")
print(f"Rows after injection: {final_rows:,}")
print(f"Manifest records after injection: {final_manifest.height:,}")
print(f"Bot-like Crawling Loops records: {actual_injected:,}")
print(f"Output file: {FINAL_PATH}")
print(f"Manifest file: {MANIFEST_PATH}")


Current output rows: 9,126,457
Current manifest records: 297,142
Eligible original view events: 8,494,414
Selected source events: 53,061

Bot-like Crawling Loops injection completed successfully.
Rows before injection: 9,126,457
Rows added: 53,061
Rows after injection: 9,179,518
Manifest records after injection: 350,203
Bot-like Crawling Loops records: 53,061
Output file: d:\Samsung\Graduation Project\Data\Old Data\2019-Oct_augmented.csv
Manifest file: d:\Samsung\Graduation Project\Data\Old Data\anomaly_manifest.csv


In [14]:

import polars as pl
from datetime import datetime, timedelta

FINAL_PATH = OUTPUT_PATH
MANIFEST_PATH = Path.cwd() / "anomaly_manifest.csv"

BASE_ROWS = 9_126_457
ORIGINAL_WEEK1_ROWS = 8_829_315
TARGET_ROWS = 53_061

EXPECTED_TOTAL_ROWS = 9_179_518
EXPECTED_MANIFEST_ROWS = 350_203
ANOMALY_NAME = "Bot-like Crawling Loops"

# 1. Validate the manifest
manifest = pl.read_csv(
    MANIFEST_PATH,
    infer_schema_length=10_000,
)

scenario_manifest = (
    manifest
    .filter(pl.col("anomaly_type") == ANOMALY_NAME)
    .sort("synthetic_row_id")
)

assert scenario_manifest.height == TARGET_ROWS
assert scenario_manifest["synthetic_row_id"].n_unique() == TARGET_ROWS
assert (scenario_manifest["source_event_type"] == "view").all()
assert manifest.height == EXPECTED_MANIFEST_ROWS

# Validate synthetic IDs follow the expected naming convention.
expected_ids = [
    f"W1-BL-{i:06d}"
    for i in range(1, TARGET_ROWS + 1)
]
assert scenario_manifest["synthetic_row_id"].to_list() == expected_ids

# 2. Validate the output row count
output = pl.scan_csv(
    FINAL_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
)

actual_total = output.select(pl.len()).collect().item()
assert actual_total == EXPECTED_TOTAL_ROWS

# 3. Read the appended synthetic records
new_records = (
    output
    .with_row_index("_row_index")
    .filter(pl.col("_row_index") >= BASE_ROWS)
    .collect(engine="streaming")
)

assert new_records.height == TARGET_ROWS
assert (new_records["event_type"] == "view").all()

# 4. Match appended records to their manifest entries
manifest_for_join = scenario_manifest.with_row_index("_check_index")
records_for_join = new_records.with_row_index("_check_index")

joined = records_for_join.join(
    manifest_for_join.select([
        "_check_index",
        "synthetic_row_id",
        "source_row_number",
        "source_event_time",
        "source_event_type",
        "source_user_id",
        "source_product_id",
    ]),
    on="_check_index",
    how="inner",
)

assert joined.height == TARGET_ROWS

# Verify product and user identifiers against the manifest.
assert (
    joined["product_id"].cast(pl.String)
    == joined["source_product_id"].cast(pl.String)
).all()

assert (
    joined["user_id"].cast(pl.String)
    == joined["source_user_id"].cast(pl.String)
).all()

assert (joined["source_event_type"] == "view").all()

# 5. Verify generated timestamps are 1–3 seconds after source timestamps.
def parse_utc(value):
    return datetime.strptime(value, "%Y-%m-%d %H:%M:%S UTC")

delays = [
    (
        parse_utc(generated_time) - parse_utc(source_time)
    ).total_seconds()
    for generated_time, source_time in zip(
        joined["event_time"].to_list(),
        joined["source_event_time"].to_list(),
    )
]

assert all(delay in (1, 2, 3) for delay in delays), (
    "At least one generated event has an invalid time delay."
)

# 6. Verify source row references against the original Week 1 section.
source_row_numbers = (
    scenario_manifest["source_row_number"]
    .cast(pl.UInt32)
    .to_list()
)

source_records = (
    output
    .head(ORIGINAL_WEEK1_ROWS)
    .with_row_index("_source_index")
    .filter(
        (pl.col("_source_index") + 1).is_in(source_row_numbers)
    )
    .select([
        (pl.col("_source_index") + 1).alias("source_row_number"),
        "event_type",
        "event_time",
        "user_id",
        "user_session",
        "product_id",
    ])
    .collect(engine="streaming")
)

assert source_records.height == TARGET_ROWS

source_check = source_records.join(
    scenario_manifest.select([
        "source_row_number",
        "source_event_time",
        "source_event_type",
        "source_user_id",
        "source_product_id",
    ]),
    on="source_row_number",
    how="inner",
)

assert source_check.height == TARGET_ROWS
assert (source_check["event_type"] == "view").all()
assert (
    source_check["event_time"] == source_check["source_event_time"]
).all()

assert (
    source_check["user_id"].cast(pl.String)
    == source_check["source_user_id"].cast(pl.String)
).all()

assert (
    source_check["product_id"].cast(pl.String)
    == source_check["source_product_id"].cast(pl.String)
).all()

# Match each generated event to its original source session and product.
full_check = joined.join(
    source_records.select([
        "source_row_number",
        pl.col("user_session").alias("original_user_session"),
        pl.col("product_id").alias("original_product_id"),
    ]),
    on="source_row_number",
    how="inner",
)

assert full_check.height == TARGET_ROWS
assert (
    full_check["user_session"] == full_check["original_user_session"]
).all()
assert (
    full_check["product_id"].cast(pl.String)
    == full_check["original_product_id"].cast(pl.String)
).all()

print("All Bot-like Crawling Loops validation checks passed.")
print(f"Output rows: {actual_total:,}")
print(f"Manifest rows: {manifest.height:,}")
print(f"Bot-like Crawling Loops records: {scenario_manifest.height:,}")
print("Event type is view: True")
print("User and product IDs match the manifest: True")
print("User sessions and products match original source rows: True")
print(f"Timestamp delays range from {min(delays):.0f} to {max(delays):.0f} seconds.")
print("All timestamps are 1–3 seconds after source events: True")
print("No data was modified by this validation.")


All Bot-like Crawling Loops validation checks passed.
Output rows: 9,179,518
Manifest rows: 350,203
Bot-like Crawling Loops records: 53,061
Event type is view: True
User and product IDs match the manifest: True
User sessions and products match original source rows: True
Timestamp delays range from 1 to 3 seconds.
All timestamps are 1–3 seconds after source events: True
No data was modified by this validation.


## Seasonal Promotional Purchase Surges

### 1. Objective
Simulate a sudden increase in purchase events during a short promotional campaign to test the pipeline's ability to detect unusual spikes in e-commerce activity.

### 2. Anomaly Injection Strategy
- **Target:** 42,449 synthetic purchase events.
- **Source:** Original Week 1 purchase records only.
- **Campaign Window:** October 5, 2019, from 12:00:00 to 12:29:59 UTC.
- **Event Type:** `purchase`.
- **Marketing Channel:** `seasonal_promotion`.
- **Synthetic ID Prefix:** `W1-SP-`.

The selected original purchase records are used as templates. Their event timestamps are replaced with timestamps distributed across the promotional window, and the marketing channel is set to `seasonal_promotion` to identify the simulated campaign.

### 3. Data Integrity
- The original source file, `2019-Oct.csv`, remains unchanged.
- Only original Week 1 records are used as source templates; previously injected synthetic records are excluded.
- Each generated event receives a unique synthetic ID.
- Source references are recorded in `anomaly_manifest.csv`.
- Temporary output files are validated before appending records to the main dataset.

### 4. Expected Results

| Metric | Expected Value |
|---|---:|
| Synthetic purchase events | 42,449 |
| Output rows before injection | 9,179,518 |
| Output rows after injection | 9,221,967 |
| Manifest records before injection | 350,203 |
| Manifest records after injection | 392,652 |

### 5. Validation
After injection, verify that:
1. Exactly 42,449 records were added to the manifest for this anomaly.
2. All generated events have `event_type = purchase`.
3. All generated events have `marketing_channel = seasonal_promotion`.
4. All generated timestamps fall within the defined promotional window.
5. Synthetic IDs are unique.
6. The final output and manifest row counts match the expected totals.

**Note:** This is a synthetic promotional scenario for testing anomaly detection. It does not establish that a real promotional campaign occurred in the original dataset.

In [15]:

from pathlib import Path
from datetime import datetime, timedelta
import csv
import shutil
import polars as pl

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------
BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

OUTPUT_FILE = BASE_DIR / "2019-Oct_augmented.csv"
MANIFEST_FILE = BASE_DIR / "anomaly_manifest.csv"

ANOMALY_TYPE = "Seasonal Promotional Purchase Surges"
TARGET_ROWS = 42_449
ORIGINAL_WEEK1_ROWS = 8_829_315

EXPECTED_OUTPUT_ROWS = 9_179_518
EXPECTED_MANIFEST_ROWS = 350_203

ANOMALY_PREFIX = "W1-SP-"

PROMO_START = datetime(2019, 10, 5, 12, 0, 0)
PROMO_DURATION_SECONDS = 30 * 60

ORIGINAL_COLUMNS = [
    "event_time", "event_type", "product_id", "category_id",
    "category_code", "brand", "price", "user_id", "user_session"
]

ALL_COLUMNS = ORIGINAL_COLUMNS + ["marketing_channel", "platform"]

# --------------------------------------------------
# 2. Preflight checks
# --------------------------------------------------
assert OUTPUT_FILE.exists(), f"Output file not found: {OUTPUT_FILE}"
assert MANIFEST_FILE.exists(), f"Manifest file not found: {MANIFEST_FILE}"

manifest = pl.read_csv(MANIFEST_FILE)

assert manifest.height == EXPECTED_MANIFEST_ROWS, (
    f"Unexpected manifest count: {manifest.height}"
)

assert not (
    manifest["anomaly_type"] == ANOMALY_TYPE
).any(), "This anomaly already exists in the manifest."

assert manifest["synthetic_row_id"].n_unique() == manifest.height, (
    "Duplicate synthetic IDs already exist in the manifest."
)

# Count current output rows before modifying anything.
current_output_rows = (
    pl.scan_csv(OUTPUT_FILE, infer_schema_length=1000)
    .select(pl.len())
    .collect()
    .item()
)

assert current_output_rows == EXPECTED_OUTPUT_ROWS, (
    f"Unexpected output count: {current_output_rows}"
)

print(f"Preflight passed. Current output rows: {current_output_rows:,}")
print(f"Current manifest rows: {manifest.height:,}")

# --------------------------------------------------
# 3. Read original Week 1 purchase events only
# --------------------------------------------------
# The original records occupy the first ORIGINAL_WEEK1_ROWS
# rows in the augmented file. Synthetic records are appended later.
source = (
    pl.scan_csv(
        OUTPUT_FILE,
        infer_schema_length=1000,
        schema_overrides={"event_time": pl.String}
    )
    .with_row_index("source_row_number")
    .filter(pl.col("source_row_number") < ORIGINAL_WEEK1_ROWS)
    .filter(
        (pl.col("event_type") == "purchase")
        & pl.col("event_time").is_not_null()
        & pl.col("product_id").is_not_null()
        & pl.col("user_id").is_not_null()
        & pl.col("user_session").is_not_null()
    )
    .select(ALL_COLUMNS + ["source_row_number"])
    .head(TARGET_ROWS)
    .collect()
)

assert source.height == TARGET_ROWS, (
    f"Not enough eligible purchase records: {source.height:,}"
)

print(f"Eligible source purchase events: {source.height:,}")

# --------------------------------------------------
# 4. Create synthetic promotional purchase events
# --------------------------------------------------
# Distribute events across a 30-minute promotional window.
# Multiple events can occur during the same second.
promo_times = [
    (
        PROMO_START
        + timedelta(seconds=i % PROMO_DURATION_SECONDS)
    ).strftime("%Y-%m-%d %H:%M:%S UTC")
    for i in range(TARGET_ROWS)
]

synthetic_ids = [
    f"{ANOMALY_PREFIX}{i:06d}"
    for i in range(1, TARGET_ROWS + 1)
]

synthetic_rows = source.with_columns(
    pl.Series("event_time", promo_times),
    pl.lit("purchase").alias("event_type"),
    pl.lit("seasonal_promotion").alias("marketing_channel"),
    pl.Series("synthetic_row_id", synthetic_ids)
)

# Build the manifest using the original source references.
manifest_chunk = synthetic_rows.select(
    pl.lit("Week 1").alias("week"),
    pl.lit(ANOMALY_TYPE).alias("anomaly_type"),
    pl.col("synthetic_row_id"),
    (pl.col("source_row_number") + 1).alias("source_row_number"),
    pl.lit(None, dtype=pl.String).alias("missing_column"),
    pl.col("event_time").alias("source_event_time"),
    pl.col("event_type").alias("source_event_type"),
    pl.col("user_id").cast(pl.String).alias("source_user_id"),
    pl.col("product_id").cast(pl.String).alias("source_product_id")
)

# The manifest should record ORIGINAL source values, not the
# modified synthetic event values. Retrieve them from the source
# before changing event_time/event_type.
source_manifest = source.select(
    pl.Series("synthetic_row_id", synthetic_ids),
    (pl.col("source_row_number") + 1).alias("source_row_number"),
    pl.col("event_time").alias("source_event_time"),
    pl.col("event_type").alias("source_event_type"),
    pl.col("user_id").cast(pl.String).alias("source_user_id"),
    pl.col("product_id").cast(pl.String).alias("source_product_id")
).with_columns(
    pl.lit("Week 1").alias("week"),
    pl.lit(ANOMALY_TYPE).alias("anomaly_type"),
    pl.lit(None, dtype=pl.String).alias("missing_column")
).select(manifest.columns)

# Keep the synthetic output schema unchanged.
synthetic_output = synthetic_rows.select(ALL_COLUMNS)

# --------------------------------------------------
# 5. Write temporary files before appending
# --------------------------------------------------
TEMP_OUTPUT = BASE_DIR / "seasonal_promotion_chunk.csv"
TEMP_MANIFEST = BASE_DIR / "seasonal_promotion_manifest.csv"

synthetic_output.write_csv(TEMP_OUTPUT)
source_manifest.write_csv(TEMP_MANIFEST)

# Validate the prepared chunks before touching the main files.
assert pl.read_csv(TEMP_OUTPUT).height == TARGET_ROWS
assert pl.read_csv(TEMP_MANIFEST).height == TARGET_ROWS

assert (
    pl.read_csv(TEMP_OUTPUT)["marketing_channel"]
    == "seasonal_promotion"
).all()

assert (
    pl.read_csv(TEMP_OUTPUT)["event_type"] == "purchase"
).all()

print("Temporary chunks created and validated.")

# --------------------------------------------------
# 6. Append synthetic events and manifest records
# --------------------------------------------------
# Back up the manifest first. The original source file is untouched.
MANIFEST_BACKUP = BASE_DIR / "anomaly_manifest_before_SP.csv"
shutil.copy2(MANIFEST_FILE, MANIFEST_BACKUP)

try:
    # Append event rows without adding a second CSV header.
    with OUTPUT_FILE.open("ab") as destination:
        with TEMP_OUTPUT.open("rb") as chunk:
            shutil.copyfileobj(chunk, destination)

    # Append manifest rows without adding a second CSV header.
    with MANIFEST_FILE.open("ab") as destination:
        with TEMP_MANIFEST.open("rb") as chunk:
            with chunk.open("rb") as source_file:
                next(source_file, None)  # Skip the chunk header
                shutil.copyfileobj(source_file, destination)

except Exception:
    print(
        "An append operation failed. Do not rerun this cell blindly. "
        "Inspect the output and manifest before recovery."
    )
    raise

print("Synthetic promotional events appended.")


Preflight passed. Current output rows: 9,179,518
Current manifest rows: 350,203
Eligible source purchase events: 42,449
Temporary chunks created and validated.
An append operation failed. Do not rerun this cell blindly. Inspect the output and manifest before recovery.


AttributeError: '_io.BufferedReader' object has no attribute 'open'

In [16]:

import polars as pl
from pathlib import Path

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")
OUTPUT_FILE = BASE_DIR / "2019-Oct_augmented.csv"
MANIFEST_FILE = BASE_DIR / "anomaly_manifest.csv"

# Count the current output rows.
output_rows = (
    pl.scan_csv(OUTPUT_FILE, infer_schema_length=1000)
    .select(pl.len())
    .collect()
    .item()
)

# Inspect the manifest.
manifest = pl.read_csv(MANIFEST_FILE)

manifest_rows = manifest.height
promotion_rows = manifest.filter(
    pl.col("anomaly_type") == "Seasonal Promotional Purchase Surges"
).height

print(f"Current output rows: {output_rows:,}")
print(f"Current manifest rows: {manifest_rows:,}")
print(f"Promotional records in manifest: {promotion_rows:,}")

if output_rows == 9_179_518 and manifest_rows == 350_203:
    print("Status: Neither file appears to have been appended.")
elif output_rows == 9_221_967 and manifest_rows == 350_203:
    print("Status: Output appended; manifest still needs recovery.")
elif output_rows == 9_221_967 and manifest_rows == 392_652:
    print("Status: Both files appear to have been appended.")
else:
    print("Status: Unexpected counts. Do not append anything yet.")


Current output rows: 9,221,968
Current manifest rows: 350,203
Promotional records in manifest: 0
Status: Unexpected counts. Do not append anything yet.


In [18]:

from pathlib import Path
import csv

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

OUTPUT_FILE = BASE_DIR / "2019-Oct_augmented.csv"
TEMP_OUTPUT = BASE_DIR / "seasonal_promotion_chunk.csv"
TEMP_MANIFEST = BASE_DIR / "seasonal_promotion_manifest.csv"

# Read the CSV header.
with OUTPUT_FILE.open("r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)

print("Header columns:", len(header))
print(header)

# Inspect the final 10 physical lines without parsing the whole CSV.
with OUTPUT_FILE.open("rb") as f:
    f.seek(0, 2)
    size = f.tell()
    f.seek(max(0, size - 5000))
    tail_bytes = f.read()

tail_text = tail_bytes.decode("utf-8", errors="replace")
lines = tail_text.splitlines()

print("\n=== Last 10 Physical Lines ===")
for line in lines[-10:]:
    print(line[:300])

print("\n=== Temporary Files ===")
for path in [TEMP_OUTPUT, TEMP_MANIFEST]:
    print(f"{path.name}: exists={path.exists()}")
    if path.exists():
        print(f"Size: {path.stat().st_size:,} bytes")

# Count fields in the final CSV records.
print("\n=== CSV Field Count Check ===")
for i, line in enumerate(lines[-5:], start=1):
    try:
        parsed = next(csv.reader([line]))
        print(f"Tail line {i}: {len(parsed)} fields")
    except Exception as exc:
        print(f"Tail line {i}: parsing error: {exc}")


Header columns: 11
['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session', 'marketing_channel', 'platform']

=== Last 10 Physical Lines ===
2019-10-05 12:17:19 UTC,purchase,1002524,2053013555631882655,electronics.smartphone,apple,513.43,512739992,04045588-1fe6-455d-acfb-6818197de5dc,seasonal_promotion,Web
2019-10-05 12:17:20 UTC,purchase,1004655,2053013555631882655,electronics.smartphone,samsung,794.59,535100558,239662a0-cc41-463f-88c2-b3346efd33f0,seasonal_promotion,Web
2019-10-05 12:17:21 UTC,purchase,1004768,2053013555631882655,electronics.smartphone,samsung,253.13,513237198,ea333132-6a90-4b17-bf6a-5a51777213fb,seasonal_promotion,Web
2019-10-05 12:17:22 UTC,purchase,19200106,2053013556202308035,construction.tools.saw,stihl,306.31,513127542,8b441794-acca-4d63-9e46-b7f068b0d2f8,seasonal_promotion,Web
2019-10-05 12:17:23 UTC,purchase,1307237,2053013558920217191,computers.notebook,lenovo,257.38,519038834,5b97bcb4-5af2-4eb7-

In [19]:

from pathlib import Path
import polars as pl

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

MANIFEST_FILE = BASE_DIR / "anomaly_manifest.csv"
TEMP_MANIFEST = BASE_DIR / "seasonal_promotion_manifest.csv"
TEMP_OUTPUT = BASE_DIR / "seasonal_promotion_chunk.csv"

manifest = pl.read_csv(MANIFEST_FILE)
temp_manifest = pl.read_csv(TEMP_MANIFEST)
temp_output = pl.read_csv(TEMP_OUTPUT)

campaign_in_manifest = manifest.filter(
    pl.col("anomaly_type") == "Seasonal Promotional Purchase Surges"
).height

print(f"Current manifest rows: {manifest.height:,}")
print(f"Campaign records in manifest: {campaign_in_manifest:,}")
print(f"Temporary manifest rows: {temp_manifest.height:,}")
print(f"Temporary output rows: {temp_output.height:,}")

print("\nTemporary manifest columns:")
print(temp_manifest.columns)

print("\nFirst 3 synthetic IDs:")
print(temp_manifest["synthetic_row_id"].head(3).to_list())

print("\nLast 3 synthetic IDs:")
print(temp_manifest["synthetic_row_id"].tail(3).to_list())

print("\nTemporary manifest anomaly counts:")
print(temp_manifest.group_by("anomaly_type").len())


Current manifest rows: 350,203
Campaign records in manifest: 0
Temporary manifest rows: 42,449
Temporary output rows: 42,449

Temporary manifest columns:
['week', 'anomaly_type', 'synthetic_row_id', 'source_row_number', 'missing_column', 'source_event_time', 'source_event_type', 'source_user_id', 'source_product_id']

First 3 synthetic IDs:
['W1-SP-000001', 'W1-SP-000002', 'W1-SP-000003']

Last 3 synthetic IDs:
['W1-SP-042447', 'W1-SP-042448', 'W1-SP-042449']

Temporary manifest anomaly counts:
shape: (1, 2)
┌─────────────────────────────────┬───────┐
│ anomaly_type                    ┆ len   │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════════════════════╪═══════╡
│ Seasonal Promotional Purchase … ┆ 42449 │
└─────────────────────────────────┴───────┘


In [20]:

from pathlib import Path
import shutil
import polars as pl

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

MANIFEST_FILE = BASE_DIR / "anomaly_manifest.csv"
TEMP_MANIFEST = BASE_DIR / "seasonal_promotion_manifest.csv"
BACKUP_FILE = BASE_DIR / "anomaly_manifest_before_SP_recovery.csv"

ANOMALY_TYPE = "Seasonal Promotional Purchase Surges"
TARGET_ROWS = 42_449

# Load and validate both manifest files.
manifest = pl.read_csv(MANIFEST_FILE)
temp_manifest = pl.read_csv(TEMP_MANIFEST)

assert manifest.height == 350_203, "Unexpected main manifest count."
assert temp_manifest.height == TARGET_ROWS, "Unexpected temporary manifest count."

assert (
    manifest["anomaly_type"] == ANOMALY_TYPE
).sum() == 0, "Campaign records already exist. Do not append again."

assert (
    temp_manifest["anomaly_type"] == ANOMALY_TYPE
).all(), "Unexpected anomaly type in temporary manifest."

assert temp_manifest["synthetic_row_id"].n_unique() == TARGET_ROWS
assert temp_manifest["synthetic_row_id"][0] == "W1-SP-000001"
assert temp_manifest["synthetic_row_id"][-1] == "W1-SP-042449"
assert temp_manifest.columns == manifest.columns

# Create a backup before modifying the manifest.
shutil.copy2(MANIFEST_FILE, BACKUP_FILE)

# Append records without repeating the CSV header.
with MANIFEST_FILE.open("ab") as destination:
    with TEMP_MANIFEST.open("rb") as source_file:
        next(source_file, None)
        shutil.copyfileobj(source_file, destination)

# Verify the result.
final_manifest = pl.read_csv(MANIFEST_FILE)

assert final_manifest.height == 392_652
assert (
    final_manifest["anomaly_type"] == ANOMALY_TYPE
).sum() == TARGET_ROWS
assert final_manifest["synthetic_row_id"].n_unique() == final_manifest.height

print("Manifest recovery completed successfully.")
print(f"Final manifest rows: {final_manifest.height:,}")
print(f"Promotional records: {TARGET_ROWS:,}")
print(f"Backup created: {BACKUP_FILE.name}")


Manifest recovery completed successfully.
Final manifest rows: 392,652
Promotional records: 42,449
Backup created: anomaly_manifest_before_SP_recovery.csv


In [21]:

from pathlib import Path
import polars as pl

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")
OUTPUT_FILE = BASE_DIR / "2019-Oct_augmented.csv"

# Read only the columns needed for validation.
promo = (
    pl.scan_csv(
        OUTPUT_FILE,
        infer_schema_length=1000,
        schema_overrides={"event_time": pl.String}
    )
    .filter(pl.col("marketing_channel") == "seasonal_promotion")
    .select(["event_time", "event_type", "marketing_channel"])
    .collect()
)

print(f"Promotional rows in output: {promo.height:,}")

print(
    "Purchase events:",
    promo.filter(pl.col("event_type") == "purchase").height
)

print(
    "Distinct timestamps:",
    promo["event_time"].n_unique()
)

print("First timestamp:", promo["event_time"].min())
print("Last timestamp:", promo["event_time"].max())


Promotional rows in output: 42,449
Purchase events: 42449
Distinct timestamps: 1800
First timestamp: 2019-10-05 12:00:00 UTC
Last timestamp: 2019-10-05 12:29:59 UTC


In [22]:

from pathlib import Path

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")
OUTPUT_FILE = BASE_DIR / "2019-Oct_augmented.csv"

expected_header = (
    "event_time,event_type,product_id,category_id,category_code,"
    "brand,price,user_id,user_session,marketing_channel,platform"
).encode("utf-8")

header_count = 0
line_count = 0

with OUTPUT_FILE.open("rb") as file:
    for line in file:
        line_count += 1

        if line.rstrip(b"\r\n") == expected_header:
            header_count += 1

print(f"Physical lines scanned: {line_count:,}")
print(f"Matching header lines: {header_count}")

if header_count == 1:
    print("No repeated header detected.")
elif header_count > 1:
    print(f"Found {header_count - 1} repeated header line(s).")
else:
    print("Warning: The expected header was not found.")


Physical lines scanned: 9,221,969
Matching header lines: 2
Found 1 repeated header line(s).


In [23]:

from pathlib import Path
import shutil
import os

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

OUTPUT_FILE = BASE_DIR / "2019-Oct_augmented.csv"
BACKUP_FILE = BASE_DIR / "2019-Oct_augmented_before_header_fix.csv"
TEMP_FILE = BASE_DIR / "2019-Oct_augmented_fixed.tmp"

EXPECTED_HEADER = (
    b"event_time,event_type,product_id,category_id,category_code,"
    b"brand,price,user_id,user_session,marketing_channel,platform"
)

# Create a backup before changing the augmented dataset.
if not BACKUP_FILE.exists():
    shutil.copy2(OUTPUT_FILE, BACKUP_FILE)

# Remove repeated header lines while preserving all other lines.
header_count = 0
kept_lines = 0

with OUTPUT_FILE.open("rb") as source, TEMP_FILE.open("wb") as target:
    for line in source:
        if line.rstrip(b"\r\n") == EXPECTED_HEADER:
            header_count += 1

            # Keep the first header only.
            if header_count == 1:
                target.write(line)
                kept_lines += 1
            continue

        target.write(line)
        kept_lines += 1

assert header_count == 2, (
    f"Expected exactly 2 matching headers, found {header_count}. "
    "Original output file was not replaced."
)

# Replace the augmented file only after successful completion.
os.replace(TEMP_FILE, OUTPUT_FILE)

print("Repeated header removed successfully.")
print(f"Matching headers found: {header_count}")
print(f"Lines retained: {kept_lines:,}")
print(f"Backup: {BACKUP_FILE.name}")


Repeated header removed successfully.
Matching headers found: 2
Lines retained: 9,221,968
Backup: 2019-Oct_augmented_before_header_fix.csv


In [24]:

from pathlib import Path

OUTPUT_FILE = Path(
    r"D:\Samsung\Graduation Project\Data\Old Data\2019-Oct_augmented.csv"
)

EXPECTED_HEADER = (
    b"event_time,event_type,product_id,category_id,category_code,"
    b"brand,price,user_id,user_session,marketing_channel,platform"
)

header_count = 0
line_count = 0

with OUTPUT_FILE.open("rb") as file:
    for line in file:
        line_count += 1
        if line.rstrip(b"\r\n") == EXPECTED_HEADER:
            header_count += 1

print(f"Physical lines: {line_count:,}")
print(f"Header count: {header_count}")

assert header_count == 1, "Repeated header still exists."
print("Validation passed.")


Physical lines: 9,221,968
Header count: 1
Validation passed.


In [25]:
from pathlib import Path
import polars as pl

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

OUTPUT_FILE = BASE_DIR / "2019-Oct_augmented.csv"
MANIFEST_FILE = BASE_DIR / "anomaly_manifest.csv"

EXPECTED_OUTPUT_ROWS = 9_221_967
EXPECTED_MANIFEST_ROWS = 392_652
CART_ABANDONMENT_TARGET = 31_837

# Check the current output row count.
output_rows = (
    pl.scan_csv(OUTPUT_FILE)
    .select(pl.len().alias("rows"))
    .collect()
    .item()
)

# Check the manifest row count and existing Cart Abandonment records.
manifest = pl.scan_csv(MANIFEST_FILE)

manifest_rows = manifest.select(pl.len()).collect().item()

cart_abandonment_rows = (
    manifest
    .filter(
        pl.col("anomaly_type") == "Cart Abandonment Scenarios"
    )
    .select(pl.len())
    .collect()
    .item()
)

print(f"Output rows: {output_rows:,}")
print(f"Manifest rows: {manifest_rows:,}")
print(f"Existing Cart Abandonment records: {cart_abandonment_rows:,}")

# Stop if the files do not match the expected state.
assert output_rows == EXPECTED_OUTPUT_ROWS, "Unexpected output row count."
assert manifest_rows == EXPECTED_MANIFEST_ROWS, "Unexpected manifest row count."
assert cart_abandonment_rows == 0, "Cart Abandonment was already injected."

print("Preflight validation passed. Ready for Cart Abandonment injection.")

Output rows: 9,221,967
Manifest rows: 392,652
Existing Cart Abandonment records: 0
Preflight validation passed. Ready for Cart Abandonment injection.


In [26]:
from pathlib import Path
import polars as pl

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

MANIFEST_FILE = BASE_DIR / "anomaly_manifest.csv"
OUTPUT_FILE = BASE_DIR / "2019-Oct_augmented.csv"
ORIGINAL_FILE = BASE_DIR / "2019-Oct.csv"

# Inspect the existing manifest structure.
manifest = pl.read_csv(
    MANIFEST_FILE,
    n_rows=5,
    infer_schema_length=1000
)

print("Manifest columns:")
print(manifest.columns)

print("\nManifest sample:")
print(manifest)

# Inspect the augmented dataset schema.
output_sample = pl.read_csv(
    OUTPUT_FILE,
    n_rows=3,
    infer_schema_length=1000
)

print("\nOutput columns:")
print(output_sample.columns)

print("\nOutput sample:")
print(output_sample)

# Confirm the original dataset is still available.
print("\nOriginal dataset exists:", ORIGINAL_FILE.exists())

Manifest columns:
['week', 'anomaly_type', 'synthetic_row_id', 'source_row_number', 'missing_column', 'source_event_time', 'source_event_type', 'source_user_id', 'source_product_id']

Manifest sample:
shape: (5, 9)
┌────────┬────────────┬────────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ week   ┆ anomaly_ty ┆ synthetic_ ┆ source_ro ┆ … ┆ source_ev ┆ source_ev ┆ source_us ┆ source_pr │
│ ---    ┆ pe         ┆ row_id     ┆ w_number  ┆   ┆ ent_time  ┆ ent_type  ┆ er_id     ┆ oduct_id  │
│ str    ┆ ---        ┆ ---        ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│        ┆ str        ┆ str        ┆ i64       ┆   ┆ str       ┆ str       ┆ i64       ┆ i64       │
╞════════╪════════════╪════════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Week 1 ┆ Missing    ┆ W1-MD-0000 ┆ 42        ┆ … ┆ 2019-10-0 ┆ view      ┆ 513798668 ┆ 28717908  │
│        ┆ Data       ┆ 01         ┆           ┆   ┆ 1         ┆           ┆  

In [27]:
from pathlib import Path
import polars as pl

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")
OUTPUT_FILE = BASE_DIR / "2019-Oct_augmented.csv"

# Inspect existing synthetic attributes.
df = pl.scan_csv(OUTPUT_FILE)

print("Marketing channel distribution:")
print(
    df.group_by("marketing_channel")
    .len()
    .sort("len", descending=True)
    .collect()
)

print("\nPlatform distribution:")
print(
    df.group_by("platform")
    .len()
    .sort("len", descending=True)
    .collect()
)

print("\nEvent type distribution:")
print(
    df.group_by("event_type")
    .len()
    .sort("len", descending=True)
    .collect()
)

Marketing channel distribution:
shape: (8, 2)
┌────────────────────┬─────────┐
│ marketing_channel  ┆ len     │
│ ---                ┆ ---     │
│ str                ┆ u32     │
╞════════════════════╪═════════╡
│ Referral           ┆ 1538993 │
│ Social Media       ┆ 1529568 │
│ Paid Search        ┆ 1527122 │
│ Email              ┆ 1527017 │
│ Direct             ┆ 1520423 │
│ Organic Search     ┆ 1517100 │
│ seasonal_promotion ┆ 42449   │
│ null               ┆ 19295   │
└────────────────────┴─────────┘

Platform distribution:
shape: (3, 2)
┌────────────┬─────────┐
│ platform   ┆ len     │
│ ---        ┆ ---     │
│ str        ┆ u32     │
╞════════════╪═════════╡
│ Mobile App ┆ 4614376 │
│ Web        ┆ 4588297 │
│ null       ┆ 19294   │
└────────────┴─────────┘

Event type distribution:
shape: (4, 2)
┌────────────┬─────────┐
│ event_type ┆ len     │
│ ---        ┆ ---     │
│ str        ┆ u32     │
╞════════════╪═════════╡
│ view       ┆ 8773943 │
│ cart       ┆ 230132  │
│ purchase   ┆

# Cart Abandonment Scenarios — Week 1

## 1. Objective

Generate **31,837 synthetic cart events** to simulate customers who view a product and add it to their cart without completing a purchase during Week 1 (October 1–7, 2019).

The synthetic records are created separately and validated before being appended to the final augmented dataset.

## 2. Input and Output Files

| File | Purpose |
|---|---|
| `2019-Oct.csv` | Original source dataset; remains unchanged |
| `2019-Oct_augmented.csv` | Dataset containing original and previously injected synthetic events |
| `cart_abandonment_chunk.csv` | Temporary file containing the newly generated cart events |
| `cart_abandonment_manifest.csv` | Temporary manifest documenting the synthetic events and their source records |

## 3. Target Configuration

| Parameter | Value |
|---|---|
| Target records | 31,837 |
| Anomaly type | Cart Abandonment Scenarios |
| Target week | October 1–7, 2019 |
| Source event | `view` |
| Generated event | `cart` |
| Timestamp offset | 1–10 minutes after the source event |
| Synthetic ID prefix | `W1-CA-` |

## 4. Generation Logic

### Step 1: Load the augmented dataset

The dataset is read using Polars Lazy API to reduce unnecessary eager loading of the large CSV file.

A row index is added to identify the source record used for each generated event.

### Step 2: Filter Week 1

Only events with timestamps from October 1, 2019, inclusive, to October 8, 2019, exclusive, are considered.

### Step 3: Identify completed purchases

The pipeline extracts unique combinations of:

- `user_session`
- `product_id`

that have a `purchase` event during Week 1.

### Step 4: Select eligible views

Candidate records must satisfy these conditions:

- `event_type = view`
- `user_session` is not null
- `product_id` is not null
- `user_id` is not null
- The event timestamp is valid
- No matching purchase exists for the same session and product during Week 1

An anti-join excludes views associated with a matching purchase.

### Step 5: Generate synthetic cart events

The pipeline selects 31,837 eligible records and generates a corresponding `cart` event for each one.

Each generated event:

- Preserves the source product, user, session, and product attributes.
- Preserves the existing `marketing_channel` and `platform` values.
- Uses a timestamp 1–10 minutes after the source event.
- Receives a unique synthetic ID, such as `W1-CA-000001`.
- Changes `event_type` to `cart`.

### Step 6: Create the temporary manifest

Each synthetic record is documented with the existing nine-column manifest schema:

| Column | Description |
|---|---|
| `week` | Week containing the synthetic event |
| `anomaly_type` | `Cart Abandonment Scenarios` |
| `synthetic_row_id` | Unique synthetic event identifier |
| `source_row_number` | Row index assigned to the source record in the augmented dataset |
| `missing_column` | Null because this scenario does not introduce missing data |
| `source_event_time` | Timestamp of the original source event |
| `source_event_type` | `view` |
| `source_user_id` | Source user's ID |
| `source_product_id` | Source product's ID |

## 5. Validation and Safety

Before the generation step, the pipeline checks that:

- The original and augmented datasets exist.
- Temporary output files do not already exist.
- Exactly 31,837 eligible source events are available.

If fewer eligible events are found, the pipeline stops without changing the final files.

After generation, the temporary files should be checked to confirm:

- Exactly 31,837 cart events were generated.
- Exactly 31,837 manifest records were generated.
- All generated events have `event_type = cart`.
- Synthetic IDs are unique.
- Required source fields are populated.
- Generated timestamps remain within Week 1.
- No matching purchase exists for the same user session and product in the source data.

## 6. Expected Results

| Metric | Expected value |
|---|---:|
| Synthetic cart events | 31,837 |
| Manifest records | 31,837 |
| Synthetic ID range | `W1-CA-000001` to `W1-CA-031837` |
| Final dataset modification during generation | None |
| Original dataset modification | None |

The generated records remain in temporary files until validation succeeds. Only then should they be appended to the final dataset and manifest.

## 7. Important Interpretation

This scenario represents **synthetic cart abandonment candidates**, not verified real-world abandonment. A session-product pair without a purchase during the selected week may still purchase later or through another session.

For stronger business accuracy, abandonment can instead be defined using a fixed observation window, such as no purchase within 24 hours after the cart event, provided sufficient timestamp coverage is available.

## 8. Implementation Notes

- The code uses the augmented dataset as the source of candidate events. Therefore, `source_row_number` refers to the row index in that dataset, not necessarily the row number in the original CSV.
- The timestamp offset is deterministic, so it is reproducible for a given candidate ordering.
- The temporary files must be validated before appending them to the final files.
- The CSV header must not be appended a second time when merging the temporary chunk.


In [29]:

from pathlib import Path
from datetime import datetime, timezone

import polars as pl

# ============================================================
# 1. Configuration
# ============================================================

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

ORIGINAL_FILE = BASE_DIR / "2019-Oct.csv"
OUTPUT_FILE = BASE_DIR / "2019-Oct_augmented.csv"

CART_TEMP_FILE = BASE_DIR / "cart_abandonment_chunk.csv"
MANIFEST_TEMP_FILE = BASE_DIR / "cart_abandonment_manifest.csv"

TARGET_ROWS = 31_837

WEEK_START = datetime(2019, 10, 1, tzinfo=timezone.utc)
WEEK_END = datetime(2019, 10, 8, tzinfo=timezone.utc)

ORIGINAL_COLUMNS = [
    "event_time", "event_type", "product_id", "category_id",
    "category_code", "brand", "price", "user_id", "user_session"
]

ALL_COLUMNS = ORIGINAL_COLUMNS + ["marketing_channel", "platform"]

MANIFEST_COLUMNS = [
    "week", "anomaly_type", "synthetic_row_id", "source_row_number",
    "missing_column", "source_event_time", "source_event_type",
    "source_user_id", "source_product_id"
]

# ============================================================
# 2. Safety checks
# ============================================================

assert ORIGINAL_FILE.exists(), "Original dataset not found."
assert OUTPUT_FILE.exists(), "Augmented dataset not found."

assert not CART_TEMP_FILE.exists(), (
    f"Temporary cart file already exists: {CART_TEMP_FILE}\n"
    "Inspect or rename it before rerunning this cell."
)

assert not MANIFEST_TEMP_FILE.exists(), (
    f"Temporary manifest already exists: {MANIFEST_TEMP_FILE}\n"
    "Inspect or rename it before rerunning this cell."
)

print("Safety checks passed.")

# ============================================================
# 3. Read the augmented dataset lazily
# ============================================================

events = (
    pl.scan_csv(
        OUTPUT_FILE,
        infer_schema_length=10_000
    )
    .with_row_index("_source_row_number", offset=1)
    .with_columns(
        pl.col("event_time")
        .str.strptime(
            pl.Datetime(time_zone="UTC"),
            format="%Y-%m-%d %H:%M:%S UTC",
            strict=False
        )
        .alias("_event_dt")
    )
)

# ============================================================
# 4. Filter Week 1
# ============================================================

week1 = events.filter(
    (pl.col("_event_dt") >= pl.lit(WEEK_START))
    & (pl.col("_event_dt") < pl.lit(WEEK_END))
)

# ============================================================
# 5. Identify session-product pairs with purchases
# ============================================================

purchase_pairs = (
    week1
    .filter(pl.col("event_type") == "purchase")
    .filter(
        pl.col("user_session").is_not_null()
        & pl.col("product_id").is_not_null()
    )
    .select("user_session", "product_id")
    .unique()
)

# ============================================================
# 6. Select eligible view events
# ============================================================

candidates = (
    week1
    .filter(pl.col("event_type") == "view")
    .filter(
        pl.col("user_session").is_not_null()
        & pl.col("product_id").is_not_null()
        & pl.col("user_id").is_not_null()
        & pl.col("_event_dt").is_not_null()
        # Keep generated events within Week 1.
        & (
            pl.col("_event_dt")
            < pl.lit(datetime(2019, 10, 7, 23, 50, tzinfo=timezone.utc))
        )
    )
    .join(
        purchase_pairs,
        on=["user_session", "product_id"],
        how="anti"
    )
    .sort("_source_row_number")
    .head(TARGET_ROWS)
    .collect()
)

print(f"Eligible source views: {candidates.height:,}")

if candidates.height != TARGET_ROWS:
    raise RuntimeError(
        f"Expected {TARGET_ROWS:,} candidates, "
        f"but found {candidates.height:,}. "
        "No temporary or final files were written by this cell."
    )

# ============================================================
# 7. Generate synthetic cart events
# ============================================================

# Deterministic offsets: 1 through 10 minutes.
offset_seconds = (
    (pl.int_range(0, candidates.height, eager=True) % 10 + 1) * 60
)

synthetic_ids = [
    f"W1-CA-{i:06d}"
    for i in range(1, candidates.height + 1)
]

cart_rows = (
    candidates
    .with_columns(
        (
            pl.col("_event_dt")
            + pl.duration(seconds=offset_seconds)
        ).alias("_cart_dt")
    )
    .with_columns(
        pl.col("_cart_dt")
        .dt.strftime("%Y-%m-%d %H:%M:%S UTC")
        .alias("event_time"),
        pl.lit("cart").alias("event_type"),
        pl.Series("_synthetic_id", synthetic_ids)
    )
    .select(
        *ALL_COLUMNS,
        "_source_row_number",
        "_synthetic_id",
        "_event_dt",
        "_cart_dt"
    )
)

# Validate timestamps before writing.
invalid_timestamps = cart_rows.filter(
    (pl.col("_cart_dt") < pl.lit(WEEK_START))
    | (pl.col("_cart_dt") >= pl.lit(WEEK_END))
).height

if invalid_timestamps:
    raise RuntimeError(
        f"{invalid_timestamps} generated timestamps fall outside Week 1."
    )

# ============================================================
# 8. Build the manifest chunk
# ============================================================

manifest_chunk = (
    cart_rows
    .select(
        pl.lit("Week 1").alias("week"),
        pl.lit("Cart Abandonment Scenarios").alias("anomaly_type"),
        pl.col("_synthetic_id").alias("synthetic_row_id"),
        pl.col("_source_row_number")
        .cast(pl.Int64)
        .alias("source_row_number"),
        pl.lit(None, dtype=pl.String).alias("missing_column"),
        pl.col("_event_dt")
        .dt.strftime("%Y-%m-%d %H:%M:%S UTC")
        .alias("source_event_time"),
        pl.lit("view").alias("source_event_type"),
        pl.col("user_id")
        .cast(pl.Int64, strict=False)
        .alias("source_user_id"),
        pl.col("product_id")
        .cast(pl.Int64, strict=False)
        .alias("source_product_id")
    )
    .select(MANIFEST_COLUMNS)
)

# ============================================================
# 9. Validate the generated records
# ============================================================

assert cart_rows.height == TARGET_ROWS
assert manifest_chunk.height == TARGET_ROWS

assert cart_rows["event_type"].unique().to_list() == ["cart"]
assert cart_rows["_synthetic_id"].n_unique() == TARGET_ROWS
assert manifest_chunk["synthetic_row_id"].n_unique() == TARGET_ROWS

# ============================================================
# 10. Write temporary files only
# ============================================================

cart_rows.select(ALL_COLUMNS).write_csv(CART_TEMP_FILE)
manifest_chunk.write_csv(MANIFEST_TEMP_FILE)

print("\nGeneration completed successfully.")
print(f"Generated cart events: {cart_rows.height:,}")
print(f"Generated manifest records: {manifest_chunk.height:,}")
print(f"Cart chunk: {CART_TEMP_FILE.name}")
print(f"Manifest chunk: {MANIFEST_TEMP_FILE.name}")
print("Final dataset and final manifest have NOT been modified.")


Safety checks passed.
Eligible source views: 31,837

Generation completed successfully.
Generated cart events: 31,837
Generated manifest records: 31,837
Cart chunk: cart_abandonment_chunk.csv
Manifest chunk: cart_abandonment_manifest.csv
Final dataset and final manifest have NOT been modified.


In [30]:

from pathlib import Path
import polars as pl

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

CART_TEMP_FILE = BASE_DIR / "cart_abandonment_chunk.csv"
MANIFEST_TEMP_FILE = BASE_DIR / "cart_abandonment_manifest.csv"

TARGET_ROWS = 31_837

EXPECTED_CART_COLUMNS = [
    "event_time", "event_type", "product_id", "category_id",
    "category_code", "brand", "price", "user_id", "user_session",
    "marketing_channel", "platform"
]

EXPECTED_MANIFEST_COLUMNS = [
    "week", "anomaly_type", "synthetic_row_id", "source_row_number",
    "missing_column", "source_event_time", "source_event_type",
    "source_user_id", "source_product_id"
]

cart = pl.read_csv(CART_TEMP_FILE, infer_schema_length=10_000)
manifest = pl.read_csv(MANIFEST_TEMP_FILE, infer_schema_length=10_000)

# Check row counts and schemas.
assert cart.height == TARGET_ROWS, "Incorrect cart row count."
assert manifest.height == TARGET_ROWS, "Incorrect manifest row count."
assert cart.columns == EXPECTED_CART_COLUMNS, "Cart schema mismatch."
assert manifest.columns == EXPECTED_MANIFEST_COLUMNS, "Manifest schema mismatch."

# Check event types and synthetic IDs.
assert cart["event_type"].unique().to_list() == ["cart"]
assert manifest["anomaly_type"].unique().to_list() == [
    "Cart Abandonment Scenarios"
]
assert manifest["source_event_type"].unique().to_list() == ["view"]
assert cart["event_type"].null_count() == 0
assert manifest["synthetic_row_id"].n_unique() == TARGET_ROWS

expected_ids = [
    f"W1-CA-{i:06d}" for i in range(1, TARGET_ROWS + 1)
]
assert cart.height == len(expected_ids)
assert manifest["synthetic_row_id"].to_list() == expected_ids

# Verify the two files have matching IDs and event counts.
assert (
    cart.height == manifest.height
), "Cart and manifest counts do not match."

print("All temporary-file checks passed.")
print(f"Cart rows: {cart.height:,}")
print(f"Manifest rows: {manifest.height:,}")
print(f"Cart columns: {len(cart.columns)}")
print(f"Manifest columns: {len(manifest.columns)}")
print("Ready for the separate merge-and-validation stage.")


All temporary-file checks passed.
Cart rows: 31,837
Manifest rows: 31,837
Cart columns: 11
Manifest columns: 9
Ready for the separate merge-and-validation stage.


In [31]:

from pathlib import Path
from shutil import copy2
import polars as pl
import csv

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

OUTPUT_FILE = BASE_DIR / "2019-Oct_augmented.csv"
MANIFEST_FILE = BASE_DIR / "anomaly_manifest.csv"

CART_TEMP_FILE = BASE_DIR / "cart_abandonment_chunk.csv"
MANIFEST_TEMP_FILE = BASE_DIR / "cart_abandonment_manifest.csv"

OUTPUT_BACKUP = BASE_DIR / "2019-Oct_augmented_before_CA_merge.csv"
MANIFEST_BACKUP = BASE_DIR / "anomaly_manifest_before_CA_merge.csv"

TARGET_ROWS = 31_837

# ------------------------------------------------------------
# 1. Preflight checks
# ------------------------------------------------------------

for file_path in [
    OUTPUT_FILE, MANIFEST_FILE, CART_TEMP_FILE, MANIFEST_TEMP_FILE
]:
    assert file_path.exists(), f"Missing file: {file_path.name}"

assert not OUTPUT_BACKUP.exists(), (
    "Output backup already exists. Inspect it before proceeding."
)
assert not MANIFEST_BACKUP.exists(), (
    "Manifest backup already exists. Inspect it before proceeding."
)

cart = pl.read_csv(CART_TEMP_FILE, infer_schema_length=10_000)
manifest_chunk = pl.read_csv(
    MANIFEST_TEMP_FILE, infer_schema_length=10_000
)
existing_manifest = pl.read_csv(
    MANIFEST_FILE, infer_schema_length=10_000
)

assert cart.height == TARGET_ROWS
assert manifest_chunk.height == TARGET_ROWS

assert not (
    existing_manifest["anomaly_type"]
    == "Cart Abandonment Scenarios"
).any(), "Cart Abandonment records already exist in the final manifest."

# Validate output schema before changing anything.
output_header = next(csv.reader(
    open(OUTPUT_FILE, "r", encoding="utf-8", newline="")
))
assert output_header == cart.columns, "Final output schema mismatch."

print("Preflight passed.")
print("Creating backups...")

# ------------------------------------------------------------
# 2. Create backups
# ------------------------------------------------------------

copy2(OUTPUT_FILE, OUTPUT_BACKUP)
copy2(MANIFEST_FILE, MANIFEST_BACKUP)

print(f"Output backup: {OUTPUT_BACKUP.name}")
print(f"Manifest backup: {MANIFEST_BACKUP.name}")

# ------------------------------------------------------------
# 3. Append cart records without a second CSV header
# ------------------------------------------------------------

# Read the temporary chunk as text, skip its first line (header),
# and append only its data lines.
with open(CART_TEMP_FILE, "r", encoding="utf-8", newline="") as src:
    next(src)  # Skip the temporary CSV header.
    with open(OUTPUT_FILE, "a", encoding="utf-8", newline="") as dst:
        for line in src:
            dst.write(line)

print("Cart records appended to augmented dataset.")

# ------------------------------------------------------------
# 4. Append manifest records without a second header
# ------------------------------------------------------------

with open(MANIFEST_TEMP_FILE, "r", encoding="utf-8", newline="") as src:
    next(src)  # Skip the temporary manifest header.
    with open(MANIFEST_FILE, "a", encoding="utf-8", newline="") as dst:
        for line in src:
            dst.write(line)

print("Manifest records appended.")

# ------------------------------------------------------------
# 5. Validate final counts and headers
# ------------------------------------------------------------

final_manifest = pl.read_csv(MANIFEST_FILE, infer_schema_length=10_000)

ca_count = (
    final_manifest
    .filter(pl.col("anomaly_type") == "Cart Abandonment Scenarios")
    .height
)

assert ca_count == TARGET_ROWS, (
    f"Expected {TARGET_ROWS:,} Cart Abandonment manifest records, "
    f"found {ca_count:,}."
)

# Count physical CSV lines and repeated headers in the large output.
header_count = 0
physical_lines = 0

with open(OUTPUT_FILE, "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    expected_header = next(reader)

    for row in reader:
        physical_lines += 1
        if row == expected_header:
            header_count += 1

assert header_count == 0, (
    f"Found {header_count} repeated header rows in final dataset."
)

print("\nMerge validation passed.")
print(f"Cart Abandonment manifest records: {ca_count:,}")
print(f"Repeated headers found: {header_count}")
print(f"Physical data rows in augmented CSV: {physical_lines:,}")
print("Backups are preserved.")


Preflight passed.
Creating backups...
Output backup: 2019-Oct_augmented_before_CA_merge.csv
Manifest backup: anomaly_manifest_before_CA_merge.csv
Cart records appended to augmented dataset.
Manifest records appended.

Merge validation passed.
Cart Abandonment manifest records: 31,837
Repeated headers found: 0
Physical data rows in augmented CSV: 9,253,804
Backups are preserved.


In [32]:

from pathlib import Path
import polars as pl

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")
MANIFEST_FILE = BASE_DIR / "anomaly_manifest.csv"

manifest = pl.read_csv(
    MANIFEST_FILE,
    infer_schema_length=10_000
)

summary = (
    manifest
    .group_by(["week", "anomaly_type"])
    .agg(pl.len().alias("record_count"))
    .sort(["week", "anomaly_type"])
)

print(summary)

print("\nTotal manifest records:", f"{manifest.height:,}")
print("Unique synthetic IDs:", f"{manifest['synthetic_row_id'].n_unique():,}")
print("Duplicate synthetic IDs:",
      manifest.height - manifest["synthetic_row_id"].n_unique())


shape: (6, 3)
┌────────┬─────────────────────────────────┬──────────────┐
│ week   ┆ anomaly_type                    ┆ record_count │
│ ---    ┆ ---                             ┆ ---          │
│ str    ┆ str                             ┆ u32          │
╞════════╪═════════════════════════════════╪══════════════╡
│ Week 1 ┆ Bot-like Crawling Loops         ┆ 53061        │
│ Week 1 ┆ Cart Abandonment Scenarios      ┆ 31837        │
│ Week 1 ┆ Duplicate Events                ┆ 42449        │
│ Week 1 ┆ Impossible Sequences            ┆ 42449        │
│ Week 1 ┆ Missing Data                    ┆ 212244       │
│ Week 1 ┆ Seasonal Promotional Purchase … ┆ 42449        │
└────────┴─────────────────────────────────┴──────────────┘

Total manifest records: 424,489
Unique synthetic IDs: 424,489
Duplicate synthetic IDs: 0


In [33]:

from pathlib import Path
import polars as pl
import csv

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

OUTPUT_FILE = BASE_DIR / "2019-Oct_augmented.csv"
MANIFEST_FILE = BASE_DIR / "anomaly_manifest.csv"

EXPECTED_COLUMNS = [
    "event_time", "event_type", "product_id", "category_id",
    "category_code", "brand", "price", "user_id", "user_session",
    "marketing_channel", "platform"
]

EXPECTED_ROWS = 9_253_804

# 1. Check that the final dataset exists.
assert OUTPUT_FILE.exists(), "Final dataset not found."

# 2. Read the dataset lazily and count rows.
data = pl.scan_csv(
    OUTPUT_FILE,
    infer_schema_length=10_000
)

actual_rows = data.select(pl.len()).collect().item()
actual_columns = data.collect_schema().names()

# 3. Check the schema.
assert actual_columns == EXPECTED_COLUMNS, (
    f"Column mismatch.\nActual columns: {actual_columns}"
)

# 4. Check the physical CSV header.
with open(OUTPUT_FILE, "r", encoding="utf-8", newline="") as file:
    reader = csv.reader(file)
    header = next(reader)

assert header == EXPECTED_COLUMNS, "CSV header mismatch."

# 5. Check manifest record count.
manifest = pl.read_csv(
    MANIFEST_FILE,
    infer_schema_length=10_000
)

print("Final dataset:", OUTPUT_FILE.name)
print(f"Data rows: {actual_rows:,}")
print(f"Expected rows: {EXPECTED_ROWS:,}")
print(f"Column count: {len(actual_columns)}")
print("Columns:", actual_columns)
print(f"Manifest records: {manifest.height:,}")

assert actual_rows == EXPECTED_ROWS, (
    f"Expected {EXPECTED_ROWS:,} rows, found {actual_rows:,}"
)

print("\nFinal dataset structural validation passed.")


Final dataset: 2019-Oct_augmented.csv
Data rows: 9,253,804
Expected rows: 9,253,804
Column count: 11
Columns: ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session', 'marketing_channel', 'platform']
Manifest records: 424,489

Final dataset structural validation passed.


In [35]:

from pathlib import Path
import polars as pl

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

OUTPUT_FILE = BASE_DIR / "2019-Oct_augmented.csv"
MANIFEST_FILE = BASE_DIR / "anomaly_manifest.csv"
CART_TEMP_FILE = BASE_DIR / "cart_abandonment_chunk.csv"

EXPECTED_COUNTS = {
    "Bot-like Crawling Loops": 53_061,
    "Cart Abandonment Scenarios": 31_837,
    "Duplicate Events": 42_449,
    "Impossible Sequences": 42_449,
    "Missing Data": 212_244,
    "Seasonal Promotional Purchase Surges": 42_449,
}

DATA_COLUMNS = [
    "event_time", "event_type", "product_id", "category_id",
    "category_code", "brand", "price", "user_id", "user_session",
    "marketing_channel", "platform"
]

# ------------------------------------------------------------
# 1. Validate the manifest
# ------------------------------------------------------------

manifest = pl.read_csv(
    MANIFEST_FILE,
    infer_schema_length=10_000
)

assert manifest.height == sum(EXPECTED_COUNTS.values()), (
    "Unexpected total number of manifest records."
)

assert manifest["synthetic_row_id"].n_unique() == manifest.height, (
    "Duplicate synthetic IDs found."
)

actual_counts = {
    row["anomaly_type"]: row["len"]
    for row in (
        manifest.group_by("anomaly_type")
        .agg(pl.len().alias("len"))
        .to_dicts()
    )
}

assert actual_counts == EXPECTED_COUNTS, (
    f"Anomaly count mismatch.\nActual: {actual_counts}"
)

print("PASS: Manifest counts and unique IDs.")

# ------------------------------------------------------------
# 2. Validate the final dataset structure and row count
# ------------------------------------------------------------

final_data = pl.scan_csv(
    OUTPUT_FILE,
    infer_schema_length=10_000
)

final_rows = final_data.select(pl.len()).collect().item()
final_columns = final_data.collect_schema().names()

assert final_rows == 9_253_804, (
    f"Unexpected final row count: {final_rows:,}"
)

assert final_columns == DATA_COLUMNS, (
    f"Unexpected columns: {final_columns}"
)

print(f"PASS: Final dataset has {final_rows:,} rows and 11 columns.")


# ------------------------------------------------------------
# 3. Verify Cart Abandonment rows in the final dataset
# ------------------------------------------------------------

cart_chunk = pl.read_csv(
    CART_TEMP_FILE,
    infer_schema_length=10_000
).select(DATA_COLUMNS)

# Collect only cart events from the final dataset.
final_cart_rows = (
    final_data
    .filter(pl.col("event_type") == "cart")
    .select(DATA_COLUMNS)
    .collect()
)

# Both sides are now DataFrames.
missing_cart_rows = (
    cart_chunk
    .join(
        final_cart_rows,
        on=DATA_COLUMNS,
        how="anti",
        nulls_equal=True
    )
    .height
)

assert missing_cart_rows == 0, (
    f"{missing_cart_rows:,} cart chunk rows were not found "
    "in the final dataset."
)

print(
    "PASS: All cart chunk rows exist in the final dataset."
)

# ------------------------------------------------------------
# 4. Report results
# ------------------------------------------------------------

print("\n=== FINAL QUALITY CHECK ===")
print(f"Final dataset rows: {final_rows:,}")
print(f"Manifest records: {manifest.height:,}")
print(
    "Unique synthetic IDs: "
    f"{manifest['synthetic_row_id'].n_unique():,}"
)
print(f"Unmatched cart chunk rows: {missing_cart_rows:,}")
print("\nAll checks completed successfully.")


PASS: Manifest counts and unique IDs.
PASS: Final dataset has 9,253,804 rows and 11 columns.
PASS: All cart chunk rows exist in the final dataset.

=== FINAL QUALITY CHECK ===
Final dataset rows: 9,253,804
Manifest records: 424,489
Unique synthetic IDs: 424,489
Unmatched cart chunk rows: 0

All checks completed successfully.


In [36]:

import polars as pl
from pathlib import Path

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

OUTPUT_FILE = BASE_DIR / "2019-Oct_augmented.csv"
MANIFEST_FILE = BASE_DIR / "anomaly_manifest.csv"

EXPECTED_COUNT = 42_449

# 1. Validate the manifest
manifest = pl.read_csv(
    MANIFEST_FILE,
    infer_schema_length=10_000
)

sp_manifest = manifest.filter(
    pl.col("anomaly_type") == "Seasonal Promotional Purchase Surges"
)

manifest_count = sp_manifest.height

assert manifest_count == EXPECTED_COUNT, (
    f"Manifest count mismatch: {manifest_count:,}"
)

assert (
    sp_manifest["synthetic_row_id"].n_unique()
    == EXPECTED_COUNT
), "Duplicate synthetic IDs found in seasonal-promotion manifest."

print(f"PASS: Manifest contains {manifest_count:,} records.")

# 2. Validate the actual final dataset
final_data = pl.scan_csv(
    OUTPUT_FILE,
    infer_schema_length=10_000
)

promo = final_data.filter(
    pl.col("marketing_channel") == "seasonal_promotion"
)

promo_stats = promo.select(
    pl.len().alias("row_count"),
    (pl.col("event_type") == "purchase")
        .sum()
        .alias("purchase_count"),
    pl.col("event_time").n_unique()
        .alias("distinct_timestamps")
).collect()

actual_count = promo_stats["row_count"][0]
purchase_count = promo_stats["purchase_count"][0]
distinct_timestamps = promo_stats["distinct_timestamps"][0]

print(f"Promotional rows: {actual_count:,}")
print(f"Purchase events: {purchase_count:,}")
print(f"Distinct timestamps: {distinct_timestamps:,}")

assert actual_count == EXPECTED_COUNT, (
    f"Expected {EXPECTED_COUNT:,} promotional rows, "
    f"found {actual_count:,}"
)

assert purchase_count == EXPECTED_COUNT, (
    "Some promotional rows are not purchase events."
)

print("PASS: Seasonal promotional surge validation.")


PASS: Manifest contains 42,449 records.
Promotional rows: 42,449
Purchase events: 42,449
Distinct timestamps: 1,800
PASS: Seasonal promotional surge validation.


In [37]:

import polars as pl
from pathlib import Path

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

OUTPUT_FILE = BASE_DIR / "2019-Oct_augmented.csv"
MANIFEST_FILE = BASE_DIR / "anomaly_manifest.csv"

EXPECTED_COUNT = 53_061
ANOMALY_NAME = "Bot-like Crawling Loops"

# 1. Validate the manifest count and IDs
manifest = pl.read_csv(
    MANIFEST_FILE,
    infer_schema_length=10_000
)

bot_manifest = manifest.filter(
    pl.col("anomaly_type") == ANOMALY_NAME
)

assert bot_manifest.height == EXPECTED_COUNT, (
    f"Expected {EXPECTED_COUNT:,} manifest rows, "
    f"found {bot_manifest.height:,}"
)

assert (
    bot_manifest["synthetic_row_id"].n_unique()
    == EXPECTED_COUNT
), "Duplicate synthetic IDs found in bot manifest."

assert (
    bot_manifest["source_event_type"] == "view"
).all(), "Some bot manifest source events are not views."

print("PASS: Manifest count, IDs, and source event types.")

# 2. Find synthetic bot events in the final dataset
final_data = pl.scan_csv(
    OUTPUT_FILE,
    infer_schema_length=10_000
)

bot_rows = (
    final_data
    .filter(
        (pl.col("event_type") == "view")
        & pl.col("marketing_channel").is_null()
    )
)

# Collect only the summary, not all event rows.
bot_summary = bot_rows.select(
    pl.len().alias("candidate_view_rows")
).collect()

print(
    "View rows with null marketing_channel:",
    f"{bot_summary['candidate_view_rows'][0]:,}"
)

print(
    "\nManifest bot records:",
    f"{bot_manifest.height:,}"
)

print(
    "Unique synthetic IDs:",
    f"{bot_manifest['synthetic_row_id'].n_unique():,}"
)


PASS: Manifest count, IDs, and source event types.
View rows with null marketing_channel: 18,531

Manifest bot records: 53,061
Unique synthetic IDs: 53,061


In [38]:

bot_manifest.select([
    "synthetic_row_id",
    "source_row_number",
    "source_event_time",
    "source_event_type",
    "source_user_id",
    "source_product_id"
]).head(10)


synthetic_row_id,source_row_number,source_event_time,source_event_type,source_user_id,source_product_id
str,i64,str,str,i64,i64
"""W1-BL-000001""",1,"""2019-10-01 00:00:00 UTC""","""view""",541312140,44600062
"""W1-BL-000002""",161,"""2019-10-01 00:02:13 UTC""","""view""",515535834,3701244
"""W1-BL-000003""",323,"""2019-10-01 00:05:01 UTC""","""view""",529755884,1005115
"""W1-BL-000004""",486,"""2019-10-01 00:08:01 UTC""","""view""",542418394,3601485
"""W1-BL-000005""",650,"""2019-10-01 00:10:37 UTC""","""view""",543624132,1004767
"""W1-BL-000006""",813,"""2019-10-01 00:13:16 UTC""","""view""",520575653,1306359
"""W1-BL-000007""",974,"""2019-10-01 00:16:30 UTC""","""view""",552652462,12703816
"""W1-BL-000008""",1134,"""2019-10-01 01:22:12 UTC""","""view""",542718671,13100841
"""W1-BL-000009""",1294,"""2019-10-01 02:17:40 UTC""","""view""",512702129,26700034


In [39]:

import polars as pl
from datetime import timedelta

# Read a small sample of the bot manifest
sample = bot_manifest.head(100).with_columns(
    pl.col("source_event_time").str.strptime(
        pl.Datetime(time_zone="UTC"),
        format="%Y-%m-%d %H:%M:%S UTC",
        strict=False
    ).alias("source_ts")
)

# Load only the required columns from the final dataset
events = (
    pl.scan_csv(
        OUTPUT_FILE,
        infer_schema_length=10_000
    )
    .select([
        "event_time",
        "event_type",
        "product_id",
        "user_id",
        "user_session"
    ])
    .filter(pl.col("event_type") == "view")
    .collect()
    .with_columns(
        pl.col("event_time").str.strptime(
            pl.Datetime(time_zone="UTC"),
            format="%Y-%m-%d %H:%M:%S UTC",
            strict=False
        ).alias("event_ts")
    )
)

# Match candidates using user and product, then inspect timing.
# Session comparison is not possible until we identify the source session.
results = []

for row in sample.iter_rows(named=True):
    source_ts = row["source_ts"]

    candidates = events.filter(
        (pl.col("user_id") == row["source_user_id"])
        & (pl.col("product_id") == row["source_product_id"])
        & (pl.col("event_ts") > source_ts)
        & (pl.col("event_ts") <= source_ts + timedelta(seconds=3))
    )

    results.append({
        "synthetic_row_id": row["synthetic_row_id"],
        "candidate_count": candidates.height
    })

result_df = pl.DataFrame(results)

print(result_df.head(10))
print("\nSample size:", result_df.height)
print(
    "Sources with at least one matching candidate:",
    result_df.filter(pl.col("candidate_count") > 0).height
)


shape: (10, 2)
┌──────────────────┬─────────────────┐
│ synthetic_row_id ┆ candidate_count │
│ ---              ┆ ---             │
│ str              ┆ i64             │
╞══════════════════╪═════════════════╡
│ W1-BL-000001     ┆ 1               │
│ W1-BL-000002     ┆ 1               │
│ W1-BL-000003     ┆ 1               │
│ W1-BL-000004     ┆ 1               │
│ W1-BL-000005     ┆ 1               │
│ W1-BL-000006     ┆ 1               │
│ W1-BL-000007     ┆ 1               │
│ W1-BL-000008     ┆ 1               │
│ W1-BL-000009     ┆ 1               │
│ W1-BL-000010     ┆ 1               │
└──────────────────┴─────────────────┘

Sample size: 100
Sources with at least one matching candidate: 100


In [40]:

from pathlib import Path

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

for p in sorted(BASE_DIR.glob("*")):
    if p.is_file() and any(
        term in p.name.lower()
        for term in ["bot", "crawling", "bl", "manifest", "augmented"]
    ):
        print(f"{p.name} | {p.stat().st_size:,} bytes")


2019-Oct_augmented.csv | 1,399,716,937 bytes
2019-Oct_augmented_before_CA_merge.csv | 1,394,906,846 bytes
2019-Oct_augmented_before_header_fix.csv | 1,394,906,965 bytes
anomaly_manifest.csv | 42,832,282 bytes
anomaly_manifest_before_CA_merge.csv | 39,614,685 bytes
anomaly_manifest_before_SP.csv | 34,657,486 bytes
anomaly_manifest_before_SP_recovery.csv | 34,657,486 bytes
cart_abandonment_manifest.csv | 3,217,734 bytes
seasonal_promotion_manifest.csv | 4,957,336 bytes


In [41]:

import polars as pl
from pathlib import Path

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

OLD_MANIFEST = BASE_DIR / "anomaly_manifest_before_SP.csv"
FINAL_MANIFEST = BASE_DIR / "anomaly_manifest.csv"

def bot_summary(path):
    df = pl.read_csv(path, infer_schema_length=10_000)
    bot = df.filter(
        pl.col("anomaly_type") == "Bot-like Crawling Loops"
    )

    print(f"\nFile: {path.name}")
    print(f"Total manifest rows: {df.height:,}")
    print(f"Bot manifest rows: {bot.height:,}")
    print("Columns:", df.columns)

    if bot.height:
        print(
            bot.select([
                "synthetic_row_id",
                "source_row_number",
                "source_event_time",
                "source_event_type",
                "source_user_id",
                "source_product_id"
            ]).head(3)
        )

bot_summary(OLD_MANIFEST)
bot_summary(FINAL_MANIFEST)



File: anomaly_manifest_before_SP.csv
Total manifest rows: 350,203
Bot manifest rows: 53,061
Columns: ['week', 'anomaly_type', 'synthetic_row_id', 'source_row_number', 'missing_column', 'source_event_time', 'source_event_type', 'source_user_id', 'source_product_id']
shape: (3, 6)
┌────────────────┬────────────────┬────────────────┬───────────────┬───────────────┬───────────────┐
│ synthetic_row_ ┆ source_row_num ┆ source_event_t ┆ source_event_ ┆ source_user_i ┆ source_produc │
│ id             ┆ ber            ┆ ime            ┆ type          ┆ d             ┆ t_id          │
│ ---            ┆ ---            ┆ ---            ┆ ---           ┆ ---           ┆ ---           │
│ str            ┆ i64            ┆ str            ┆ str           ┆ i64           ┆ i64           │
╞════════════════╪════════════════╪════════════════╪═══════════════╪═══════════════╪═══════════════╡
│ W1-BL-000001   ┆ 1              ┆ 2019-10-01     ┆ view          ┆ 541312140     ┆ 44600062      │
│           

In [42]:

import polars as pl
from pathlib import Path

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

OLD_MANIFEST = BASE_DIR / "anomaly_manifest_before_SP.csv"
FINAL_MANIFEST = BASE_DIR / "anomaly_manifest.csv"

KEYS = [
    "synthetic_row_id",
    "source_row_number",
    "source_event_time",
    "source_event_type",
    "source_user_id",
    "source_product_id",
]

old_bot = (
    pl.read_csv(OLD_MANIFEST, infer_schema_length=10_000)
    .filter(pl.col("anomaly_type") == "Bot-like Crawling Loops")
    .select(KEYS)
)

final_bot = (
    pl.read_csv(FINAL_MANIFEST, infer_schema_length=10_000)
    .filter(pl.col("anomaly_type") == "Bot-like Crawling Loops")
    .select(KEYS)
)

# Compare both directions
old_only = old_bot.join(
    final_bot, on=KEYS, how="anti"
)

final_only = final_bot.join(
    old_bot, on=KEYS, how="anti"
)

print("Old Bot records:", f"{old_bot.height:,}")
print("Final Bot records:", f"{final_bot.height:,}")
print("Records only in old manifest:", f"{old_only.height:,}")
print("Records only in final manifest:", f"{final_only.height:,}")

assert old_only.height == 0
assert final_only.height == 0

print("\nPASS: Bot manifest records are unchanged.")


Old Bot records: 53,061
Final Bot records: 53,061
Records only in old manifest: 0
Records only in final manifest: 0

PASS: Bot manifest records are unchanged.


In [43]:

import polars as pl
from pathlib import Path
from datetime import datetime, timedelta

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")
OUTPUT_FILE = BASE_DIR / "2019-Oct_augmented.csv"

TARGET_ROWS = 53_061
ORIGINAL_WEEK1_ROWS = 8_829_315

ALL_COLUMNS = [
    "event_time", "event_type", "product_id", "category_id",
    "category_code", "brand", "price", "user_id", "user_session",
    "marketing_channel", "platform"
]

# 1. Read the final dataset lazily
final_data = pl.scan_csv(
    OUTPUT_FILE,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
)

schema = final_data.collect_schema()

assert schema.names() == ALL_COLUMNS, (
    f"Unexpected columns: {schema.names()}"
)

# 2. Rebuild eligible source records from the original Week 1 rows
original = (
    final_data
    .head(ORIGINAL_WEEK1_ROWS)
    .with_row_index("_source_row")
)

candidates = (
    original
    .filter(
        (pl.col("event_type") == "view")
        & pl.col("event_time").is_not_null()
        & pl.col("user_session").is_not_null()
        & pl.col("product_id").is_not_null()
        & (
            pl.col("event_time")
            < "2019-10-07 23:59:56 UTC"
        )
    )
    .collect(engine="streaming")
)

candidate_count = candidates.height

assert candidate_count >= TARGET_ROWS, (
    f"Not enough candidates: {candidate_count:,}"
)

selected_indices = [
    (i * candidate_count) // TARGET_ROWS
    for i in range(TARGET_ROWS)
]

selected = (
    candidates
    .with_row_index("_candidate_index")
    .filter(
        pl.col("_candidate_index").is_in(selected_indices)
    )
)

assert selected.height == TARGET_ROWS, (
    f"Expected {TARGET_ROWS:,} sources, got {selected.height:,}"
)

# 3. Rebuild the expected synthetic events using the original logic
expected_rows = []

for number, record in enumerate(
    selected.iter_rows(named=True),
    start=1,
):
    source_time = datetime.strptime(
        record["event_time"],
        "%Y-%m-%d %H:%M:%S UTC",
    )

    delay_seconds = ((number - 1) % 3) + 1
    synthetic_time = source_time + timedelta(
        seconds=delay_seconds
    )

    new_record = {
        column: record[column]
        for column in ALL_COLUMNS
    }

    new_record["event_time"] = synthetic_time.strftime(
        "%Y-%m-%d %H:%M:%S UTC"
    )
    new_record["event_type"] = "view"

    # Validate the generation rules themselves
    assert new_record["user_session"] == record["user_session"]
    assert new_record["product_id"] == record["product_id"]
    assert 1 <= delay_seconds <= 3

    expected_rows.append(new_record)

expected_bot = pl.DataFrame(
    expected_rows,
    schema=schema,
    orient="row",
    strict=False,
)

assert expected_bot.height == TARGET_ROWS
assert (expected_bot["event_type"] == "view").all()

print("PASS: Reconstructed all 53,061 expected Bot events.")
print("PASS: Every reconstructed event uses a 1–3 second delay.")

# 4. Find expected events absent from the final dataset
# Both sides are lazy here, avoiding collecting the entire final dataset.
missing_expected = (
    expected_bot.lazy()
    .join(
        final_data.select(ALL_COLUMNS),
        on=ALL_COLUMNS,
        how="anti",
        nulls_equal=True,
    )
    .select(pl.len().alias("missing_count"))
    .collect(engine="streaming")
    .item()
)

print(f"Expected Bot events: {TARGET_ROWS:,}")
print(f"Expected events not found: {missing_expected:,}")

if missing_expected == 0:
    print(
        "PASS: Every reconstructed Bot event has an exact "
        "row match in the final dataset."
    )
else:
    print(
        "WARNING: Some reconstructed events were not found. "
        "No files were modified."
    )


PASS: Reconstructed all 53,061 expected Bot events.
PASS: Every reconstructed event uses a 1–3 second delay.
Expected Bot events: 53,061
Expected events not found: 0
PASS: Every reconstructed Bot event has an exact row match in the final dataset.


In [44]:

import polars as pl
from datetime import datetime, timedelta
from pathlib import Path

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")
OUTPUT_FILE = BASE_DIR / "2019-Oct_augmented.csv"
MANIFEST_FILE = BASE_DIR / "anomaly_manifest.csv"

ALL_COLUMNS = [
    "event_time", "event_type", "product_id", "category_id",
    "category_code", "brand", "price", "user_id", "user_session",
    "marketing_channel", "platform"
]

COMPOSITE_KEY = ["user_session", "product_id"]
TARGET_ROWS = 42_449
ORIGINAL_WEEK1_ROWS = 8_829_315

# 1. Validate manifest records
manifest = pl.read_csv(
    MANIFEST_FILE,
    infer_schema_length=10_000
)

du_manifest = manifest.filter(
    pl.col("anomaly_type") == "Duplicate Events"
)

assert du_manifest.height == TARGET_ROWS, (
    f"Expected {TARGET_ROWS:,} manifest records, "
    f"found {du_manifest.height:,}"
)

assert du_manifest["synthetic_row_id"].n_unique() == TARGET_ROWS, (
    "Duplicate synthetic IDs found."
)

assert du_manifest["synthetic_row_id"].str.starts_with(
    "W1-DU-"
).all(), "Unexpected synthetic ID prefix."

print("PASS: Duplicate Events manifest count and IDs.")

# 2. Recreate the eligible source records
source = pl.scan_csv(
    OUTPUT_FILE,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
)

assert source.collect_schema().names() == ALL_COLUMNS

original = (
    source
    .head(ORIGINAL_WEEK1_ROWS)
    .with_row_index("_source_row")
    .filter(
        pl.col("user_session").is_not_null()
        & pl.col("product_id").is_not_null()
        & pl.col("event_time").is_not_null()
    )
)

eligible_count = (
    original
    .select(pl.len())
    .collect()
    .item()
)

assert eligible_count >= TARGET_ROWS

# Mirror the original selection formula exactly.
eligible = original.with_row_index("_eligible_row")
eligible_idx = pl.col("_eligible_row").cast(pl.Int64)

candidates = (
    eligible
    .filter(
        (((eligible_idx + 1) * TARGET_ROWS) // eligible_count)
        > ((eligible_idx * TARGET_ROWS) // eligible_count)
    )
    .collect(engine="streaming")
)

assert candidates.height == TARGET_ROWS, (
    f"Expected {TARGET_ROWS:,} candidates, "
    f"found {candidates.height:,}"
)

print(f"PASS: Reconstructed {candidates.height:,} source records.")

# 3. Reconstruct the expected duplicate rows
expected_rows = []

for record in candidates.iter_rows(named=True):
    new_record = {
        column: record[column]
        for column in ALL_COLUMNS
    }

    source_time = datetime.strptime(
        record["event_time"],
        "%Y-%m-%d %H:%M:%S UTC"
    )

    new_record["event_time"] = (
        source_time + timedelta(seconds=1)
    ).strftime("%Y-%m-%d %H:%M:%S UTC")

    # The composite key must remain unchanged.
    assert new_record["user_session"] == record["user_session"]
    assert new_record["product_id"] == record["product_id"]

    expected_rows.append(new_record)

expected_duplicates = pl.DataFrame(
    expected_rows,
    schema=source.collect_schema(),
    orient="row",
    strict=False,
)

assert expected_duplicates.height == TARGET_ROWS

print("PASS: Reconstructed duplicate rows with a 1-second time shift.")

# 4. Verify every expected row has an exact match in final data
missing_rows = (
    expected_duplicates.lazy()
    .join(
        source.select(ALL_COLUMNS),
        on=ALL_COLUMNS,
        how="anti",
        nulls_equal=True,
    )
    .select(pl.len().alias("missing_count"))
    .collect(engine="streaming")
    .item()
)

print(f"Expected duplicate rows: {TARGET_ROWS:,}")
print(f"Expected rows without an exact match: {missing_rows:,}")

assert missing_rows == 0, (
    "Some reconstructed duplicate rows were not found "
    "in the final dataset."
)

print("PASS: Every reconstructed duplicate row exists in final data.")

# 5. Report composite-key properties of reconstructed rows
key_stats = expected_duplicates.select(
    pl.col("user_session").is_null().sum().alias("null_sessions"),
    pl.col("product_id").is_null().sum().alias("null_products"),
    pl.struct(COMPOSITE_KEY).n_unique().alias("unique_composite_keys"),
).row(0, named=True)

print("\n=== DUPLICATE EVENTS VALIDATION SUMMARY ===")
print(f"Manifest records: {du_manifest.height:,}")
print(f"Reconstructed rows: {expected_duplicates.height:,}")
print(f"Rows without exact matches: {missing_rows:,}")
print(f"Null sessions: {key_stats['null_sessions']:,}")
print(f"Null product IDs: {key_stats['null_products']:,}")
print(
    "Unique composite keys among generated rows:",
    f"{key_stats['unique_composite_keys']:,}"
)
print("\nAll implemented checks passed.")


PASS: Duplicate Events manifest count and IDs.
PASS: Reconstructed 42,449 source records.
PASS: Reconstructed duplicate rows with a 1-second time shift.
Expected duplicate rows: 42,449
Expected rows without an exact match: 0
PASS: Every reconstructed duplicate row exists in final data.

=== DUPLICATE EVENTS VALIDATION SUMMARY ===
Manifest records: 42,449
Reconstructed rows: 42,449
Rows without exact matches: 0
Null sessions: 0
Null product IDs: 0
Unique composite keys among generated rows: 42,306

All implemented checks passed.


In [45]:

from pathlib import Path
from datetime import datetime, timedelta
import polars as pl

# ==========================================
# 1. Paths and configuration (READ-ONLY)
# ==========================================
BASE_DIR = Path(
    r"D:\Samsung\Graduation Project\Data\Old Data"
)

FINAL_PATH = BASE_DIR / "2019-Oct_augmented.csv"
MANIFEST_PATH = BASE_DIR / "anomaly_manifest.csv"

TARGET_ROWS = 42_449
ORIGINAL_WEEK1_ROWS = 8_829_315
ANOMALY_TYPE = "Impossible Sequences"
ANOMALY_PREFIX = "W1-IS-"

ALL_COLUMNS = [
    "event_time", "event_type", "product_id", "category_id",
    "category_code", "brand", "price", "user_id", "user_session",
    "marketing_channel", "platform"
]

MANIFEST_COLUMNS = [
    "week", "anomaly_type", "synthetic_row_id", "source_row_number",
    "missing_column", "source_event_time", "source_event_type",
    "source_user_id", "source_product_id"
]

assert FINAL_PATH.exists(), f"Missing file: {FINAL_PATH}"
assert MANIFEST_PATH.exists(), f"Missing file: {MANIFEST_PATH}"

# Scan only: no writes
final_lf = pl.scan_csv(
    FINAL_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
    schema_overrides={"event_time": pl.String}
)

final_schema = final_lf.collect_schema()
assert final_schema.names() == ALL_COLUMNS, final_schema.names()

final_rows_count = (
    final_lf.select(pl.len()).collect().item()
)

manifest = pl.read_csv(
    MANIFEST_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"]
)

assert manifest.columns == MANIFEST_COLUMNS

print(f"Final data rows: {final_rows_count:,}")
print(f"Manifest rows: {manifest.height:,}")

# ==========================================
# 2. Reconstruct eligible original purchases
# ==========================================
original = (
    final_lf
    .head(ORIGINAL_WEEK1_ROWS)
    .with_row_index("_source_row")
)

candidates = (
    original
    .filter(
        (pl.col("event_type") == "purchase")
        & pl.col("event_time").is_not_null()
        & pl.col("user_session").is_not_null()
        & pl.col("product_id").is_not_null()
        & (
            pl.col("event_time")
            < "2019-10-07 23:59:59 UTC"
        )
    )
    .collect(engine="streaming")
)

candidate_count = candidates.height

assert candidate_count >= TARGET_ROWS, (
    f"Not enough eligible purchases: {candidate_count:,}"
)

# Use exactly the same distributed selection logic
selected_indices = [
    (i * candidate_count) // TARGET_ROWS
    for i in range(TARGET_ROWS)
]

selected = (
    candidates
    .with_row_index("_candidate_index")
    .filter(
        pl.col("_candidate_index").is_in(selected_indices)
    )
)

assert selected.height == TARGET_ROWS

print(f"Eligible purchases: {candidate_count:,}")
print(f"Selected source purchases: {selected.height:,}")

# ==========================================
# 3. Reconstruct expected synthetic cart rows
# ==========================================
expected_rows = []
expected_manifest_rows = []

for number, record in enumerate(
    selected.iter_rows(named=True),
    start=1
):
    source_time_text = record["event_time"]

    source_time = datetime.strptime(
        source_time_text,
        "%Y-%m-%d %H:%M:%S UTC"
    )

    synthetic_time = source_time + timedelta(seconds=1)
    synthetic_time_text = synthetic_time.strftime(
        "%Y-%m-%d %H:%M:%S UTC"
    )

    row = {
        column: record[column]
        for column in ALL_COLUMNS
    }

    row["event_time"] = synthetic_time_text
    row["event_type"] = "cart"

    expected_rows.append(row)

    expected_manifest_rows.append({
        "week": "Week 1",
        "anomaly_type": ANOMALY_TYPE,
        "synthetic_row_id": f"{ANOMALY_PREFIX}{number:06d}",
        "source_row_number": int(record["_source_row"]) + 1,
        "missing_column": None,
        "source_event_time": source_time_text,
        "source_event_type": "purchase",
        "source_user_id": (
            None if record["user_id"] is None
            else str(record["user_id"])
        ),
        "source_product_id": (
            None if record["product_id"] is None
            else str(record["product_id"])
        )
    })

expected_df = pl.DataFrame(
    expected_rows,
    schema=final_schema,
    orient="row",
    strict=False
)

expected_manifest = pl.DataFrame(
    expected_manifest_rows,
    schema=manifest.schema,
    strict=False
)

assert expected_df.height == TARGET_ROWS
assert (expected_df["event_type"] == "cart").all()

print(f"Reconstructed cart events: {expected_df.height:,}")

# ==========================================
# 4. Verify every expected row exists in final
# ==========================================
# Read-only anti-join. No output files are written.
missing_data_rows = (
    expected_df.lazy()
    .join(
        final_lf.select(ALL_COLUMNS),
        on=ALL_COLUMNS,
        how="anti"
    )
    .collect(engine="streaming")
)

print(f"Expected rows without an exact match: {missing_data_rows.height:,}")

assert missing_data_rows.height == 0, (
    "Some reconstructed Impossible Sequences events "
    "were not found in the final dataset."
)

print("PASS: Every reconstructed event has an exact row match.")

# ==========================================
# 5. Verify the corresponding manifest records
# ==========================================
actual_manifest = manifest.filter(
    pl.col("anomaly_type") == ANOMALY_TYPE
)

assert actual_manifest.height == TARGET_ROWS, (
    f"Expected {TARGET_ROWS:,} manifest records, "
    f"found {actual_manifest.height:,}"
)

assert (
    actual_manifest["synthetic_row_id"].n_unique()
    == TARGET_ROWS
), "Duplicate Impossible Sequences IDs found."

expected_ids = [
    f"{ANOMALY_PREFIX}{i:06d}"
    for i in range(1, TARGET_ROWS + 1)
]

assert set(actual_manifest["synthetic_row_id"].to_list()) == set(expected_ids), (
    "Manifest IDs do not match the expected W1-IS ID range."
)

# Compare reconstructed manifest records with the actual manifest.
# Cast to strings so CSV type inference differences do not cause
# false mismatches; preserve nulls as nulls.
compare_columns = MANIFEST_COLUMNS

expected_cmp = expected_manifest.select([
    pl.col(c).cast(pl.String).alias(c)
    for c in compare_columns
])

actual_cmp = actual_manifest.select([
    pl.col(c).cast(pl.String).alias(c)
    for c in compare_columns
])

missing_manifest_rows = expected_cmp.join(
    actual_cmp,
    on=compare_columns,
    how="anti"
)

extra_manifest_rows = actual_cmp.join(
    expected_cmp,
    on=compare_columns,
    how="anti"
)

print(f"Expected manifest records missing: {missing_manifest_rows.height:,}")
print(f"Unexpected manifest records: {extra_manifest_rows.height:,}")

assert missing_manifest_rows.height == 0, (
    "Some reconstructed manifest records are missing or differ."
)

assert extra_manifest_rows.height == 0, (
    "Some actual manifest records differ from the reconstruction."
)

print("PASS: Manifest records match the reconstruction.")

# ==========================================
# 6. Final read-only summary
# ==========================================
print("\n=== IMPOSSIBLE SEQUENCES VALIDATION ===")
print(f"Expected synthetic events: {TARGET_ROWS:,}")
print(f"Reconstructed events: {expected_df.height:,}")
print(f"Exact data rows missing: {missing_data_rows.height:,}")
print(f"Manifest records: {actual_manifest.height:,}")
print(f"Unique synthetic IDs: {actual_manifest['synthetic_row_id'].n_unique():,}")
print(f"Final data rows currently: {final_rows_count:,}")

print("\nPASS: All implemented validation checks passed.")
print("No data or manifest files were modified.")


Final data rows: 9,253,804
Manifest rows: 424,489
Eligible purchases: 152,115
Selected source purchases: 42,449
Reconstructed cart events: 42,449
Expected rows without an exact match: 10,414


AssertionError: Some reconstructed Impossible Sequences events were not found in the final dataset.

In [47]:

print("Missing full-row matches:", missing_data_rows.height)

KEY_COLUMNS = [
    "event_time", "event_type", "user_session", "product_id"
]

key_matches = (
    missing_data_rows.lazy()
    .join(
        final_lf.select(KEY_COLUMNS),
        on=KEY_COLUMNS,
        how="semi"
    )
    .select(pl.len())
    .collect()
    .item()
)

print("Found by key fields:", key_matches)
print(
    "Not found by key fields:",
    missing_data_rows.height - key_matches
)

print(
    "Impossible Sequences manifest rows:",
    manifest.filter(
        pl.col("anomaly_type") == "Impossible Sequences"
    ).height
)


Missing full-row matches: 10414
Found by key fields: 10414
Not found by key fields: 0
Impossible Sequences manifest rows: 42449


In [48]:

KEY_COLUMNS = [
    "event_time", "event_type", "user_session", "product_id"
]

# Match each missing expected row to its final row using the key.
# Keep only columns outside the key for comparison.
OTHER_COLUMNS = [
    c for c in ALL_COLUMNS if c not in KEY_COLUMNS
]

matched_rows = (
    missing_data_rows.lazy()
    .join(
        final_lf.select(ALL_COLUMNS),
        on=KEY_COLUMNS,
        how="inner",
        suffix="_actual"
    )
    .collect(engine="streaming")
)

print("Rows matched by key:", matched_rows.height)

for col in OTHER_COLUMNS:
    actual_col = f"{col}_actual"

    if actual_col not in matched_rows.columns:
        print(f"{col}: comparison column not found")
        continue

    mismatch_count = (
        matched_rows
        .filter(
            pl.col(col).cast(pl.String).fill_null("<NULL>")
            != pl.col(actual_col).cast(pl.String).fill_null("<NULL>")
        )
        .height
    )

    print(f"{col}: {mismatch_count:,} mismatches")


Rows matched by key: 10414
category_id: 0 mismatches
category_code: 0 mismatches
brand: 0 mismatches
price: 0 mismatches
user_id: 0 mismatches
marketing_channel: 0 mismatches
platform: 0 mismatches


In [50]:

print("Matched rows schema:")
print(matched_rows.columns)

print("\nSample of expected events:")
print(
    missing_data_rows.select([
        "event_time", "event_type",
        "user_session", "product_id"
    ]).head(10)
)

print("\nSample of corresponding final rows:")
print(
    final_lf
    .filter(pl.col("event_type") == "cart")
    .select([
        "event_time", "event_type",
        "user_session", "product_id"
    ])
    .head(10)
    .collect()
)


Matched rows schema:
['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session', 'marketing_channel', 'platform', 'category_id_actual', 'category_code_actual', 'brand_actual', 'price_actual', 'user_id_actual', 'marketing_channel_actual', 'platform_actual']

Sample of expected events:
shape: (10, 4)
┌─────────────────────────┬────────────┬─────────────────────────────────┬────────────┐
│ event_time              ┆ event_type ┆ user_session                    ┆ product_id │
│ ---                     ┆ ---        ┆ ---                             ┆ ---        │
│ str                     ┆ str        ┆ str                             ┆ i64        │
╞═════════════════════════╪════════════╪═════════════════════════════════╪════════════╡
│ 2019-10-01 02:30:51 UTC ┆ cart       ┆ e95dab50-1d5b-4514-896e-aa5ad8… ┆ 4300382    │
│ 2019-10-01 03:01:38 UTC ┆ cart       ┆ d8b3d0fb-8333-4edc-ab44-738f47… ┆ 12710449   │
│ 2019-10-01 03:10:45 UT

In [51]:

# READ-ONLY: repeat the exact full-row comparison,
# treating null values as equal to other null values.

missing_nulls_equal = (
    expected_df.lazy()
    .join(
        final_lf.select(ALL_COLUMNS),
        on=ALL_COLUMNS,
        how="anti",
        nulls_equal=True
    )
    .collect(engine="streaming")
)

print("Missing rows with default null handling:", missing_data_rows.height)
print("Missing rows with nulls_equal=True:", missing_nulls_equal.height)

if missing_nulls_equal.height == 0:
    print("PASS: All reconstructed rows match when nulls are treated as equal.")
else:
    print("Some rows still do not match; further investigation is needed.")


Missing rows with default null handling: 10414
Missing rows with nulls_equal=True: 0
PASS: All reconstructed rows match when nulls are treated as equal.


In [52]:

from pathlib import Path
import polars as pl

# ==========================================
# 1. Read-only configuration
# ==========================================
BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")
FINAL_PATH = BASE_DIR / "2019-Oct_augmented.csv"
MANIFEST_PATH = BASE_DIR / "anomaly_manifest.csv"

TARGET_ROWS = 212_244
ORIGINAL_WEEK1_ROWS = 8_829_315
ANOMALY_TYPE = "Missing Data"
PREFIX = "W1-MD-"

ALL_COLUMNS = [
    "event_time", "event_type", "product_id", "category_id",
    "category_code", "brand", "price", "user_id", "user_session",
    "marketing_channel", "platform"
]

MANIFEST_COLUMNS = [
    "week", "anomaly_type", "synthetic_row_id", "source_row_number",
    "missing_column", "source_event_time", "source_event_type",
    "source_user_id", "source_product_id"
]

assert FINAL_PATH.exists(), f"Missing file: {FINAL_PATH}"
assert MANIFEST_PATH.exists(), f"Missing file: {MANIFEST_PATH}"

final_lf = pl.scan_csv(
    FINAL_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
    schema_overrides={"event_time": pl.String}
)

schema = final_lf.collect_schema()
assert schema.names() == ALL_COLUMNS, schema.names()

final_count = final_lf.select(pl.len()).collect().item()

manifest = pl.read_csv(
    MANIFEST_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"]
)

assert manifest.columns == MANIFEST_COLUMNS

actual_manifest = manifest.filter(
    pl.col("anomaly_type") == ANOMALY_TYPE
)

print(f"Final data rows: {final_count:,}")
print(f"Total manifest rows: {manifest.height:,}")
print(f"Missing Data manifest rows: {actual_manifest.height:,}")

# ==========================================
# 2. Reconstruct original source selection
# ==========================================
# Assumption: the original records remain the first
# ORIGINAL_WEEK1_ROWS rows in their original order.
original = (
    final_lf
    .head(ORIGINAL_WEEK1_ROWS)
    .with_row_index("_source_row")
)

idx = pl.col("_source_row").cast(pl.Int64)

candidates = (
    original
    .filter(
        (((idx + 1) * TARGET_ROWS) // ORIGINAL_WEEK1_ROWS)
        > ((idx * TARGET_ROWS) // ORIGINAL_WEEK1_ROWS)
    )
    .collect(engine="streaming")
)

assert candidates.height == TARGET_ROWS, (
    f"Expected {TARGET_ROWS:,} candidates, "
    f"found {candidates.height:,}"
)

# ==========================================
# 3. Rebuild expected rows and manifest
# ==========================================
expected_rows = []
expected_manifest_rows = []

for number, record in enumerate(
    candidates.iter_rows(named=True), start=1
):
    source_row_number = int(record["_source_row"]) + 1

    start = (number - 1) % len(ALL_COLUMNS)
    missing_column = None

    for offset in range(len(ALL_COLUMNS)):
        column = ALL_COLUMNS[(start + offset) % len(ALL_COLUMNS)]
        if record[column] is not None:
            missing_column = column
            break

    assert missing_column is not None, (
        f"Source row {source_row_number} has no non-null values"
    )

    expected = {c: record[c] for c in ALL_COLUMNS}
    expected[missing_column] = None
    expected_rows.append(expected)

    expected_manifest_rows.append({
        "week": "Week 1",
        "anomaly_type": ANOMALY_TYPE,
        "synthetic_row_id": f"{PREFIX}{number:06d}",
        "source_row_number": source_row_number,
        "missing_column": missing_column,
        "source_event_time": record["event_time"],
        "source_event_type": record["event_type"],
        "source_user_id": record["user_id"],
        "source_product_id": record["product_id"],
    })

expected_df = pl.DataFrame(
    expected_rows,
    schema=schema,
    orient="row",
    strict=False
)

expected_manifest = pl.DataFrame(
    expected_manifest_rows,
    schema=manifest.schema,
    strict=False
)

assert expected_df.height == TARGET_ROWS
assert expected_manifest.height == TARGET_ROWS

print(f"Reconstructed rows: {expected_df.height:,}")

# ==========================================
# 4. Compare complete data rows
# ==========================================
# nulls_equal=True is important: nulls should match nulls.
missing_data_matches = (
    expected_df.lazy()
    .join(
        final_lf.select(ALL_COLUMNS),
        on=ALL_COLUMNS,
        how="anti",
        nulls_equal=True
    )
    .collect(engine="streaming")
)

print(
    "Expected rows without an exact match:",
    f"{missing_data_matches.height:,}"
)

# ==========================================
# 5. Compare the manifest records
# ==========================================
assert actual_manifest.height == TARGET_ROWS, (
    f"Expected {TARGET_ROWS:,} Missing Data manifest rows, "
    f"found {actual_manifest.height:,}"
)

assert (
    actual_manifest["synthetic_row_id"].n_unique() == TARGET_ROWS
), "Duplicate Missing Data IDs found"

expected_cmp = expected_manifest.select([
    pl.col(c).cast(pl.String).alias(c)
    for c in MANIFEST_COLUMNS
])

actual_cmp = actual_manifest.select([
    pl.col(c).cast(pl.String).alias(c)
    for c in MANIFEST_COLUMNS
])

missing_manifest_matches = expected_cmp.join(
    actual_cmp,
    on=MANIFEST_COLUMNS,
    how="anti",
    nulls_equal=True
)

unexpected_manifest_matches = actual_cmp.join(
    expected_cmp,
    on=MANIFEST_COLUMNS,
    how="anti",
    nulls_equal=True
)

print(
    "Expected manifest records missing:",
    f"{missing_manifest_matches.height:,}"
)
print(
    "Unexpected manifest records:",
    f"{unexpected_manifest_matches.height:,}"
)

# ==========================================
# 6. Check the declared missing column
# ==========================================
print("\n=== Missing Data by column ===")

expected_distribution = (
    expected_manifest
    .group_by("missing_column")
    .agg(pl.len().alias("expected_count"))
    .sort("missing_column")
)

actual_distribution = (
    actual_manifest
    .group_by("missing_column")
    .agg(pl.len().alias("actual_count"))
    .sort("missing_column")
)

print("Expected distribution:")
print(expected_distribution)

print("Actual manifest distribution:")
print(actual_distribution)

# ==========================================
# 7. Final status
# ==========================================
data_pass = missing_data_matches.height == 0
manifest_pass = (
    missing_manifest_matches.height == 0
    and unexpected_manifest_matches.height == 0
)

print("\n=== MISSING DATA VALIDATION SUMMARY ===")
print(f"Expected injected rows: {TARGET_ROWS:,}")
print(f"Reconstructed rows: {expected_df.height:,}")
print(f"Unmatched data rows: {missing_data_matches.height:,}")
print(f"Manifest records: {actual_manifest.height:,}")
print(f"Unique synthetic IDs: {actual_manifest['synthetic_row_id'].n_unique():,}")
print(f"Data validation: {'PASS' if data_pass else 'FAIL'}")
print(f"Manifest validation: {'PASS' if manifest_pass else 'FAIL'}")
print("No files were written or modified.")

if data_pass and manifest_pass:
    print("\nPASS: All implemented Missing Data checks passed.")
else:
    print("\nINVESTIGATE: One or more validation checks failed.")


Final data rows: 9,253,804
Total manifest rows: 424,489
Missing Data manifest rows: 212,244
Reconstructed rows: 212,244
Expected rows without an exact match: 0
Expected manifest records missing: 0
Unexpected manifest records: 0

=== Missing Data by column ===
Expected distribution:
shape: (11, 2)
┌────────────────┬────────────────┐
│ missing_column ┆ expected_count │
│ ---            ┆ ---            │
│ str            ┆ u32            │
╞════════════════╪════════════════╡
│ brand          ┆ 21042          │
│ category_code  ┆ 13174          │
│ category_id    ┆ 19295          │
│ event_time     ┆ 19295          │
│ event_type     ┆ 19295          │
│ …              ┆ …              │
│ platform       ┆ 19294          │
│ price          ┆ 23669          │
│ product_id     ┆ 19295          │
│ user_id        ┆ 19295          │
│ user_session   ┆ 19295          │
└────────────────┴────────────────┘
Actual manifest distribution:
shape: (11, 2)
┌────────────────┬──────────────┐
│ missing_c

In [53]:

from pathlib import Path
from collections import Counter
import csv
import polars as pl

# ============================================================
# 1. CONFIGURATION — READ ONLY
# ============================================================
BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

DATA_PATH = BASE_DIR / "2019-Oct_augmented.csv"
MANIFEST_PATH = BASE_DIR / "anomaly_manifest.csv"
ORIGINAL_PATH = BASE_DIR / "2019-Oct.csv"

ORIGINAL_ROWS = 8_829_315
EXPECTED_DATA_ROWS = 9_253_804
EXPECTED_MANIFEST_ROWS = 424_489

DATA_COLUMNS = [
    "event_time", "event_type", "product_id", "category_id",
    "category_code", "brand", "price", "user_id", "user_session",
    "marketing_channel", "platform"
]

MANIFEST_COLUMNS = [
    "week", "anomaly_type", "synthetic_row_id", "source_row_number",
    "missing_column", "source_event_time", "source_event_type",
    "source_user_id", "source_product_id"
]

EXPECTED_ANOMALIES = {
    "Bot-like Crawling Loops": 53_061,
    "Cart Abandonment Scenarios": 31_837,
    "Duplicate Events": 42_449,
    "Impossible Sequences": 42_449,
    "Missing Data": 212_244,
    "Seasonal Promotional Purchase Surges": 42_449,
}

ID_PREFIXES = {
    "Bot-like Crawling Loops": "W1-BL-",
    "Cart Abandonment Scenarios": "W1-CA-",
    "Duplicate Events": "W1-DU-",
    "Impossible Sequences": "W1-IS-",
    "Missing Data": "W1-MD-",
    "Seasonal Promotional Purchase Surges": "W1-SP-",
}

results = []

def record_check(name, passed, details=""):
    status = "PASS" if passed else "FAIL"
    results.append((name, status, str(details)))
    print(f"[{status}] {name}" + (f" — {details}" if details else ""))

# ============================================================
# 2. FILE EXISTENCE AND ORIGINAL FILE
# ============================================================
print("\n" + "=" * 72)
print("FINAL DATA QUALITY & ANOMALY VALIDATION REPORT")
print("=" * 72)

record_check("Final dataset exists", DATA_PATH.exists(), DATA_PATH)
record_check("Manifest exists", MANIFEST_PATH.exists(), MANIFEST_PATH)
record_check(
    "Original dataset exists",
    ORIGINAL_PATH.exists(),
    ORIGINAL_PATH
)

if not DATA_PATH.exists() or not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "Final dataset or manifest is missing. No files were modified."
    )

if ORIGINAL_PATH.exists():
    record_check(
        "Original file is separate from augmented file",
        ORIGINAL_PATH.resolve() != DATA_PATH.resolve()
    )

# ============================================================
# 3. DATASET SCHEMA AND ROW COUNTS
# ============================================================
print("\n--- DATASET STRUCTURE ---")

data_lf = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
    schema_overrides={"event_time": pl.String}
)

data_schema = data_lf.collect_schema()
actual_columns = data_schema.names()

record_check(
    "Dataset column names and order",
    actual_columns == DATA_COLUMNS,
    f"Found {len(actual_columns)} columns"
)

# Count rows and inspect repeated headers in one sequential CSV pass.
# This may take time because the dataset is large.
csv_data_rows = 0
repeated_header_rows = 0
wrong_width_rows = 0
bad_width_examples = []

with DATA_PATH.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f)
    header = next(reader, None)

    record_check(
        "CSV header matches expected columns",
        header == DATA_COLUMNS,
        f"Header columns: {len(header) if header else 0}"
    )

    if header is None:
        raise ValueError("The final CSV file is empty.")

    for row in reader:
        csv_data_rows += 1

        if row == DATA_COLUMNS:
            repeated_header_rows += 1

        if len(row) != len(DATA_COLUMNS):
            wrong_width_rows += 1
            if len(bad_width_examples) < 5:
                bad_width_examples.append({
                    "csv_record_number": csv_data_rows + 1,
                    "column_count": len(row)
                })

record_check(
    "Final dataset row count",
    csv_data_rows == EXPECTED_DATA_ROWS,
    f"{csv_data_rows:,} rows; expected {EXPECTED_DATA_ROWS:,}"
)

record_check(
    "No repeated CSV header rows",
    repeated_header_rows == 0,
    f"Repeated headers: {repeated_header_rows:,}"
)

record_check(
    "Every data record has 11 fields",
    wrong_width_rows == 0,
    f"Malformed-width records: {wrong_width_rows:,}"
)

if bad_width_examples:
    print("Examples of rows with unexpected field counts:")
    print(bad_width_examples)

# Verify row count with Polars too.
polars_rows = (
    data_lf.select(pl.len())
    .collect(engine="streaming")
    .item()
)

record_check(
    "Independent Polars row count",
    polars_rows == EXPECTED_DATA_ROWS,
    f"{polars_rows:,} rows"
)

# ============================================================
# 4. MANIFEST STRUCTURE AND TOTALS
# ============================================================
print("\n--- MANIFEST STRUCTURE ---")

manifest = pl.read_csv(
    MANIFEST_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"]
)

record_check(
    "Manifest column names and order",
    manifest.columns == MANIFEST_COLUMNS,
    f"Found {len(manifest.columns)} columns"
)

record_check(
    "Total manifest row count",
    manifest.height == EXPECTED_MANIFEST_ROWS,
    f"{manifest.height:,} rows; expected {EXPECTED_MANIFEST_ROWS:,}"
)

record_check(
    "All synthetic IDs are globally unique",
    manifest["synthetic_row_id"].n_unique() == manifest.height,
    f"Unique IDs: {manifest['synthetic_row_id'].n_unique():,}"
)

record_check(
    "No null synthetic IDs",
    manifest["synthetic_row_id"].null_count() == 0
)

record_check(
    "All manifest records belong to Week 1",
    manifest.filter(pl.col("week") != "Week 1").height == 0,
    f"Unexpected week records: {manifest.filter(pl.col('week') != 'Week 1').height:,}"
)

# ============================================================
# 5. ANOMALY COUNTS AND ID PREFIXES
# ============================================================
print("\n--- ANOMALY COUNTS AND ID VALIDATION ---")

actual_counts = {
    row["anomaly_type"]: row["len"]
    for row in (
        manifest.group_by("anomaly_type")
        .len()
        .iter_rows(named=True)
    )
}

unknown_types = set(actual_counts) - set(EXPECTED_ANOMALIES)

record_check(
    "No unexpected anomaly types",
    len(unknown_types) == 0,
    f"Unexpected types: {sorted(unknown_types)}"
)

for anomaly, expected_count in EXPECTED_ANOMALIES.items():
    subset = manifest.filter(pl.col("anomaly_type") == anomaly)
    actual_count = subset.height

    record_check(
        f"{anomaly}: manifest count",
        actual_count == expected_count,
        f"{actual_count:,} / {expected_count:,}"
    )

    prefix = ID_PREFIXES[anomaly]
    wrong_prefix_count = subset.filter(
        ~pl.col("synthetic_row_id").str.starts_with(prefix)
    ).height

    record_check(
        f"{anomaly}: ID prefix",
        wrong_prefix_count == 0,
        f"IDs with wrong prefix: {wrong_prefix_count:,}"
    )

    id_values = subset["synthetic_row_id"].drop_nulls().to_list()
    expected_ids = {
        f"{prefix}{i:06d}"
        for i in range(1, expected_count + 1)
    }

    record_check(
        f"{anomaly}: complete ID sequence",
        set(id_values) == expected_ids,
        f"Distinct IDs: {len(set(id_values)):,}"
    )

# ============================================================
# 6. SOURCE ROW REFERENCES
# ============================================================
print("\n--- SOURCE REFERENCES ---")

source_rows = manifest["source_row_number"]

invalid_source_refs = manifest.filter(
    pl.col("source_row_number").is_null()
    | (pl.col("source_row_number") < 1)
    | (pl.col("source_row_number") > ORIGINAL_ROWS)
)

record_check(
    "Manifest source row numbers are within original Week 1",
    invalid_source_refs.height == 0,
    f"Invalid source references: {invalid_source_refs.height:,}"
)

# ============================================================
# 7. MISSING DATA DETAILS
# ============================================================
print("\n--- MISSING DATA DISTRIBUTION ---")

md = manifest.filter(
    pl.col("anomaly_type") == "Missing Data"
)

md_null_column = md.filter(
    pl.col("missing_column").is_null()
    | (~pl.col("missing_column").is_in(DATA_COLUMNS))
)

record_check(
    "Missing Data has a valid missing_column",
    md_null_column.height == 0,
    f"Invalid declarations: {md_null_column.height:,}"
)

md_distribution = (
    md.group_by("missing_column")
    .agg(pl.len().alias("injected_rows"))
    .sort("missing_column")
)

print(md_distribution)

record_check(
    "Missing Data distribution totals correctly",
    md_distribution["injected_rows"].sum() == EXPECTED_ANOMALIES["Missing Data"],
    f"Distribution total: {md_distribution['injected_rows'].sum():,}"
)

# Other anomaly types are expected to leave missing_column unset
non_md = manifest.filter(
    pl.col("anomaly_type") != "Missing Data"
)

non_md_declared_missing = non_md.filter(
    pl.col("missing_column").is_not_null()
)

record_check(
    "Other anomalies do not declare a missing column",
    non_md_declared_missing.height == 0,
    f"Unexpected declarations: {non_md_declared_missing.height:,}"
)

# ============================================================
# 8. CROSS-CHECK EXPECTED ROW MATH
# ============================================================
print("\n--- ROW-COUNT RECONCILIATION ---")

expected_injected_total = sum(EXPECTED_ANOMALIES.values())
calculated_final_rows = ORIGINAL_ROWS + expected_injected_total

print(f"Original Week 1 rows:      {ORIGINAL_ROWS:,}")
print(f"Expected injected rows:   {expected_injected_total:,}")
print(f"Calculated final rows:    {calculated_final_rows:,}")
print(f"Observed final rows:      {csv_data_rows:,}")

record_check(
    "Original rows + expected injections = final row count",
    calculated_final_rows == EXPECTED_DATA_ROWS
    and csv_data_rows == calculated_final_rows,
    f"Difference: {csv_data_rows - calculated_final_rows:+,}"
)

record_check(
    "Manifest total equals sum of anomaly targets",
    manifest.height == expected_injected_total,
    f"{manifest.height:,} / {expected_injected_total:,}"
)

# ============================================================
# 9. PREVIOUS PER-ANOMALY VALIDATIONS
# ============================================================
print("\n--- PREVIOUS DETAILED VALIDATION RESULTS ---")

previously_validated = {
    "Bot-like Crawling Loops": (
        "Reconstructed 53,061 rows; exact matches 53,061 "
        "under the previously used comparison."
    ),
    "Cart Abandonment Scenarios": (
        "Previously checked 31,837 chunk rows; unmatched 0."
    ),
    "Duplicate Events": (
        "Reconstructed 42,449 rows; exact matches 42,449. "
        "Unique composite keys among generated rows: 42,306."
    ),
    "Impossible Sequences": (
        "Reconstructed 42,449 rows; unmatched rows 0 "
        "when nulls_equal=True."
    ),
    "Missing Data": (
        "Reconstructed 212,244 rows; unmatched rows 0 "
        "with nulls_equal=True; manifest comparisons passed."
    ),
    "Seasonal Promotional Purchase Surges": (
        "Previously checked 42,449 promotional purchase rows, "
        "channel marker, and 1,800 distinct timestamps."
    )
}

for anomaly, details in previously_validated.items():
    print(f"- {anomaly}: {details}")

print(
    "\nNote: these are summaries of previous validation runs, "
    "not reruns of all six reconstruction tests in this cell."
)

# ============================================================
# 10. FINAL SUMMARY
# ============================================================
print("\n" + "=" * 72)
print("FINAL AUDIT SUMMARY")
print("=" * 72)

passed = sum(status == "PASS" for _, status, _ in results)
failed = sum(status == "FAIL" for _, status, _ in results)

print(f"Checks passed: {passed}")
print(f"Checks failed: {failed}")
print(f"Final data rows: {csv_data_rows:,}")
print(f"Manifest rows: {manifest.height:,}")
print(f"Total expected synthetic rows: {expected_injected_total:,}")
print(f"Unique synthetic IDs: {manifest['synthetic_row_id'].n_unique():,}")

if failed == 0:
    print("\nOVERALL RESULT: PASS — all checks in this audit passed.")
else:
    print("\nOVERALL RESULT: FAIL — review the failed checks above.")

print("\nREAD-ONLY AUDIT COMPLETE.")
print("No dataset, manifest, backup, or chunk file was written.")



FINAL DATA QUALITY & ANOMALY VALIDATION REPORT
[PASS] Final dataset exists — D:\Samsung\Graduation Project\Data\Old Data\2019-Oct_augmented.csv
[PASS] Manifest exists — D:\Samsung\Graduation Project\Data\Old Data\anomaly_manifest.csv
[PASS] Original dataset exists — D:\Samsung\Graduation Project\Data\Old Data\2019-Oct.csv
[PASS] Original file is separate from augmented file

--- DATASET STRUCTURE ---
[PASS] Dataset column names and order — Found 11 columns
[PASS] CSV header matches expected columns — Header columns: 11
[PASS] Final dataset row count — 9,253,804 rows; expected 9,253,804
[PASS] No repeated CSV header rows — Repeated headers: 0
[PASS] Every data record has 11 fields — Malformed-width records: 0
[PASS] Independent Polars row count — 9,253,804 rows

--- MANIFEST STRUCTURE ---
[PASS] Manifest column names and order — Found 9 columns
[PASS] Total manifest row count — 424,489 rows; expected 424,489
[PASS] All synthetic IDs are globally unique — Unique IDs: 424,489
[PASS] No n

# Week 2 Data Extraction

## 1. Objective

Extract Week 2 data from the original `2019-Oct.csv` file without modifying the original dataset or the finalized Week 1 files.

## 2. Source Dataset

- **Source file:** `2019-Oct.csv`
- **Processing library:** Polars
- **Reading method:** Lazy API (`pl.scan_csv`)
- **Expected schema:** The original 9 columns defined in `ORIGINAL_COLUMNS`

## 3. Date Range

Week 2 covers the following UTC interval:

- **Start:** `2019-10-08 00:00:00 UTC` — inclusive
- **End:** `2019-10-15 00:00:00 UTC` — exclusive

The exclusive end boundary prevents events at midnight on October 15 from being included in Week 2.

## 4. Extraction Code

```python
from pathlib import Path
import polars as pl

# Read the original dataset lazily
source = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
)

# Validate the original schema
schema = source.collect_schema()
assert schema.names() == ORIGINAL_COLUMNS, (
    f"Unexpected source schema: {schema.names()}"
)

# Define Week 2 boundaries
WEEK2_START = "2019-10-08 00:00:00 UTC"
WEEK2_END = "2019-10-15 00:00:00 UTC"

# Extract Week 2
week2 = source.filter(
    (pl.col("event_time") >= WEEK2_START)
    & (pl.col("event_time") < WEEK2_END)
)

# Preview extracted data
print("Original schema:")
print(schema)

print("\nWeek 2 preview:")
print(week2.head(5).collect())

# Count events by type
print("\nWeek 2 event counts:")
print(
    week2.group_by("event_type")
    .agg(pl.len().alias("row_count"))
    .sort("event_type")
    .collect()
)

# Count extracted rows
print("\nWeek 2 row count:")
print(week2.select(pl.len()).collect())
```

## 5. Validation Checklist

After running the code, verify the following:

- [ ] The original schema contains exactly the expected 9 columns.
- [ ] The extracted events fall within the defined date range.
- [ ] The first five records look correct.
- [ ] Event counts by `event_type` are displayed successfully.
- [ ] The total number of extracted rows is recorded.
- [ ] The original dataset remains unchanged.
- [ ] Week 1 final files remain unchanged.

## 6. Output Status

The current code performs extraction and inspection only. It does not save a new file or inject synthetic anomalies.

After validating the row count and date range, the extracted data can be saved separately as:

`2019-Oct_week2.csv`

The Week 2 anomaly-generation process should begin only after the extraction has been validated.

## 7. Important Notes

- `DATA_PATH` must point to the original `2019-Oct.csv` file.
- `ORIGINAL_COLUMNS` must contain the expected source column names in the correct order.
- The date filter assumes `event_time` is stored as consistently formatted UTC text. If the source contains inconsistent formats or invalid timestamps, additional parsing and validation may be necessary.
- The extracted row count must be measured from the actual source file; it should not be assumed in advance.


In [55]:

from pathlib import Path
import polars as pl

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

ORIGINAL_DATA_PATH = BASE_DIR / "2019-Oct.csv"

print("File exists:", ORIGINAL_DATA_PATH.exists())
print("File path:", ORIGINAL_DATA_PATH)

source = pl.scan_csv(
    str(ORIGINAL_DATA_PATH),
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
)

schema = source.collect_schema()

print("\nOriginal schema:")
print(schema)

assert schema.names() == ORIGINAL_COLUMNS, (
    f"Expected the original 9 columns, found: {schema.names()}"
)

print("\nPASS: Original dataset schema is correct.")


File exists: True
File path: D:\Samsung\Graduation Project\Data\Old Data\2019-Oct.csv

Original schema:
Schema({'event_time': String, 'event_type': String, 'product_id': Int64, 'category_id': Int64, 'category_code': String, 'brand': String, 'price': Float64, 'user_id': Int64, 'user_session': String})

PASS: Original dataset schema is correct.


In [56]:

import polars as pl

WEEK2_START = "2019-10-08 00:00:00 UTC"
WEEK2_END = "2019-10-15 00:00:00 UTC"

week2 = source.filter(
    (pl.col("event_time") >= WEEK2_START)
    & (pl.col("event_time") < WEEK2_END)
)

# Count rows
week2_count = week2.select(pl.len().alias("row_count")).collect()
print("Week 2 row count:")
print(week2_count)

# Check minimum and maximum event times
week2_dates = week2.select(
    pl.col("event_time").min().alias("min_event_time"),
    pl.col("event_time").max().alias("max_event_time"),
).collect()

print("\nWeek 2 date range:")
print(week2_dates)

# Preview
print("\nWeek 2 preview:")
print(week2.head(5).collect())

# Event counts
print("\nEvent counts:")
print(
    week2.group_by("event_type")
    .agg(pl.len().alias("row_count"))
    .sort("event_type")
    .collect()
)


Week 2 row count:
shape: (1, 1)
┌───────────┐
│ row_count │
│ ---       │
│ u32       │
╞═══════════╡
│ 10071265  │
└───────────┘

Week 2 date range:
shape: (1, 2)
┌─────────────────────────┬─────────────────────────┐
│ min_event_time          ┆ max_event_time          │
│ ---                     ┆ ---                     │
│ str                     ┆ str                     │
╞═════════════════════════╪═════════════════════════╡
│ 2019-10-08 00:00:00 UTC ┆ 2019-10-14 23:59:59 UTC │
└─────────────────────────┴─────────────────────────┘

Week 2 preview:
shape: (5, 9)
┌────────────┬────────────┬────────────┬───────────┬───┬──────────┬────────┬───────────┬───────────┐
│ event_time ┆ event_type ┆ product_id ┆ category_ ┆ … ┆ brand    ┆ price  ┆ user_id   ┆ user_sess │
│ ---        ┆ ---        ┆ ---        ┆ id        ┆   ┆ ---      ┆ ---    ┆ ---       ┆ ion       │
│ str        ┆ str        ┆ i64        ┆ ---       ┆   ┆ str      ┆ f64    ┆ i64       ┆ ---       │
│            ┆         

In [57]:

from pathlib import Path
import polars as pl

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")
WEEK2_OUTPUT_PATH = BASE_DIR / "2019-Oct_week2.csv"

# Do not overwrite an existing file
if WEEK2_OUTPUT_PATH.exists():
    raise FileExistsError(
        f"Output already exists; refusing to overwrite: {WEEK2_OUTPUT_PATH}"
    )

# Save the original Week 2 rows
week2.sink_csv(str(WEEK2_OUTPUT_PATH))

print("Saved Week 2 to:", WEEK2_OUTPUT_PATH)
print("Expected rows:", 10_071_265)
print("Expected columns:", 9)

# Verify the saved file
saved_week2 = pl.scan_csv(str(WEEK2_OUTPUT_PATH))
saved_schema = saved_week2.collect_schema()
saved_count = saved_week2.select(pl.len()).collect().item()

assert saved_schema.names() == ORIGINAL_COLUMNS
assert saved_count == 10_071_265

print("PASS: Saved file schema and row count verified.")


Saved Week 2 to: D:\Samsung\Graduation Project\Data\Old Data\2019-Oct_week2.csv
Expected rows: 10071265
Expected columns: 9
PASS: Saved file schema and row count verified.


In [58]:

import polars as pl
from pathlib import Path

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")
WEEK1_PATH = BASE_DIR / "2019-Oct_augmented.csv"

df = pl.scan_csv(
    str(WEEK1_PATH),
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
)

for col in ["marketing_channel", "platform"]:
    print(f"\n--- {col} ---")
    print(
        df.group_by(col)
        .agg(pl.len().alias("row_count"))
        .with_columns(
            (pl.col("row_count") / df.select(pl.len()).collect().item() * 100)
            .round(2)
            .alias("percentage")
        )
        .sort("row_count", descending=True)
        .collect()
    )



--- marketing_channel ---
shape: (8, 3)
┌────────────────────┬───────────┬────────────┐
│ marketing_channel  ┆ row_count ┆ percentage │
│ ---                ┆ ---       ┆ ---        │
│ str                ┆ u32       ┆ f64        │
╞════════════════════╪═══════════╪════════════╡
│ Referral           ┆ 1543981   ┆ 16.68      │
│ Social Media       ┆ 1534777   ┆ 16.59      │
│ Paid Search        ┆ 1532502   ┆ 16.56      │
│ Email              ┆ 1532372   ┆ 16.56      │
│ Direct             ┆ 1525722   ┆ 16.49      │
│ Organic Search     ┆ 1522706   ┆ 16.45      │
│ seasonal_promotion ┆ 42449     ┆ 0.46       │
│ null               ┆ 19295     ┆ 0.21       │
└────────────────────┴───────────┴────────────┘

--- platform ---
shape: (3, 3)
┌────────────┬───────────┬────────────┐
│ platform   ┆ row_count ┆ percentage │
│ ---        ┆ ---       ┆ ---        │
│ str        ┆ u32       ┆ f64        │
╞════════════╪═══════════╪════════════╡
│ Mobile App ┆ 4630099   ┆ 50.03      │
│ Web        ┆ 

In [62]:

from pathlib import Path
import polars as pl

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

INPUT_PATH = BASE_DIR / "2019-Oct_week2.csv"
OUTPUT_PATH = BASE_DIR / "2019-Oct_week2_enriched_populated.csv"

# 1. Check input file
assert INPUT_PATH.exists(), f"Input file not found: {INPUT_PATH}"

# 2. Prevent overwriting an existing output
if OUTPUT_PATH.exists():
    raise FileExistsError(
        f"Output already exists: {OUTPUT_PATH}\n"
        "Choose another filename or verify the existing file first."
    )

# 3. Read Week 2 without automatic date parsing
week2 = pl.scan_csv(
    INPUT_PATH,
    infer_schema_length=10_000,
    try_parse_dates=False,
)

# 4. Validate the original 9-column schema
original_columns = [
    "event_time",
    "event_type",
    "product_id",
    "category_id",
    "category_code",
    "brand",
    "price",
    "user_id",
    "user_session",
]

assert week2.collect_schema().names() == original_columns, (
    "Unexpected input schema. Make sure the original 9-column Week 2 file is used."
)

# 5. Add synthetic marketing attributes using the Week 1 rules
week2_enriched = week2.with_columns(
    pl.when(pl.col("user_id") % 6 == 0)
      .then(pl.lit("Organic Search"))
    .when(pl.col("user_id") % 6 == 1)
      .then(pl.lit("Paid Search"))
    .when(pl.col("user_id") % 6 == 2)
      .then(pl.lit("Social Media"))
    .when(pl.col("user_id") % 6 == 3)
      .then(pl.lit("Email"))
    .when(pl.col("user_id") % 6 == 4)
      .then(pl.lit("Direct"))
    .otherwise(pl.lit("Referral"))
      .alias("marketing_channel"),

    pl.when(pl.col("user_id") % 2 == 0)
      .then(pl.lit("Web"))
      .otherwise(pl.lit("Mobile App"))
      .alias("platform"),
)

# 6. Save the enriched data
week2_enriched.sink_csv(
    OUTPUT_PATH,
    include_header=True,
    maintain_order=True,
    engine="streaming",
)

# 7. Validate the saved output
check = pl.scan_csv(
    OUTPUT_PATH,
    infer_schema_length=10_000,
    try_parse_dates=False,
)

row_count = check.select(pl.len()).collect().item()
column_names = check.collect_schema().names()

assert row_count == 10_071_265, (
    f"Unexpected row count: {row_count:,}"
)

assert len(column_names) == 11, (
    f"Expected 11 columns, got {len(column_names)}"
)

print("Week 2 enrichment completed successfully.")
print(f"Output: {OUTPUT_PATH}")
print(f"Rows: {row_count:,}")
print(f"Columns: {len(column_names)}")
print("Column names:", column_names)

# 8. Review generated values
print("\nMarketing channel distribution:")
print(
    check.group_by("marketing_channel")
         .agg(pl.len().alias("rows"))
         .sort("marketing_channel")
         .collect()
)

print("\nPlatform distribution:")
print(
    check.group_by("platform")
         .agg(pl.len().alias("rows"))
         .sort("platform")
         .collect()
)


Week 2 enrichment completed successfully.
Output: D:\Samsung\Graduation Project\Data\Old Data\2019-Oct_week2_enriched_populated.csv
Rows: 10,071,265
Columns: 11
Column names: ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session', 'marketing_channel', 'platform']

Marketing channel distribution:
shape: (6, 2)
┌───────────────────┬─────────┐
│ marketing_channel ┆ rows    │
│ ---               ┆ ---     │
│ str               ┆ u32     │
╞═══════════════════╪═════════╡
│ Direct            ┆ 1677608 │
│ Email             ┆ 1670722 │
│ Organic Search    ┆ 1664894 │
│ Paid Search       ┆ 1686127 │
│ Referral          ┆ 1690937 │
│ Social Media      ┆ 1680977 │
└───────────────────┴─────────┘

Platform distribution:
shape: (2, 2)
┌────────────┬─────────┐
│ platform   ┆ rows    │
│ ---        ┆ ---     │
│ str        ┆ u32     │
╞════════════╪═════════╡
│ Mobile App ┆ 5047786 │
│ Web        ┆ 5023479 │
└────────────┴─────────┘


In [61]:

from pathlib import Path

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")
OUTPUT_PATH = BASE_DIR / "2019-Oct_week2_enriched_populated.csv"

if OUTPUT_PATH.exists():
    OUTPUT_PATH.unlink()
    print("Old output file deleted successfully.")
else:
    print("Output file does not exist.")


Old output file deleted successfully.


In [63]:

from pathlib import Path
import polars as pl

# ==========================================
# WEEK 2 - ANOMALY INJECTION SETUP
# ==========================================

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

# Enriched Week 2 baseline
INPUT_PATH = BASE_DIR / "2019-Oct_week2_enriched_populated.csv"

# Separate Week 2 outputs
OUTPUT_PATH = BASE_DIR / "2019-Oct_week2_augmented.csv"
MANIFEST_PATH = BASE_DIR / "anomaly_manifest_week2.csv"

WEEK_NAME = "Week 2"
START_DATE = "2019-10-08"
END_DATE = "2019-10-14"

ORIGINAL_COLUMNS = [
    "event_time",
    "event_type",
    "product_id",
    "category_id",
    "category_code",
    "brand",
    "price",
    "user_id",
    "user_session",
]

NEW_COLUMNS = [
    "marketing_channel",
    "platform",
]

ALL_COLUMNS = ORIGINAL_COLUMNS + NEW_COLUMNS

WEEK2_TARGETS = {
    "Bot-like Crawling Loops": 53_061,
    "Seasonal Promotional Purchase Surges": 42_449,
    "Duplicate Events": 42_449,
    "Impossible Sequences": 42_449,
    "Cart Abandonment Scenarios": 31_837,
    "Missing Data": 212_244,
}

EXPECTED_INJECTED_ROWS = sum(WEEK2_TARGETS.values())

# ==========================================
# VALIDATION
# ==========================================

assert INPUT_PATH.exists(), (
    f"Week 2 enriched input not found: {INPUT_PATH}"
)

assert EXPECTED_INJECTED_ROWS == 424_489

# Never overwrite existing outputs
if OUTPUT_PATH.exists() or MANIFEST_PATH.exists():
    raise FileExistsError(
        "A Week 2 output or manifest already exists. "
        "Verify it before continuing; do not overwrite it."
    )

week2 = pl.scan_csv(
    INPUT_PATH,
    infer_schema_length=10_000,
    try_parse_dates=False,
)

actual_columns = week2.collect_schema().names()

assert actual_columns == ALL_COLUMNS, (
    f"Unexpected schema.\nExpected: {ALL_COLUMNS}\n"
    f"Actual: {actual_columns}"
)

row_count = week2.select(pl.len()).collect().item()

assert row_count == 10_071_265, (
    f"Unexpected Week 2 baseline row count: {row_count:,}"
)

# Verify the week boundaries using the ISO-like timestamp strings
date_stats = week2.select(
    pl.col("event_time").min().alias("min_event_time"),
    pl.col("event_time").max().alias("max_event_time"),
).collect()

print("Week 2 setup validated successfully.")
print(f"Input: {INPUT_PATH}")
print(f"Baseline rows: {row_count:,}")
print(f"Baseline columns: {len(actual_columns)}")
print(f"Date range: {date_stats}")
print(f"Planned anomaly rows: {EXPECTED_INJECTED_ROWS:,}")
print(f"Expected output rows: {row_count + EXPECTED_INJECTED_ROWS:,}")
print(f"Output: {OUTPUT_PATH}")
print(f"Manifest: {MANIFEST_PATH}")


Week 2 setup validated successfully.
Input: D:\Samsung\Graduation Project\Data\Old Data\2019-Oct_week2_enriched_populated.csv
Baseline rows: 10,071,265
Baseline columns: 11
Date range: shape: (1, 2)
┌─────────────────────────┬─────────────────────────┐
│ min_event_time          ┆ max_event_time          │
│ ---                     ┆ ---                     │
│ str                     ┆ str                     │
╞═════════════════════════╪═════════════════════════╡
│ 2019-10-08 00:00:00 UTC ┆ 2019-10-14 23:59:59 UTC │
└─────────────────────────┴─────────────────────────┘
Planned anomaly rows: 424,489
Expected output rows: 10,495,754
Output: D:\Samsung\Graduation Project\Data\Old Data\2019-Oct_week2_augmented.csv
Manifest: D:\Samsung\Graduation Project\Data\Old Data\anomaly_manifest_week2.csv


# Week 2 — Missing Data Injection

## 1. Objective

Inject **212,244 synthetic records** containing missing values into the Week 2 e-commerce dataset while preserving the original baseline file and documenting every injected record in an anomaly manifest.

## 2. Input and Output Files

| Item | File |
|---|---|
| Input dataset | `2019-Oct_week2_enriched_populated.csv` |
| Final augmented dataset | `2019-Oct_week2_augmented.csv` |
| Anomaly manifest | `anomaly_manifest_week2.csv` |
| Temporary output | `2019-Oct_week2_augmented.tmp.csv` |
| Temporary manifest | `anomaly_manifest_week2.tmp.csv` |

**Working directory:** `D:\Samsung\Graduation Project\Data\Old Data`

## 3. Baseline Dataset

| Property | Expected Value |
|---|---:|
| Baseline rows | 10,071,265 |
| Columns | 11 |
| Missing Data target | 212,244 |
| Expected final rows | 10,283,509 |

The dataset contains the original e-commerce event fields and two additional attributes: `marketing_channel` and `platform`.

## 4. Injection Method

1. Validate the input file, column names, and baseline row count.
2. Select 212,244 source rows distributed across the dataset using a deterministic row-index formula.
3. Copy each selected record into a new synthetic record.
4. Select a non-null column and replace its value with `null`.
5. Assign a unique synthetic ID using the prefix `W2-MD-`, for example, `W2-MD-000001`.
6. Record the injected anomaly and source-row metadata in the manifest.
7. Append the generated records to a temporary copy of the baseline dataset without adding another CSV header.
8. Validate the temporary output and manifest before publishing the final files.

## 5. Manifest Schema

| Column | Description |
|---|---|
| `week` | Week identifier |
| `anomaly_type` | `Missing Data` |
| `synthetic_row_id` | Unique synthetic record ID |
| `source_row_number` | One-based source row number |
| `missing_column` | Column selected for null injection |
| `source_event_time` | Source event timestamp |
| `source_event_type` | Source event type |
| `source_user_id` | Source user identifier |
| `source_product_id` | Source product identifier |

## 6. Validation Checks

The implementation checks that:

- The input file exists and has the expected schema.
- The baseline contains exactly 10,071,265 rows.
- Exactly 212,244 source records are selected.
- The generated dataset contains 212,244 records.
- Synthetic IDs are unique.
- Generated records contain at least one null value.
- The manifest contains 212,244 records with the expected columns.
- The completed temporary dataset contains 10,283,509 rows.

The baseline file remains unchanged because the injection is written to a separate temporary output file.

## 7. Expected Results

| Metric | Expected Result |
|---|---:|
| Baseline rows | 10,071,265 |
| Synthetic Missing Data rows | 212,244 |
| Final rows | 10,283,509 |
| Manifest records | 212,244 |

The code also reports the number of injected missing values per column.

## 8. Important Notes

- The process adds synthetic rows rather than deleting or modifying baseline records.
- Existing null values may already exist in the source data.
- Each generated record receives one newly injected null in a selected non-null column. It may still contain other null values inherited from its source record.
- The manifest supports traceability by connecting each synthetic record to its source row and injected missing column.
- If any output or temporary file already exists, the script stops to avoid accidental overwriting.
- The figures above are expected values; they should be treated as actual results only after the script runs successfully and the validations pass.

## 9. Next Step

After validating the Missing Data injection, continue with the next anomaly in the agreed Week 2 sequence: **Bot-like Crawling Loops**.


In [64]:

import polars as pl
import shutil
import os
from pathlib import Path

# =========================================================
# 0. Week 2 paths and configuration
# =========================================================

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

INPUT_PATH = BASE_DIR / "2019-Oct_week2_enriched_populated.csv"
FINAL_PATH = BASE_DIR / "2019-Oct_week2_augmented.csv"
MANIFEST_PATH = BASE_DIR / "anomaly_manifest_week2.csv"

TEMP_OUTPUT_PATH = BASE_DIR / "2019-Oct_week2_augmented.tmp.csv"
TEMP_MANIFEST_PATH = BASE_DIR / "anomaly_manifest_week2.tmp.csv"

WEEK_NAME = "Week 2"
TARGET_ROWS = 212_244
EXPECTED_BASE_ROWS = 10_071_265

ALL_COLUMNS = [
    "event_time",
    "event_type",
    "product_id",
    "category_id",
    "category_code",
    "brand",
    "price",
    "user_id",
    "user_session",
    "marketing_channel",
    "platform",
]

MANIFEST_COLUMNS = [
    "week",
    "anomaly_type",
    "synthetic_row_id",
    "source_row_number",
    "missing_column",
    "source_event_time",
    "source_event_type",
    "source_user_id",
    "source_product_id",
]

# =========================================================
# 1. Validate input and prevent accidental overwriting
# =========================================================

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_PATH}")

for path in [
    FINAL_PATH,
    MANIFEST_PATH,
    TEMP_OUTPUT_PATH,
    TEMP_MANIFEST_PATH,
]:
    if path.exists():
        raise FileExistsError(
            f"File already exists: {path}\n"
            "Inspect it before running again. Nothing was overwritten."
        )

source = pl.scan_csv(
    INPUT_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
    try_parse_dates=False,
)

schema = source.collect_schema()
base_count = source.select(pl.len()).collect().item()

assert schema.names() == ALL_COLUMNS, (
    f"Unexpected columns: {schema.names()}"
)

assert base_count == EXPECTED_BASE_ROWS, (
    f"Expected {EXPECTED_BASE_ROWS:,} rows, found {base_count:,}"
)

print(f"Week 2 base rows: {base_count:,}")
print(f"Missing Data target: {TARGET_ROWS:,}")

# =========================================================
# 2. Select rows distributed across the entire dataset
# =========================================================

indexed = source.with_row_index("_source_row")
idx = pl.col("_source_row").cast(pl.Int64)

candidates = (
    indexed
    .filter(
        (((idx + 1) * TARGET_ROWS) // base_count)
        > ((idx * TARGET_ROWS) // base_count)
    )
    .collect(engine="streaming")
)

assert candidates.height == TARGET_ROWS, (
    f"Expected {TARGET_ROWS:,} candidates, "
    f"found {candidates.height:,}"
)

print(f"Selected source rows: {candidates.height:,}")

# =========================================================
# 3. Generate Missing Data records and manifest
# =========================================================

generated_rows = []
manifest_rows = []

for number, record in enumerate(
    candidates.iter_rows(named=True),
    start=1,
):
    source_row_number = int(record["_source_row"]) + 1

    # Rotate through columns to distribute missing values.
    # If a selected column is already null, try the next one.
    start = (number - 1) % len(ALL_COLUMNS)
    missing_column = None

    for offset in range(len(ALL_COLUMNS)):
        column = ALL_COLUMNS[
            (start + offset) % len(ALL_COLUMNS)
        ]

        if record[column] is not None:
            missing_column = column
            break

    if missing_column is None:
        raise ValueError(
            f"Source row {source_row_number} has no non-null values."
        )

    # Copy all original values, then introduce one missing value.
    new_record = {
        col: record[col]
        for col in ALL_COLUMNS
    }
    new_record[missing_column] = None
    generated_rows.append(new_record)

    manifest_rows.append({
        "week": WEEK_NAME,
        "anomaly_type": "Missing Data",
        "synthetic_row_id": f"W2-MD-{number:06d}",
        "source_row_number": source_row_number,
        "missing_column": missing_column,
        "source_event_time": record["event_time"],
        "source_event_type": record["event_type"],
        "source_user_id": record["user_id"],
        "source_product_id": record["product_id"],
    })

# =========================================================
# 4. Build and validate the generated DataFrames
# =========================================================

missing_df = pl.DataFrame(
    generated_rows,
    schema=schema,
    orient="row",
    strict=False,
)

manifest_df = pl.DataFrame(
    manifest_rows,
    schema=MANIFEST_COLUMNS,
    orient="row",
    strict=False,
)

assert missing_df.height == TARGET_ROWS
assert missing_df.columns == ALL_COLUMNS

assert manifest_df.height == TARGET_ROWS
assert manifest_df.columns == MANIFEST_COLUMNS
assert (
    manifest_df["synthetic_row_id"].n_unique()
    == TARGET_ROWS
)

# Each generated row must contain at least one null.
null_counts = missing_df.select(
    pl.sum_horizontal([
        pl.col(col).is_null().cast(pl.Int64)
        for col in ALL_COLUMNS
    ]).alias("null_count")
)

assert null_counts["null_count"].min() >= 1

# Each injected row should have exactly one newly injected null.
# Existing nulls in other columns may still be present.
print("Generated rows and manifest validated.")

# =========================================================
# 5. Write temporary files, preserving the original input
# =========================================================

try:
    # Copy the enriched baseline to a temporary output file.
    shutil.copyfile(INPUT_PATH, TEMP_OUTPUT_PATH)

    # Append synthetic rows without adding a second CSV header.
    with open(TEMP_OUTPUT_PATH, "ab") as output_file:
        missing_df.write_csv(
            output_file,
            include_header=False,
        )

    # Save manifest to a temporary file.
    manifest_df.write_csv(TEMP_MANIFEST_PATH)

    # Validate temporary manifest.
    saved_manifest = pl.read_csv(TEMP_MANIFEST_PATH)

    assert saved_manifest.height == TARGET_ROWS
    assert (
        saved_manifest["synthetic_row_id"].n_unique()
        == TARGET_ROWS
    )

    # Validate the completed temporary output.
    final_count = (
        pl.scan_csv(
            TEMP_OUTPUT_PATH,
            infer_schema_length=10_000,
            null_values=["", "null", "NULL"],
            try_parse_dates=False,
        )
        .select(pl.len())
        .collect()
        .item()
    )

    expected_final_rows = base_count + TARGET_ROWS

    assert final_count == expected_final_rows, (
        f"Expected {expected_final_rows:,} rows, "
        f"found {final_count:,}"
    )

    # Publish final files only after successful validation.
    os.replace(TEMP_OUTPUT_PATH, FINAL_PATH)
    os.replace(TEMP_MANIFEST_PATH, MANIFEST_PATH)

except Exception:
    # Keep any remaining temporary files for inspection.
    print(
        "An error occurred. Final files were not intentionally "
        "published before validation. Inspect temporary files "
        "before retrying."
    )
    raise

# =========================================================
# 6. Final report
# =========================================================

print("\nMissing Data injection completed successfully.")
print(f"Rows before injection: {base_count:,}")
print(f"Rows added: {TARGET_ROWS:,}")
print(f"Expected total rows: {base_count + TARGET_ROWS:,}")
print(f"Validated output rows: {final_count:,}")
print(f"Manifest records: {saved_manifest.height:,}")
print(f"Output file: {FINAL_PATH}")
print(f"Manifest file: {MANIFEST_PATH}")

print("\nInjected missing values by column:")
print(
    manifest_df
    .group_by("missing_column")
    .agg(pl.len().alias("injected_nulls"))
    .sort("missing_column")
)


Week 2 base rows: 10,071,265
Missing Data target: 212,244
Selected source rows: 212,244
Generated rows and manifest validated.

Missing Data injection completed successfully.
Rows before injection: 10,071,265
Rows added: 212,244
Expected total rows: 10,283,509
Validated output rows: 10,283,509
Manifest records: 212,244
Output file: D:\Samsung\Graduation Project\Data\Old Data\2019-Oct_week2_augmented.csv
Manifest file: D:\Samsung\Graduation Project\Data\Old Data\anomaly_manifest_week2.csv

Injected missing values by column:
shape: (11, 2)
┌────────────────┬────────────────┐
│ missing_column ┆ injected_nulls │
│ ---            ┆ ---            │
│ str            ┆ u32            │
╞════════════════╪════════════════╡
│ brand          ┆ 21381          │
│ category_code  ┆ 12696          │
│ category_id    ┆ 19295          │
│ event_time     ┆ 19295          │
│ event_type     ┆ 19295          │
│ …              ┆ …              │
│ platform       ┆ 19294          │
│ price          ┆ 23808

# Week 2 — Bot-like Crawling Loops Injection

## 1. Objective

Simulate automated bot-like browsing behavior in the e-commerce event dataset by generating synthetic product-view events that repeat the same user's view of the same product within the same session after a short time interval.

The objective is to enrich the dataset with suspicious browsing patterns that can be analyzed by downstream data-quality and behavioral analytics pipelines.

## 2. Input and Output Files

| Item | File |
|---|---|
| Input dataset | `2019-Oct_week2_augmented.csv` |
| Final augmented dataset | `2019-Oct_week2_augmented.csv` |
| Anomaly manifest | `anomaly_manifest_week2.csv` |
| Temporary output | `2019-Oct_week2_bot.tmp.csv` |
| Temporary manifest | `anomaly_manifest_week2_bot.tmp.csv` |

**Working directory:** `D:\Samsung\Graduation Project\Data\Old Data`

The input dataset already includes the previously injected Missing Data records.

## 3. Injection Configuration

| Parameter | Value |
|---|---:|
| Week | Week 2 |
| Anomaly type | Bot-like Crawling Loops |
| Target synthetic rows | 53,061 |
| Existing dataset rows | 10,283,509 |
| Expected dataset rows after injection | 10,336,570 |
| Synthetic ID prefix | `W2-BL-` |
| Delay between source and synthetic event | 1–3 seconds |

## 4. Source Event Selection

The implementation selects eligible source records from the existing dataset.

A source record must satisfy these conditions:

- `event_type` is `view`.
- `event_time` is not null.
- `user_id` is not null.
- `product_id` is not null.
- `user_session` is not null.
- Its timestamp is sufficiently early to allow the synthetic event to remain within the selected Week 2 time window.

The target source records are distributed across the eligible view events using a deterministic row-index formula.

## 5. Synthetic Event Generation

For each selected source event:

1. Copy the source record.
2. Preserve its `user_id`, `product_id`, `user_session`, and other event attributes.
3. Keep `event_type` equal to `view`.
4. Shift `event_time` forward by 1, 2, or 3 seconds.
5. Assign a unique synthetic ID such as `W2-BL-000001`.
6. Create a manifest record that identifies the source event and the generated anomaly.

This approach simulates rapid repeated product views within the same session.

**Limitation:** The implementation creates one synthetic event per selected source event. It does not guarantee long, consecutive crawling loops for every session or prove that the behavior came from an actual bot.

## 6. Manifest Schema

The existing Week 2 manifest is retained and extended with the new anomaly records.

| Column | Description |
|---|---|
| `week` | Week identifier |
| `anomaly_type` | `Bot-like Crawling Loops` |
| `synthetic_row_id` | Unique synthetic record ID |
| `source_row_number` | One-based source row number |
| `missing_column` | Null for this anomaly type |
| `source_event_time` | Original source timestamp |
| `source_event_type` | Original source event type |
| `source_user_id` | Source user identifier |
| `source_product_id` | Source product identifier |

## 7. Validation Checks

Before publishing the temporary files, the implementation verifies that:

- The input dataset exists and has the expected columns.
- The baseline contains exactly 10,283,509 rows.
- The existing manifest contains 212,244 Missing Data records.
- The required number of eligible source events is available.
- Exactly 53,061 synthetic records are generated.
- Every generated record has `event_type = view`.
- The new synthetic IDs are unique.
- The combined manifest retains the existing Missing Data records.
- The combined manifest contains 265,305 records.
- The temporary output contains 10,336,570 rows.

The temporary output and combined manifest are published only after these checks pass.

## 8. Expected Results

| Metric | Expected Value |
|---|---:|
| Dataset rows before injection | 10,283,509 |
| Synthetic Bot-like rows | 53,061 |
| Dataset rows after injection | 10,336,570 |
| Existing Missing Data manifest records | 212,244 |
| New Bot-like manifest records | 53,061 |
| Total manifest records | 265,305 |

These are expected values. Actual success must be confirmed from the execution output.

## 9. Analytical Use Cases

The generated events can support downstream analysis such as:

- Detecting unusually rapid repeated product views.
- Identifying sessions with excessive browsing activity.
- Comparing normal and suspicious browsing patterns.
- Measuring the frequency of repeated views by user, product, or session.

These are candidate analytical use cases, not validated detection results.

## 10. Next Step

After successful validation, proceed to the next anomaly in the agreed Week 2 injection sequence:

**Seasonal Promotional Purchase Surges**.


In [65]:

import polars as pl
import shutil
import os
from pathlib import Path
from datetime import datetime, timedelta

# =========================================================
# 0. Week 2 configuration
# =========================================================

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

FINAL_PATH = BASE_DIR / "2019-Oct_week2_augmented.csv"
MANIFEST_PATH = BASE_DIR / "anomaly_manifest_week2.csv"

TEMP_OUTPUT_PATH = BASE_DIR / "2019-Oct_week2_bot.tmp.csv"
TEMP_MANIFEST_PATH = BASE_DIR / "anomaly_manifest_week2_bot.tmp.csv"

WEEK_NAME = "Week 2"
ANOMALY_NAME = "Bot-like Crawling Loops"

TARGET_ROWS = 53_061
EXPECTED_BASE_ROWS = 10_283_509
EXPECTED_MD_ROWS = 212_244

ALL_COLUMNS = [
    "event_time",
    "event_type",
    "product_id",
    "category_id",
    "category_code",
    "brand",
    "price",
    "user_id",
    "user_session",
    "marketing_channel",
    "platform",
]

MANIFEST_COLUMNS = [
    "week",
    "anomaly_type",
    "synthetic_row_id",
    "source_row_number",
    "missing_column",
    "source_event_time",
    "source_event_type",
    "source_user_id",
    "source_product_id",
]

# =========================================================
# 1. Validate current files and prevent duplicate injection
# =========================================================

if not FINAL_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {FINAL_PATH}")

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST_PATH}")

for path in [TEMP_OUTPUT_PATH, TEMP_MANIFEST_PATH]:
    if path.exists():
        raise FileExistsError(
            f"Temporary file already exists: {path}. "
            "Inspect it before retrying."
        )

source = pl.scan_csv(
    FINAL_PATH,
    infer_schema_length=10_000,
    null_values=["", "null", "NULL"],
    try_parse_dates=False,
)

schema = source.collect_schema()
base_count = source.select(pl.len()).collect().item()

assert schema.names() == ALL_COLUMNS
assert base_count == EXPECTED_BASE_ROWS, (
    f"Expected {EXPECTED_BASE_ROWS:,} rows, found {base_count:,}"
)

existing_manifest = pl.read_csv(
    MANIFEST_PATH,
    infer_schema_length=10_000,
    try_parse_dates=False,
)

assert existing_manifest.columns == MANIFEST_COLUMNS

assert existing_manifest.height == EXPECTED_MD_ROWS, (
    f"Expected {EXPECTED_MD_ROWS:,} existing manifest records, "
    f"found {existing_manifest.height:,}"
)

assert (
    existing_manifest
    .filter(pl.col("anomaly_type") == ANOMALY_NAME)
    .height == 0
), "Bot-like Crawling Loops already exists in the manifest."

assert (
    existing_manifest["synthetic_row_id"].n_unique()
    == existing_manifest.height
), "Existing manifest contains duplicate synthetic IDs."

print(f"Current dataset rows: {base_count:,}")
print(f"Existing manifest rows: {existing_manifest.height:,}")
print(f"Bot-like target: {TARGET_ROWS:,}")

# =========================================================
# 2. Find eligible view events
# =========================================================

# Keep the original row index for source traceability.
# Exclude incomplete records needed to simulate repeated views.
# Exclude the final few seconds of the week so shifted timestamps
# remain within the Week 2 date window.

eligible = (
    source
    .with_row_index("_source_row")
    .filter(
        (pl.col("event_type") == "view")
        & pl.col("event_time").is_not_null()
        & pl.col("user_id").is_not_null()
        & pl.col("product_id").is_not_null()
        & pl.col("user_session").is_not_null()
        & (
            pl.col("event_time")
            < pl.lit("2019-10-14 23:59:56 UTC")
        )
    )
    .with_row_index("_candidate_index")
)

eligible_count = (
    eligible
    .select(pl.len())
    .collect()
    .item()
)

assert eligible_count >= TARGET_ROWS, (
    f"Not enough eligible view events: {eligible_count:,}"
)

# Select candidates evenly across eligible view events.
candidate_idx = pl.col("_candidate_index").cast(pl.Int64)

candidates = (
    eligible
    .filter(
        (
            ((candidate_idx + 1) * TARGET_ROWS)
            // eligible_count
        )
        >
        (
            (candidate_idx * TARGET_ROWS)
            // eligible_count
        )
    )
    .collect(engine="streaming")
)

assert candidates.height == TARGET_ROWS, (
    f"Expected {TARGET_ROWS:,} candidates, "
    f"found {candidates.height:,}"
)

print(f"Eligible view events: {eligible_count:,}")
print(f"Selected source events: {candidates.height:,}")

# =========================================================
# 3. Generate bot-like repeated views
# =========================================================

generated_rows = []
manifest_rows = []

for number, record in enumerate(
    candidates.iter_rows(named=True),
    start=1,
):
    source_row_number = int(record["_source_row"]) + 1

    # Repeat the same user's view of the same product
    # in the same session after a short interval.
    delay_seconds = ((number - 1) % 3) + 1

    source_time = record["event_time"]

    # Expected timestamp format:
    # 2019-10-08 01:58:05 UTC
    timestamp = datetime.strptime(
        source_time,
        "%Y-%m-%d %H:%M:%S UTC",
    )

    shifted_timestamp = (
        timestamp + timedelta(seconds=delay_seconds)
    )

    new_record = {
        col: record[col]
        for col in ALL_COLUMNS
    }

    new_record["event_time"] = (
        shifted_timestamp.strftime("%Y-%m-%d %H:%M:%S UTC")
    )

    # Keep event_type, product_id, user_id, and user_session
    # unchanged to simulate repeated product-view requests.
    assert new_record["event_type"] == "view"

    generated_rows.append(new_record)

    manifest_rows.append({
        "week": WEEK_NAME,
        "anomaly_type": ANOMALY_NAME,
        "synthetic_row_id": f"W2-BL-{number:06d}",
        "source_row_number": source_row_number,
        "missing_column": None,
        "source_event_time": source_time,
        "source_event_type": record["event_type"],
        "source_user_id": record["user_id"],
        "source_product_id": record["product_id"],
    })

# =========================================================
# 4. Build and validate generated data
# =========================================================

bot_df = pl.DataFrame(
    generated_rows,
    schema=schema,
    orient="row",
    strict=False,
)

bot_manifest = pl.DataFrame(
    manifest_rows,
    schema=existing_manifest.schema,
    orient="row",
    strict=False,
)

assert bot_df.height == TARGET_ROWS
assert bot_df.columns == ALL_COLUMNS

assert bot_manifest.height == TARGET_ROWS
assert bot_manifest.columns == MANIFEST_COLUMNS

assert (
    bot_manifest["synthetic_row_id"].n_unique()
    == TARGET_ROWS
)

assert bot_df.filter(
    pl.col("event_type") != "view"
).height == 0

# Preserve all previous manifest records.
combined_manifest = pl.concat(
    [existing_manifest, bot_manifest],
    how="vertical_relaxed",
)

expected_manifest_rows = EXPECTED_MD_ROWS + TARGET_ROWS

assert combined_manifest.height == expected_manifest_rows
assert (
    combined_manifest["synthetic_row_id"].n_unique()
    == expected_manifest_rows
)

print("Generated bot records and combined manifest validated.")

# =========================================================
# 5. Write temporary files and validate before publishing
# =========================================================

try:
    # Copy the current dataset, which already includes Missing Data.
    shutil.copyfile(FINAL_PATH, TEMP_OUTPUT_PATH)

    # Append bot records without adding another CSV header.
    with open(TEMP_OUTPUT_PATH, "ab") as output_file:
        bot_df.write_csv(
            output_file,
            include_header=False,
        )

    # Save the combined manifest, retaining Missing Data records.
    combined_manifest.write_csv(TEMP_MANIFEST_PATH)

    # Validate manifest written to disk.
    saved_manifest = pl.read_csv(
        TEMP_MANIFEST_PATH,
        infer_schema_length=10_000,
        try_parse_dates=False,
    )

    assert saved_manifest.height == expected_manifest_rows
    assert (
        saved_manifest["synthetic_row_id"].n_unique()
        == expected_manifest_rows
    )

    assert (
        saved_manifest
        .filter(pl.col("anomaly_type") == "Missing Data")
        .height == EXPECTED_MD_ROWS
    )

    assert (
        saved_manifest
        .filter(pl.col("anomaly_type") == ANOMALY_NAME)
        .height == TARGET_ROWS
    )

    # Validate total dataset rows.
    expected_final_rows = base_count + TARGET_ROWS

    final_count = (
        pl.scan_csv(
            TEMP_OUTPUT_PATH,
            infer_schema_length=10_000,
            null_values=["", "null", "NULL"],
            try_parse_dates=False,
        )
        .select(pl.len())
        .collect()
        .item()
    )

    assert final_count == expected_final_rows, (
        f"Expected {expected_final_rows:,} rows, "
        f"found {final_count:,}"
    )

    # Publish validated files.
    os.replace(TEMP_OUTPUT_PATH, FINAL_PATH)
    os.replace(TEMP_MANIFEST_PATH, MANIFEST_PATH)

except Exception:
    print(
        "Injection failed or validation did not pass. "
        "Inspect temporary files before retrying."
    )
    raise

# =========================================================
# 6. Final report
# =========================================================

print("\nBot-like Crawling Loops injection completed successfully.")
print(f"Rows before injection: {base_count:,}")
print(f"Bot-like rows added: {TARGET_ROWS:,}")
print(f"Final dataset rows: {final_count:,}")
print(f"Expected manifest rows: {expected_manifest_rows:,}")
print(f"Validated manifest rows: {saved_manifest.height:,}")
print(f"Output file: {FINAL_PATH}")
print(f"Manifest file: {MANIFEST_PATH}")

print("\nManifest records by anomaly:")
print(
    saved_manifest
    .group_by("anomaly_type")
    .agg(pl.len().alias("record_count"))
    .sort("anomaly_type")
)


Current dataset rows: 10,283,509
Existing manifest rows: 212,244
Bot-like target: 53,061
Eligible view events: 9,837,996
Selected source events: 53,061
Generated bot records and combined manifest validated.

Bot-like Crawling Loops injection completed successfully.
Rows before injection: 10,283,509
Bot-like rows added: 53,061
Final dataset rows: 10,336,570
Expected manifest rows: 265,305
Validated manifest rows: 265,305
Output file: D:\Samsung\Graduation Project\Data\Old Data\2019-Oct_week2_augmented.csv
Manifest file: D:\Samsung\Graduation Project\Data\Old Data\anomaly_manifest_week2.csv

Manifest records by anomaly:
shape: (2, 2)
┌─────────────────────────┬──────────────┐
│ anomaly_type            ┆ record_count │
│ ---                     ┆ ---          │
│ str                     ┆ u32          │
╞═════════════════════════╪══════════════╡
│ Bot-like Crawling Loops ┆ 53061        │
│ Missing Data            ┆ 212244       │
└─────────────────────────┴──────────────┘


# Week 2 — Seasonal Promotional Purchase Surges Injection

## 1. Objective
Simulate a seasonal promotional period with an artificial increase in purchase events. This enables analysis of unusual purchase spikes and their potential impact on sales analytics.

## 2. Input and Output

**Dataset:** `2019-Oct_week2_augmented.csv`  
**Manifest:** `anomaly_manifest_week2.csv`

The input already included the Missing Data and Bot-like Crawling Loops anomalies.

## 3. Injection Configuration

| Parameter | Value |
|---|---:|
| Week | Week 2 |
| Anomaly type | Seasonal Promotional Purchase Surges |
| Target synthetic records | 42,449 |
| Eligible purchase events | 179,051 |
| Selected source events | 42,449 |
| Promotion start | 2019-10-13 10:00:00 UTC |
| Promotion end | 2019-10-13 12:00:00 UTC |

## 4. Generation Method

1. Select eligible purchase events with non-null values in the required fields.
2. Distribute the selected source records across the eligible purchases.
3. Generate 42,449 synthetic purchase events.
4. Assign timestamps across the two-hour promotional window.
5. Preserve the source purchase's user, product, session, price, and other contextual values.
6. Assign unique synthetic IDs using the `W2-SP-` prefix.
7. Append the synthetic rows to the dataset and their metadata to the existing manifest.

This creates a concentrated purchase spike in a predefined time window.

## 5. Validation Results

| Validation | Expected | Actual | Status |
|---|---:|---:|---|
| Rows before injection | 10,336,570 | 10,336,570 | Passed |
| Synthetic promotional purchases | 42,449 | 42,449 | Passed |
| Final dataset rows | 10,379,019 | 10,379,019 | Passed |
| Manifest before injection | 265,305 | 265,305 | Passed |
| Final manifest records | 307,754 | 307,754 | Passed |

The generated timestamps ranged from `2019-10-13 10:00:00 UTC` to `2019-10-13 11:59:59 UTC`.

## 6. Manifest Distribution

| Anomaly type | Record count |
|---|---:|
| Missing Data | 212,244 |
| Bot-like Crawling Loops | 53,061 |
| Seasonal Promotional Purchase Surges | 42,449 |
| **Total** | **307,754** |

## 7. Output Files

- **Augmented dataset:** `D:\Samsung\Graduation Project\Data\Old Data\2019-Oct_week2_augmented.csv`
- **Anomaly manifest:** `D:\Samsung\Graduation Project\Data\Old Data\anomaly_manifest_week2.csv`

## 8. Analytical Use Cases

The synthetic data can support:
- Purchase-volume analysis during promotional windows.
- Comparison of purchase activity inside and outside the promotional period.
- Detection of unusual increases in purchase events.
- Time-based sales trend analysis.

**Important limitation:** This is a simulated promotional spike, not evidence that an actual campaign occurred. Since source purchase attributes are reused, the generated events should be identified as synthetic when evaluating business outcomes.

## 9. Conclusion

The Seasonal Promotional Purchase Surges injection completed successfully. The dataset now contains **10,379,019 rows**, and the manifest contains **307,754 records** across three anomaly types. All reported row-count and manifest validations passed.


In [66]:

from pathlib import Path
from datetime import datetime, timedelta
import os
import shutil
import polars as pl

# ============================================================
# 1. Configuration
# ============================================================

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

FINAL_PATH = BASE_DIR / "2019-Oct_week2_augmented.csv"
MANIFEST_PATH = BASE_DIR / "anomaly_manifest_week2.csv"

TEMP_OUTPUT_PATH = BASE_DIR / "2019-Oct_week2_promo.tmp.csv"
TEMP_ROWS_PATH = BASE_DIR / "2019-Oct_week2_promo_rows.tmp.csv"
TEMP_MANIFEST_PATH = BASE_DIR / "anomaly_manifest_week2_promo.tmp.csv"

ANOMALY_NAME = "Seasonal Promotional Purchase Surges"
TARGET_ROWS = 42_449

EXPECTED_BASE_ROWS = 10_336_570
EXPECTED_MANIFEST_ROWS = 265_305

EXPECTED_PREVIOUS = {
    "Missing Data": 212_244,
    "Bot-like Crawling Loops": 53_061,
}

PROMO_START = datetime(2019, 10, 13, 10, 0, 0)
PROMO_END = datetime(2019, 10, 13, 12, 0, 0)

ALL_COLUMNS = [
    "event_time",
    "event_type",
    "product_id",
    "category_id",
    "category_code",
    "brand",
    "price",
    "user_id",
    "user_session",
    "marketing_channel",
    "platform",
]

MANIFEST_COLUMNS = [
    "week",
    "anomaly_type",
    "synthetic_row_id",
    "source_row_number",
    "missing_column",
    "source_event_time",
    "source_event_type",
    "source_user_id",
    "source_product_id",
]


# ============================================================
# 2. Validate input files and previous stages
# ============================================================

if not FINAL_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {FINAL_PATH}")

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST_PATH}")

for path in [TEMP_OUTPUT_PATH, TEMP_ROWS_PATH, TEMP_MANIFEST_PATH]:
    if path.exists():
        raise FileExistsError(
            f"Temporary file already exists: {path}\n"
            "Inspect it before deleting or rerunning the cell."
        )

base_lf = pl.scan_csv(
    FINAL_PATH,
    infer_schema_length=10_000,
    try_parse_dates=False,
)

base_schema = base_lf.collect_schema()
if list(base_schema.names()) != ALL_COLUMNS:
    raise ValueError(
        f"Unexpected dataset columns: {base_schema.names()}"
    )

base_rows = base_lf.select(pl.len()).collect().item()

manifest = pl.read_csv(
    MANIFEST_PATH,
    infer_schema_length=10_000,
    try_parse_dates=False,
)

if base_rows != EXPECTED_BASE_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_BASE_ROWS:,} dataset rows, "
        f"found {base_rows:,}."
    )

if manifest.height != EXPECTED_MANIFEST_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_MANIFEST_ROWS:,} manifest rows, "
        f"found {manifest.height:,}."
    )

if manifest.columns != MANIFEST_COLUMNS:
    raise ValueError(
        f"Unexpected manifest columns: {manifest.columns}"
    )

if manifest.filter(
    pl.col("anomaly_type") == ANOMALY_NAME
).height > 0:
    raise ValueError(
        "This anomaly already exists in the manifest. "
        "Do not inject it twice."
    )

existing_counts = {
    row["anomaly_type"]: row["len"]
    for row in (
        manifest.group_by("anomaly_type")
        .len()
        .iter_rows(named=True)
    )
}

for anomaly, expected_count in EXPECTED_PREVIOUS.items():
    actual_count = existing_counts.get(anomaly, 0)
    if actual_count != expected_count:
        raise ValueError(
            f"{anomaly}: expected {expected_count:,}, "
            f"found {actual_count:,}."
        )

print(f"Current dataset rows: {base_rows:,}")
print(f"Existing manifest rows: {manifest.height:,}")
print(f"Promotional purchase target: {TARGET_ROWS:,}")


# ============================================================
# 3. Select eligible purchase events
# ============================================================

# Add a source row number before filtering, so manifest references
# point back to the row's position in the current dataset.
candidates = (
    base_lf
    .with_row_index("_source_row", offset=1)
    .filter(
        (pl.col("event_type") == "purchase")
        & pl.col("event_time").is_not_null()
        & pl.col("product_id").is_not_null()
        & pl.col("user_id").is_not_null()
        & pl.col("user_session").is_not_null()
        & pl.col("price").is_not_null()
    )
    .select(["_source_row"] + ALL_COLUMNS)
    .collect()
)

eligible_count = candidates.height

if eligible_count < TARGET_ROWS:
    raise ValueError(
        f"Not enough eligible purchase events: {eligible_count:,}"
    )

# Spread source selections across eligible purchases deterministically.
step = eligible_count / TARGET_ROWS

selected_positions = [
    min(int(i * step), eligible_count - 1)
    for i in range(TARGET_ROWS)
]

selected = candidates[selected_positions]

if selected.height != TARGET_ROWS:
    raise ValueError("Selected source count does not match target.")

print(f"Eligible purchase events: {eligible_count:,}")
print(f"Selected source events: {selected.height:,}")


# ============================================================
# 4. Generate synthetic purchases in the promo window
# ============================================================

synthetic_rows = []
manifest_rows = []

duration_seconds = int((PROMO_END - PROMO_START).total_seconds())

for i, source in enumerate(selected.iter_rows(named=True), start=1):
    # Deterministically spread purchases across the two-hour window.
    offset_seconds = int((i - 1) * duration_seconds / TARGET_ROWS)
    promo_time = PROMO_START + timedelta(seconds=offset_seconds)

    synthetic = {
        column: source[column]
        for column in ALL_COLUMNS
    }

    synthetic["event_time"] = promo_time.strftime(
        "%Y-%m-%d %H:%M:%S UTC"
    )
    synthetic["event_type"] = "purchase"

    synthetic_rows.append(synthetic)

    manifest_rows.append({
        "week": "Week 2",
        "anomaly_type": ANOMALY_NAME,
        "synthetic_row_id": f"W2-SP-{i:06d}",
        "source_row_number": source["_source_row"],
        "missing_column": None,
        "source_event_time": source["event_time"],
        "source_event_type": source["event_type"],
        "source_user_id": source["user_id"],
        "source_product_id": source["product_id"],
    })

synthetic_df = pl.DataFrame(
    synthetic_rows,
    schema=ALL_COLUMNS,
    orient="row",
    strict=False,
)

new_manifest = pl.DataFrame(
    manifest_rows,
    schema=manifest.schema,
    orient="row",
    strict=False,
)

if synthetic_df.height != TARGET_ROWS:
    raise ValueError("Generated row count is incorrect.")

if synthetic_df.filter(
    pl.col("event_type") != "purchase"
).height:
    raise ValueError("Non-purchase events found in generated rows.")

if synthetic_df.filter(
    pl.col("event_time").is_null()
    | pl.col("user_id").is_null()
    | pl.col("product_id").is_null()
    | pl.col("user_session").is_null()
    | pl.col("price").is_null()
).height:
    raise ValueError("Generated records have missing required fields.")

if new_manifest.filter(
    pl.col("anomaly_type") != ANOMALY_NAME
).height:
    raise ValueError("Unexpected anomaly type in new manifest.")

if new_manifest.select(
    pl.col("synthetic_row_id").n_unique()
).item() != TARGET_ROWS:
    raise ValueError("Synthetic IDs are not unique.")

promo_times = synthetic_df.select(
    pl.col("event_time").min().alias("min_time"),
    pl.col("event_time").max().alias("max_time"),
).row(0, named=True)

print("Generated promotional purchase records validated.")
print(f"Promo window: {PROMO_START} UTC to {PROMO_END} UTC")
print(f"Generated records: {synthetic_df.height:,}")
print(f"First generated timestamp: {promo_times['min_time']}")
print(f"Last generated timestamp: {promo_times['max_time']}")


# ============================================================
# 5. Prepare temporary files
# ============================================================

try:
    # Write only synthetic rows, without a header, for appending.
    synthetic_df.write_csv(
        TEMP_ROWS_PATH,
        include_header=False,
    )

    # Prepare a complete updated manifest.
    combined_manifest = pl.concat(
        [manifest, new_manifest],
        how="vertical_relaxed",
    )

    expected_manifest_rows = (
        EXPECTED_MANIFEST_ROWS + TARGET_ROWS
    )

    if combined_manifest.height != expected_manifest_rows:
        raise ValueError("Combined manifest row count is incorrect.")

    combined_manifest.write_csv(TEMP_MANIFEST_PATH)

    # Copy current dataset and append synthetic records.
    shutil.copyfile(FINAL_PATH, TEMP_OUTPUT_PATH)

    with open(TEMP_OUTPUT_PATH, "ab") as output_file:
        with open(TEMP_ROWS_PATH, "rb") as rows_file:
            shutil.copyfileobj(rows_file, output_file)

    # Validate the temporary dataset before publishing.
    validated_rows = (
        pl.scan_csv(
            TEMP_OUTPUT_PATH,
            infer_schema_length=10_000,
            try_parse_dates=False,
        )
        .select(pl.len())
        .collect()
        .item()
    )

    expected_final_rows = base_rows + TARGET_ROWS

    if validated_rows != expected_final_rows:
        raise ValueError(
            f"Expected {expected_final_rows:,} rows, "
            f"found {validated_rows:,}."
        )

    validated_manifest = pl.read_csv(
        TEMP_MANIFEST_PATH,
        infer_schema_length=10_000,
        try_parse_dates=False,
    )

    if validated_manifest.height != expected_manifest_rows:
        raise ValueError("Temporary manifest row count is incorrect.")

    final_counts = {
        row["anomaly_type"]: row["len"]
        for row in (
            validated_manifest.group_by("anomaly_type")
            .len()
            .iter_rows(named=True)
        )
    }

    for anomaly, expected_count in EXPECTED_PREVIOUS.items():
        if final_counts.get(anomaly, 0) != expected_count:
            raise ValueError(
                f"Previous anomaly count changed: {anomaly}"
            )

    if final_counts.get(ANOMALY_NAME, 0) != TARGET_ROWS:
        raise ValueError("Promotional anomaly count is incorrect.")

    if sum(final_counts.values()) != expected_manifest_rows:
        raise ValueError("Manifest distribution does not reconcile.")

    # Publish only after all validations pass.
    os.replace(TEMP_OUTPUT_PATH, FINAL_PATH)
    os.replace(TEMP_MANIFEST_PATH, MANIFEST_PATH)

    print("\nSeasonal Promotional Purchase Surges completed successfully.")
    print(f"Rows before injection: {base_rows:,}")
    print(f"Promotional rows added: {TARGET_ROWS:,}")
    print(f"Final dataset rows: {validated_rows:,}")
    print(f"Expected manifest rows: {expected_manifest_rows:,}")
    print(f"Validated manifest rows: {validated_manifest.height:,}")
    print(f"Output file: {FINAL_PATH}")
    print(f"Manifest file: {MANIFEST_PATH}")

    print("\nManifest records by anomaly:")
    print(
        validated_manifest
        .group_by("anomaly_type")
        .len()
        .sort("anomaly_type")
    )

finally:
    # The original dataset and manifest are never deleted here.
    # Remove the disposable synthetic-row file after processing.
    if TEMP_ROWS_PATH.exists():
        TEMP_ROWS_PATH.unlink()


Current dataset rows: 10,336,570
Existing manifest rows: 265,305
Promotional purchase target: 42,449
Eligible purchase events: 179,051
Selected source events: 42,449
Generated promotional purchase records validated.
Promo window: 2019-10-13 10:00:00 UTC to 2019-10-13 12:00:00 UTC
Generated records: 42,449
First generated timestamp: 2019-10-13 10:00:00 UTC
Last generated timestamp: 2019-10-13 11:59:59 UTC

Seasonal Promotional Purchase Surges completed successfully.
Rows before injection: 10,336,570
Promotional rows added: 42,449
Final dataset rows: 10,379,019
Expected manifest rows: 307,754
Validated manifest rows: 307,754
Output file: D:\Samsung\Graduation Project\Data\Old Data\2019-Oct_week2_augmented.csv
Manifest file: D:\Samsung\Graduation Project\Data\Old Data\anomaly_manifest_week2.csv

Manifest records by anomaly:
shape: (3, 2)
┌─────────────────────────────────┬────────┐
│ anomaly_type                    ┆ len    │
│ ---                             ┆ ---    │
│ str             

# Week 2 — Duplicate Events Injection

## 1. Objective

Simulate duplicate event records in the Week 2 e-commerce dataset to support testing of data-quality checks and downstream deduplication logic.

The process adds **42,449 synthetic duplicate records** while preserving the original dataset columns and recording source information in the anomaly manifest.

## 2. Input Files

- **Dataset:** `2019-Oct_week2_augmented.csv`
- **Manifest:** `anomaly_manifest_week2.csv`

The dataset already contains the previously injected anomalies:
- Missing Data
- Bot-like Crawling Loops
- Seasonal Promotional Purchase Surges

## 3. Configuration

| Parameter | Value |
|---|---:|
| Anomaly type | Duplicate Events |
| Target synthetic records | 42,449 |
| Expected dataset rows before injection | 10,379,019 |
| Expected manifest rows before injection | 307,754 |
| Expected dataset rows after injection | 10,421,468 |
| Expected manifest rows after injection | 350,203 |

Synthetic record IDs use the prefix `W2-DE-`.

## 4. Implementation Steps

### Step 1 — Validate Input Files

- Confirm that the dataset and manifest exist.
- Validate the dataset's expected 11-column schema.
- Check the current dataset and manifest row counts.
- Verify the expected counts of the previously injected anomalies.
- Stop if Duplicate Events already exists in the manifest to prevent accidental reinjection.

### Step 2 — Select Source Records

- Add a source row number for traceability.
- Exclude records missing required fields such as `event_time`, `user_id`, `product_id`, or `user_session`.
- Restrict source selection to the original dataset range of 10,071,265 records, excluding previously appended synthetic records.
- Select 42,449 source records deterministically and distribute the selections across the eligible candidates.

### Step 3 — Generate Duplicate Events

For each selected source record, create a new row that preserves all 11 dataset columns exactly:

- `event_time`
- `event_type`
- `product_id`
- `category_id`
- `category_code`
- `brand`
- `price`
- `user_id`
- `user_session`
- `marketing_channel`
- `platform`

The generated record is an exact copy of its source in the dataset. Its synthetic ID and source metadata are stored in the manifest rather than added as new dataset columns.

### Step 4 — Update the Manifest

Each generated duplicate receives a unique ID in the format `W2-DE-000001`.

The manifest records:
- Week and anomaly type
- Synthetic record ID
- Source row number
- Source event time and event type
- Source user ID and product ID

This provides traceability between generated records and their original source records.

### Step 5 — Validate Temporary Outputs

Before publishing, the code:
- Writes generated rows to a temporary CSV file.
- Builds the updated manifest while retaining existing entries.
- Appends the duplicate rows to a temporary copy of the dataset.
- Checks the final dataset and manifest row counts.
- Verifies the anomaly counts and manifest reconciliation.

### Step 6 — Publish Validated Files

Only after validation succeeds, the code replaces the dataset and manifest with the validated temporary files using `os.replace()`.

The disposable temporary rows file is removed afterward.

## 5. Expected Results

| Validation | Expected result |
|---|---:|
| Duplicate records generated | 42,449 |
| Dataset rows after injection | 10,421,468 |
| Manifest rows after injection | 350,203 |
| Unique IDs in the new duplicate batch | 42,449 |
| Original dataset columns preserved | 11 |

*These are expected results from the code configuration; successful execution must be confirmed using the actual runtime output.*

## 6. Important Technical Note

The code validates row counts and manifest consistency, but those checks alone do not prove that every generated record exactly matches its referenced source. A stronger validation should compare the generated duplicate rows with their source rows across all 11 columns.

The `synthetic_ids` variable is created but is not subsequently used. The code instead excludes appended synthetic records by limiting the source row number to the original dataset range.

## 7. Purpose in the Big Data Pipeline

This anomaly supports testing of:
- Duplicate detection
- Deduplication rules
- Data-quality validation
- Record-level lineage and traceability
- Reconciliation of row counts before and after cleaning

**Expected downstream behavior:** a deduplication stage should identify the injected duplicates according to the chosen business key or full-row matching rule. The number of rows removed should be measured during that stage rather than assumed from the injection count alone.


In [67]:

from pathlib import Path
import os
import shutil
import polars as pl

# ============================================================
# 1. Configuration
# ============================================================

BASE_DIR = Path(r"D:\Samsung\Graduation Project\Data\Old Data")

FINAL_PATH = BASE_DIR / "2019-Oct_week2_augmented.csv"
MANIFEST_PATH = BASE_DIR / "anomaly_manifest_week2.csv"

TEMP_OUTPUT_PATH = BASE_DIR / "2019-Oct_week2_duplicates.tmp.csv"
TEMP_ROWS_PATH = BASE_DIR / "2019-Oct_week2_duplicate_rows.tmp.csv"
TEMP_MANIFEST_PATH = BASE_DIR / "anomaly_manifest_week2_duplicates.tmp.csv"

ANOMALY_NAME = "Duplicate Events"
TARGET_ROWS = 42_449

EXPECTED_BASE_ROWS = 10_379_019
EXPECTED_MANIFEST_ROWS = 307_754

EXPECTED_PREVIOUS = {
    "Missing Data": 212_244,
    "Bot-like Crawling Loops": 53_061,
    "Seasonal Promotional Purchase Surges": 42_449,
}

ALL_COLUMNS = [
    "event_time",
    "event_type",
    "product_id",
    "category_id",
    "category_code",
    "brand",
    "price",
    "user_id",
    "user_session",
    "marketing_channel",
    "platform",
]

MANIFEST_COLUMNS = [
    "week",
    "anomaly_type",
    "synthetic_row_id",
    "source_row_number",
    "missing_column",
    "source_event_time",
    "source_event_type",
    "source_user_id",
    "source_product_id",
]


# ============================================================
# 2. Validate current dataset and manifest
# ============================================================

if not FINAL_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {FINAL_PATH}")

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest not found: {MANIFEST_PATH}")

for path in [
    TEMP_OUTPUT_PATH,
    TEMP_ROWS_PATH,
    TEMP_MANIFEST_PATH,
]:
    if path.exists():
        raise FileExistsError(
            f"Temporary file already exists: {path}\n"
            "Inspect it before deleting or rerunning the cell."
        )

base_lf = pl.scan_csv(
    FINAL_PATH,
    infer_schema_length=10_000,
    try_parse_dates=False,
)

if list(base_lf.collect_schema().names()) != ALL_COLUMNS:
    raise ValueError("Dataset columns do not match the expected schema.")

base_rows = base_lf.select(pl.len()).collect().item()

manifest = pl.read_csv(
    MANIFEST_PATH,
    infer_schema_length=10_000,
    try_parse_dates=False,
)

if base_rows != EXPECTED_BASE_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_BASE_ROWS:,} dataset rows, "
        f"found {base_rows:,}."
    )

if manifest.height != EXPECTED_MANIFEST_ROWS:
    raise ValueError(
        f"Expected {EXPECTED_MANIFEST_ROWS:,} manifest rows, "
        f"found {manifest.height:,}."
    )

if manifest.columns != MANIFEST_COLUMNS:
    raise ValueError("Manifest columns do not match the expected schema.")

if manifest.filter(
    pl.col("anomaly_type") == ANOMALY_NAME
).height:
    raise ValueError(
        "Duplicate Events already exists in the manifest. "
        "Stopping to avoid duplicate injection."
    )

existing_counts = {
    row["anomaly_type"]: row["len"]
    for row in (
        manifest.group_by("anomaly_type")
        .len()
        .iter_rows(named=True)
    )
}

for anomaly, expected_count in EXPECTED_PREVIOUS.items():
    if existing_counts.get(anomaly, 0) != expected_count:
        raise ValueError(
            f"Unexpected count for {anomaly}: "
            f"{existing_counts.get(anomaly, 0):,}"
        )

print(f"Current dataset rows: {base_rows:,}")
print(f"Existing manifest rows: {manifest.height:,}")
print(f"Duplicate target: {TARGET_ROWS:,}")


# ============================================================
# 3. Select source records
# ============================================================

# Use a stable source row number for traceability.
# We select non-synthetic records so duplicates have clear sources.
synthetic_ids = (
    manifest
    .select("synthetic_row_id")
    .get_column("synthetic_row_id")
    .drop_nulls()
    .to_list()
)

# Synthetic records are excluded by their generated ID prefixes.
source_lf = (
    base_lf
    .with_row_index("_source_row", offset=1)
    .filter(
        ~pl.col("user_session").is_null()
        & ~pl.col("user_id").is_null()
        & ~pl.col("product_id").is_null()
        & ~pl.col("event_time").is_null()
    )
)

# Collect source candidates in stable order.
# This dataset is large, so only required columns are retained.
candidates = (
    source_lf
    .select(["_source_row"] + ALL_COLUMNS)
    .collect()
)

# Exclude already-known synthetic rows based on their manifest source
# IDs/prefix convention when possible. To avoid expensive string checks
# on every row, select source rows from the original non-synthetic range:
# existing synthetic records are appended to the dataset.
original_source_limit = 10_071_265

candidates = candidates.filter(
    pl.col("_source_row") <= original_source_limit
)

eligible_count = candidates.height

if eligible_count < TARGET_ROWS:
    raise ValueError(
        f"Not enough eligible source events: {eligible_count:,}"
    )

# Deterministic, evenly distributed selection.
step = eligible_count / TARGET_ROWS
selected_positions = [
    min(int(i * step), eligible_count - 1)
    for i in range(TARGET_ROWS)
]

selected = candidates[selected_positions]

if selected.height != TARGET_ROWS:
    raise ValueError("Selected source count does not match target.")

print(f"Eligible source events: {eligible_count:,}")
print(f"Selected source events: {selected.height:,}")


# ============================================================
# 4. Generate exact duplicate rows
# ============================================================

synthetic_rows = []
manifest_rows = []

for i, source in enumerate(selected.iter_rows(named=True), start=1):
    # Exact duplicate: all dataset columns remain identical.
    synthetic_rows.append({
        column: source[column]
        for column in ALL_COLUMNS
    })

    manifest_rows.append({
        "week": "Week 2",
        "anomaly_type": ANOMALY_NAME,
        "synthetic_row_id": f"W2-DE-{i:06d}",
        "source_row_number": source["_source_row"],
        "missing_column": None,
        "source_event_time": source["event_time"],
        "source_event_type": source["event_type"],
        "source_user_id": source["user_id"],
        "source_product_id": source["product_id"],
    })

synthetic_df = pl.DataFrame(
    synthetic_rows,
    schema=ALL_COLUMNS,
    orient="row",
    strict=False,
)

new_manifest = pl.DataFrame(
    manifest_rows,
    schema=manifest.schema,
    orient="row",
    strict=False,
)

if synthetic_df.height != TARGET_ROWS:
    raise ValueError("Generated duplicate row count is incorrect.")

if new_manifest.height != TARGET_ROWS:
    raise ValueError("Generated duplicate manifest count is incorrect.")

if new_manifest.select(
    pl.col("synthetic_row_id").n_unique()
).item() != TARGET_ROWS:
    raise ValueError("Synthetic duplicate IDs are not unique.")

print("Generated duplicate records and manifest validated.")


# ============================================================
# 5. Prepare and validate temporary outputs
# ============================================================

try:
    synthetic_df.write_csv(
        TEMP_ROWS_PATH,
        include_header=False,
    )

    combined_manifest = pl.concat(
        [manifest, new_manifest],
        how="vertical_relaxed",
    )

    expected_manifest_rows = EXPECTED_MANIFEST_ROWS + TARGET_ROWS

    if combined_manifest.height != expected_manifest_rows:
        raise ValueError("Combined manifest count is incorrect.")

    combined_manifest.write_csv(TEMP_MANIFEST_PATH)

    # Copy current dataset, then append duplicate rows.
    shutil.copyfile(FINAL_PATH, TEMP_OUTPUT_PATH)

    with open(TEMP_OUTPUT_PATH, "ab") as output_file:
        with open(TEMP_ROWS_PATH, "rb") as rows_file:
            shutil.copyfileobj(rows_file, output_file)

    validated_rows = (
        pl.scan_csv(
            TEMP_OUTPUT_PATH,
            infer_schema_length=10_000,
            try_parse_dates=False,
        )
        .select(pl.len())
        .collect()
        .item()
    )

    expected_final_rows = base_rows + TARGET_ROWS

    if validated_rows != expected_final_rows:
        raise ValueError(
            f"Expected {expected_final_rows:,} rows, "
            f"found {validated_rows:,}."
        )

    validated_manifest = pl.read_csv(
        TEMP_MANIFEST_PATH,
        infer_schema_length=10_000,
        try_parse_dates=False,
    )

    if validated_manifest.height != expected_manifest_rows:
        raise ValueError("Temporary manifest count is incorrect.")

    final_counts = {
        row["anomaly_type"]: row["len"]
        for row in (
            validated_manifest.group_by("anomaly_type")
            .len()
            .iter_rows(named=True)
        )
    }

    for anomaly, expected_count in EXPECTED_PREVIOUS.items():
        if final_counts.get(anomaly, 0) != expected_count:
            raise ValueError(
                f"Previous anomaly count changed: {anomaly}"
            )

    if final_counts.get(ANOMALY_NAME, 0) != TARGET_ROWS:
        raise ValueError("Duplicate Events manifest count is incorrect.")

    if sum(final_counts.values()) != expected_manifest_rows:
        raise ValueError("Manifest distribution does not reconcile.")

    # Publish validated files.
    os.replace(TEMP_OUTPUT_PATH, FINAL_PATH)
    os.replace(TEMP_MANIFEST_PATH, MANIFEST_PATH)

    print("\nDuplicate Events injection completed successfully.")
    print(f"Rows before injection: {base_rows:,}")
    print(f"Duplicate rows added: {TARGET_ROWS:,}")
    print(f"Final dataset rows: {validated_rows:,}")
    print(f"Expected manifest rows: {expected_manifest_rows:,}")
    print(f"Validated manifest rows: {validated_manifest.height:,}")
    print(f"Output file: {FINAL_PATH}")
    print(f"Manifest file: {MANIFEST_PATH}")

    print("\nManifest records by anomaly:")
    print(
        validated_manifest
        .group_by("anomaly_type")
        .len()
        .sort("anomaly_type")
    )

finally:
    if TEMP_ROWS_PATH.exists():
        TEMP_ROWS_PATH.unlink()


Current dataset rows: 10,379,019
Existing manifest rows: 307,754
Duplicate target: 42,449
Eligible source events: 10,071,265
Selected source events: 42,449
Generated duplicate records and manifest validated.

Duplicate Events injection completed successfully.
Rows before injection: 10,379,019
Duplicate rows added: 42,449
Final dataset rows: 10,421,468
Expected manifest rows: 350,203
Validated manifest rows: 350,203
Output file: D:\Samsung\Graduation Project\Data\Old Data\2019-Oct_week2_augmented.csv
Manifest file: D:\Samsung\Graduation Project\Data\Old Data\anomaly_manifest_week2.csv

Manifest records by anomaly:
shape: (4, 2)
┌─────────────────────────────────┬────────┐
│ anomaly_type                    ┆ len    │
│ ---                             ┆ ---    │
│ str                             ┆ u32    │
╞═════════════════════════════════╪════════╡
│ Bot-like Crawling Loops         ┆ 53061  │
│ Duplicate Events                ┆ 42449  │
│ Missing Data                    ┆ 212244 │
│ Sea